# Observatório FAPESP — agente da curadoria semanal
**Ana Beatriz Mação de Barros Ferreira · governança digital, proteção de dados e Mercosul**

Este notebook reúne coleta recente, catálogo de fontes, classificação por palavras-chave,
tabelas, gráficos interativos e uma lista de até cinco publicações para leitura.
Os cinco eixos são GDPR/LGPD, IA e dados, governança digital no Mercosul, UE–Mercosul e Sul Global.

**Como usar:** execute **Ambiente de execução → Executar tudo**. Para explorar depois,
abra o arquivo **painel_interativo.html** incluído no ZIP da última seção.
Deixe **DATA_REFERENCIA** vazia para usar a data atual de Brasília; informe AAAA-MM-DD para um recorte histórico.
Edite os demais parâmetros abaixo para mudar o período ou acrescentar CSVs dos outros coletores.

O notebook inclui uma cópia real dos dados e do catálogo usados na criação: funciona também
sem rede, identificando a idade da base. A coleta ao vivo monitora **NIC.br, EDPB, Mercociudades
e Senado Federal**; os demais notebooks não são executados por esta rotina.
Nenhuma chave de API nem GPU é necessária. A análise usa regras transparentes e não cria notícias.

In [ ]:
#@title 1. Parâmetros da pesquisa
import os
from pathlib import Path
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

DATA_REFERENCIA = "" #@param {type:"string"}
JANELA_DIAS = 7 #@param {type:"integer"}
ATUALIZAR_NOTICIAS = True #@param {type:"boolean"}
BAIXAR_BASE_GITHUB = True #@param {type:"boolean"}
INCLUIR_NOTAS_DO_CATALOGO = False #@param {type:"boolean"}
CORPUS_GRAFICOS = "todos" #@param ["todos", "scrapers", "catalogo"]
PASTA_CSV_EXTRA = "" #@param {type:"string"}
CAMINHO_CATALOGO_PROPRIO = "" #@param {type:"string"}
SALVAR_NO_GOOGLE_DRIVE = False #@param {type:"boolean"}
PASTA_DRIVE = "/content/drive/MyDrive/FAPESP/Curadoria" #@param {type:"string"}
BAIXAR_ZIP_AO_TERMINAR = True #@param {type:"boolean"}

REFERENCIA = date.fromisoformat(DATA_REFERENCIA) if DATA_REFERENCIA else datetime.now(ZoneInfo("America/Sao_Paulo")).date()
if not 1 <= JANELA_DIAS <= 366:
    raise ValueError("Escolha uma janela de 1 a 366 dias.")
if CORPUS_GRAFICOS not in {"todos", "scrapers", "catalogo"}:
    raise ValueError("CORPUS_GRAFICOS deve ser todos, scrapers ou catalogo.")
MODO_TEXTO = "title_summary" if INCLUIR_NOTAS_DO_CATALOGO else "title"
BASE = Path("/content") if Path("/content").exists() else Path.cwd()
AGENTE = BASE / "agente_curadoria_fapesp"
AGENTE.mkdir(parents=True, exist_ok=True)
MODO_OFFLINE = os.getenv("CURADORIA_OFFLINE") == "1"
if MODO_OFFLINE:
    ATUALIZAR_NOTICIAS = BAIXAR_BASE_GITHUB = False
print("Referência:", REFERENCIA, "| Janela:", JANELA_DIAS, "dias | Texto:", MODO_TEXTO)

In [ ]:
#@title 2. Preparar as dependências
import importlib.util
import subprocess
import sys

pacotes = {"requests": "requests>=2.32,<3", "bs4": "beautifulsoup4>=4.12,<5",
           "IPython": "ipython>=8,<10", "matplotlib": "matplotlib>=3.7,<4",
           "plotly": "plotly>=5.20,<7"}
faltantes = [spec for module, spec in pacotes.items() if importlib.util.find_spec(module) is None]
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])
print("Dependências disponíveis. CPU é suficiente.")

In [ ]:
#@title 3. Carregar o agente e a base real incluída
import json
ARQUIVOS_AGENTE = {'scripts/curadoria_agent.py': '"""Agente reproduzível de tabelas e gráficos da curadoria FAPESP.\n\nAnalisa títulos, sem inferir o conteúdo integral das publicações. Lê a base dos\nscrapers e, opcionalmente, o catálogo Markdown, mantendo os dois corpus separados.\nNão consulta uma API de IA, não inventa dados e nunca escreve nos arquivos de entrada.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport base64\nimport csv\nimport hashlib\nimport html\nimport json\nimport re\nimport sys\nimport textwrap\nimport unicodedata\nfrom collections import Counter\nfrom datetime import date, datetime, timedelta, timezone\nfrom itertools import combinations\nfrom pathlib import Path\nfrom urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit\nfrom zoneinfo import ZoneInfo\n\nROOT = Path(__file__).resolve().parents[1]\nMONTHS = dict(zip(\n    "janeiro fevereiro marco abril maio junho julho agosto setembro outubro novembro dezembro".split(),\n    range(1, 13)))\nMONTHS.update(dict(zip(\n    "enero febrero marzo abril mayo junio julio agosto septiembre octubre noviembre diciembre".split(),\n    range(1, 13))))\nMONTHS.update(dict(zip(\n    "january february march april may june july august september october november december".split(),\n    range(1, 13))))\n\n\ndef normalize(value: str) -> str:\n    value = unicodedata.normalize("NFKD", str(value).lower())\n    value = "".join(c for c in value if not unicodedata.combining(c))\n    return " ".join(re.findall(r"[^\\W_]+", value, flags=re.UNICODE))\n\n\ndef contains(text: str, phrase: str) -> bool:\n    """Whole words/phrases: \'IA\' must not match \'viagem\' or \'social\'."""\n    return bool(phrase) and f" {phrase} " in f" {text} "\n\n\ndef safe_url(raw: str) -> str:\n    try:\n        parsed = urlsplit(str(raw).strip())\n        if parsed.scheme not in {"https", "http"} or not parsed.hostname or parsed.username:\n            return ""\n        query = [(k, v) for k, v in parse_qsl(parsed.query, keep_blank_values=True)\n                 if not k.lower().startswith("utm_") and k.lower() not in {"fbclid", "gclid"}]\n        return urlunsplit((parsed.scheme.lower(), parsed.netloc.lower(),\n                           parsed.path or "/", urlencode(sorted(query)), ""))\n    except ValueError:\n        return ""\n\n\ndef publication_date(raw: str) -> tuple[str, str]:\n    """Return an ISO day or month; never substitute first_seen for publication."""\n    value = normalize(raw)\n    iso = re.fullmatch(r"(\\d{4}-\\d{2}-\\d{2})(?:T.*)?", raw.strip())\n    if iso:\n        try:\n            return date.fromisoformat(iso[1]).isoformat(), "day"\n        except ValueError:\n            return "", "unknown"\n    numeric = re.fullmatch(r"(\\d{1,2})/(\\d{1,2})/(\\d{4})", raw.strip())\n    named = re.fullmatch(r"(\\d{1,2})(?:o)? de (\\w+) de (\\d{4})", value)\n    try:\n        if numeric:\n            return date(int(numeric[3]), int(numeric[2]), int(numeric[1])).isoformat(), "day"\n        if named and named[2] in MONTHS:\n            return date(int(named[3]), MONTHS[named[2]], int(named[1])).isoformat(), "day"\n        month = re.fullmatch(r"(\\w+) de (\\d{4})", value)\n        if month and month[1] in MONTHS:\n            return f"{int(month[2]):04d}-{MONTHS[month[1]]:02d}", "month"\n        if re.fullmatch(r"\\d{4}-\\d{2}", raw.strip()):\n            date.fromisoformat(raw.strip() + "-01")\n            return raw.strip(), "month"\n        if re.fullmatch(r"\\d{4}", raw.strip()) and 1 <= int(raw.strip()) <= 9999:\n            return raw.strip(), "year"\n        for fmt in ("%d %B %Y", "%B %d, %Y", "%d %b %Y", "%B %Y"):\n            try:\n                parsed = datetime.strptime(raw.strip(), fmt)\n                return (parsed.strftime("%Y-%m"), "month") if fmt == "%B %Y" else (parsed.date().isoformat(), "day")\n            except ValueError:\n                pass\n    except ValueError:\n        pass\n    return "", "unknown"\n\n\ndef seen_date(raw: str) -> str:\n    try:\n        parsed = datetime.fromisoformat(raw.replace("Z", "+00:00"))\n        if parsed.tzinfo is None:\n            parsed = parsed.replace(tzinfo=timezone.utc)\n        return parsed.astimezone(ZoneInfo("America/Sao_Paulo")).date().isoformat()\n    except (ValueError, AttributeError):\n        return ""\n\n\ndef load_config(path: Path) -> dict:\n    config = json.loads(path.read_text(encoding="utf-8"))\n    known = set(config["keywords"])\n    required = set(config["relevance_keywords"])\n    for rule in config["axes"].values():\n        required.update(rule.get("any_keywords", []))\n        for group in rule.get("all_groups", []):\n            required.update(group)\n    if required - known:\n        raise ValueError(f"Palavras-chave não definidas: {sorted(required - known)}")\n    return config\n\n\ndef read_news(path: Path) -> list[dict]:\n    with path.open(newline="", encoding="utf-8-sig") as handle:\n        reader = csv.DictReader(handle)\n        if not {"source_id", "title", "published_date", "url"}.issubset(reader.fieldnames or []):\n            raise ValueError("CSV deve conter source_id, title, published_date e url")\n        return [{**row, "publication_type": row.get("publication_type") or "Notícia coletada",\n                 "summary": row.get("summary", ""), "access": row.get("access", ""),\n                 "evidence": row.get("evidence", ""), "corpus": "scrapers",\n                 "manually_selected": False} for row in reader]\n\n\ndef read_catalog(path: Path) -> list[dict]:\n    """Parse existing ### entries without turning outline headings into records."""\n    entries = []\n    current = None\n    for line in path.read_text(encoding="utf-8").splitlines():\n        if line.startswith("### "):\n            if current:\n                entries.append(current)\n            current = {"title": line[4:].strip(), "links": []}\n        elif re.match(r"^#{1,2} ", line):\n            if current:\n                entries.append(current)\n            current = None\n        elif current:\n            match = re.match(r"^- \\*\\*(.+?):\\*\\*\\s*(.*)", line)\n            if match:\n                key, value = normalize(match[1]), match[2].strip()\n                current[key] = value\n                if key in {"link", "texto integral", "apresentacao oficial da ue"}:\n                    current["links"].extend(re.findall(r"https?://[^\\s<>]+", value))\n    if current:\n        entries.append(current)\n    rows = []\n    for entry in entries:\n        if not entry["links"]:\n            raise ValueError(f"Entrada do catálogo sem link: {entry[\'title\']}")\n        source = next((entry[k] for k in ("instituicao", "instituicoes", "fonte", "autores", "autoras",\n                      "autor", "autora", "instituicao responsavel pelas diretrizes") if entry.get(k)), "Não informada")\n        published = entry.get("data", entry.get("atualizacao documental mais recente listada", ""))\n        rows.append({"source_id": source, "title": entry["title"], "published_date": published,\n                     "published_raw": published, "url": entry["links"][0],\n                     "other_links": "; ".join(entry["links"][1:]), "first_seen": "", "last_seen": "",\n                     "publication_type": entry.get("tipo", "Não informado"),\n                     "summary": entry.get("uso no relatorio", ""),\n                     "evidence": entry.get("classificacao", ""), "access": entry.get("acesso", ""),\n                     "corpus": "catalogo", "manually_selected": True})\n    return rows\n\n\ndef prepare(rows: list[dict], config: dict, analysis_text: str = "title") -> tuple[list[dict], dict]:\n    if analysis_text not in {"title", "title_summary"}:\n        raise ValueError("analysis_text deve ser title ou title_summary")\n    unique = {}\n    stats = {"input_rows": len(rows), "duplicates_removed": 0, "invalid_records": 0}\n    aliases = {label: sorted({normalize(alias) for alias in phrases}, key=len, reverse=True)\n               for label, phrases in config["keywords"].items()}\n    for original in rows:\n        row = dict(original)\n        row["title"] = " ".join(str(row.get("title", "")).split())\n        canonical = safe_url(row.get("url", ""))\n        if not canonical or not row["title"]:\n            stats["invalid_records"] += 1\n            continue\n        parsed, precision = publication_date(str(row.get("published_date", "")))\n        row.update(canonical_url=canonical, published_date=parsed, date_precision=precision,\n                   published_raw=row.get("published_raw") or original.get("published_date", ""),\n                   source=config["source_names"].get(row["source_id"], row["source_id"]),\n                   first_seen_date=seen_date(row.get("first_seen", "")))\n        row["analysis_text"] = row["title"]\n        if analysis_text == "title_summary" and row.get("summary"):\n            row["analysis_text"] += " " + str(row["summary"])\n        text = normalize(row["analysis_text"])\n        matches = {label: [alias for alias in phrases if contains(text, alias)]\n                   for label, phrases in aliases.items()}\n        row["keywords"] = [label for label, hits in matches.items() if hits]\n        row["candidate"] = bool(set(row["keywords"]) & set(config["relevance_keywords"]))\n        topics = []\n        if row["candidate"] or row.get("manually_selected"):\n            for label, rule in config["axes"].items():\n                any_ok = not rule.get("any_keywords") or bool(set(rule["any_keywords"]) & set(row["keywords"]))\n                all_ok = all(set(group) & set(row["keywords"]) for group in rule.get("all_groups", []))\n                if any_ok and all_ok:\n                    topics.append(label)\n        row["topics"] = topics\n        row["match_evidence"] = "; ".join(f"{label}: {\', \'.join(hits)}" for label, hits in matches.items() if hits)\n        if canonical in unique:\n            stats["duplicates_removed"] += 1\n            old = unique[canonical]\n            # Duplicate URLs represent one document. Prefer usable dates, retain earliest detection.\n            if old["date_precision"] == "day" and precision != "day":\n                row.update(published_date=old["published_date"], date_precision=old["date_precision"],\n                           published_raw=old["published_raw"])\n            detections = sorted(x for x in [old.get("first_seen_date"), row["first_seen_date"]] if x)\n            if detections:\n                row["first_seen_date"] = detections[0]\n                timestamps = [value for value in [old.get("first_seen"), row.get("first_seen")]\n                              if value and seen_date(value) == detections[0]]\n                if timestamps:\n                    row["first_seen"] = min(timestamps)\n            row["also_sources"] = sorted(set(old.get("also_sources", [old["source"]])) | {row["source"]})\n        unique[canonical] = row\n    return sorted(unique.values(), key=lambda r: (r["published_date"], r["title"]), reverse=True), stats\n\n\ndef in_window(row: dict, start: str, end: str) -> bool:\n    return row["date_precision"] == "day" and start <= row["published_date"] <= end\n\n\ndef summarize(rows: list[dict], config: dict, as_of: date, days: int, diagnostics: dict | None = None) -> dict:\n    start = (as_of - timedelta(days=days - 1)).isoformat()\n    end = as_of.isoformat()\n    week = [r for r in rows if in_window(r, start, end)]\n    eligible = [r for r in rows if r["candidate"] and\n                (not r["published_date"] or r["published_date"] <= end)]\n    recent = [r for r in week if r["candidate"]]\n    prior_start = (as_of - timedelta(days=2 * days - 1)).isoformat()\n    prior_end = (as_of - timedelta(days=days)).isoformat()\n    prior = [r for r in eligible if in_window(r, prior_start, prior_end)]\n    report_sources = {r["source_id"]: r for r in (diagnostics or {}).get("sources", [])}\n    source_rows = []\n    source_ids = set(r["source_id"] for r in rows) | set(report_sources)\n    for source_id in sorted(source_ids):\n        selected = [r for r in rows if r["source_id"] == source_id]\n        status = report_sources.get(source_id, {})\n        source_rows.append({"fonte": config["source_names"].get(source_id, source_id),\n                            "total_base": len(selected),\n                            "candidatas_base": sum(r["candidate"] for r in selected),\n                            "publicadas_janela": sum(in_window(r, start, end) for r in selected),\n                            "candidatas_janela": sum(r["candidate"] and in_window(r, start, end) for r in selected),\n                            "links_detectados_janela": sum(start <= r["first_seen_date"] <= end for r in selected),\n                            "sem_data_exata": sum(r["date_precision"] != "day" for r in selected),\n                            "status_ultima_coleta": status.get("status", "não informado"),\n                            "observacao": status.get("error", status.get("coverage_note", ""))})\n    topic_rows = [{"tema": axis, "candidatas_historico": sum(axis in r["topics"] for r in eligible),\n                   "candidatas_janela": sum(axis in r["topics"] for r in recent)} for axis in config["axes"]]\n    daily = [{"data": (as_of - timedelta(days=days - 1 - i)).isoformat(),\n              "publicadas": sum(r["published_date"] == (as_of - timedelta(days=days - 1 - i)).isoformat() for r in week),\n              "candidatas": sum(r["published_date"] == (as_of - timedelta(days=days - 1 - i)).isoformat() for r in recent)}\n             for i in range(days)]\n    monthly = Counter(r["published_date"][:7] for r in eligible if r["date_precision"] in {"day", "month"})\n    terms = []\n    for label in config["keywords"]:\n        historical = sum(label in r["keywords"] for r in eligible)\n        current = sum(label in r["keywords"] for r in recent)\n        previous = sum(label in r["keywords"] for r in prior)\n        terms.append({"palavra_chave": label, "documentos_historico": historical,\n                      "documentos_janela": current, "documentos_janela_anterior": previous,\n                      "variacao_documentos": current - previous,\n                      "percentual_candidatas_janela": round(100 * current / len(recent), 2) if recent else ""})\n    terms.sort(key=lambda r: (-r["documentos_historico"], r["palavra_chave"]))\n    stopwords = set(normalize(config["stopwords"]).split())\n    occurrences, documents = Counter(), Counter()\n    for row in eligible:\n        words = [w for w in normalize(row.get("analysis_text", row["title"])).split()\n                 if len(w) >= 3 and not w.isdigit() and w not in stopwords]\n        occurrences.update(words)\n        documents.update(set(words))\n    words = [{"palavra": w, "ocorrencias": n, "documentos": documents[w]}\n             for w, n in occurrences.most_common(100)]\n    pairs = Counter()\n    for row in eligible:\n        pairs.update(combinations(sorted(set(row["keywords"])), 2))\n    cooccurrences = [{"termo_1": a, "termo_2": b, "documentos": n}\n                     for (a, b), n in sorted(pairs.items(), key=lambda p: (-p[1], p[0]))]\n    return {"start": start, "end": end, "prior_start": prior_start, "prior_end": prior_end,\n            "rows": rows, "week": week, "eligible": eligible, "recent": recent,\n            "sources": source_rows, "topics": topic_rows, "daily": daily,\n            "monthly": [{"mes": m, "candidatas": n} for m, n in sorted(monthly.items())],\n            "keywords": terms, "words": words, "cooccurrences": cooccurrences,\n            "counts": {"documents": len(rows), "candidates": len(eligible), "published_window": len(week),\n                       "candidates_window": len(recent), "candidates_previous_window": len(prior),\n                       "without_exact_date": sum(r["date_precision"] != "day" for r in rows),\n                       "future_publications": sum(r["published_date"] > end for r in rows),\n                       "candidates_without_axis": sum(not r["topics"] for r in eligible)}}\n\n\ndef write_csv(path: Path, rows: list[dict], fields: list[str]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", newline="", encoding="utf-8-sig") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")\n        writer.writeheader()\n        for row in rows:\n            # Neutralize spreadsheet formulas when CSVs are opened in Excel/Sheets.\n            clean = {}\n            for key in fields:\n                value = row.get(key, "")\n                if isinstance(value, list):\n                    value = "; ".join(value)\n                if isinstance(value, str) and value.lstrip().startswith(("=", "+", "-", "@")):\n                    value = "\'" + value\n                clean[key] = value\n            writer.writerow(clean)\n\n\ndef save_tables(folder: Path, data: dict) -> None:\n    document_fields = ["title", "source", "publication_type", "published_date", "date_precision",\n                       "published_raw", "url", "other_links", "candidate", "topics", "keywords",\n                       "match_evidence", "first_seen", "first_seen_date", "last_seen", "summary", "evidence", "access",\n                       "manually_selected", "analysis_text"]\n    for filename, key in [("documentos", "rows"), ("publicadas_janela", "week"), ("candidatas_janela", "recent")]:\n        write_csv(folder / f"{filename}.csv", data[key], document_fields)\n    for filename, key, fields in [\n        ("por_fonte", "sources", ["fonte", "total_base", "candidatas_base", "publicadas_janela", "candidatas_janela", "links_detectados_janela", "sem_data_exata", "status_ultima_coleta", "observacao"]),\n        ("por_tema", "topics", ["tema", "candidatas_historico", "candidatas_janela"]),\n        ("por_dia", "daily", ["data", "publicadas", "candidatas"]),\n        ("por_mes", "monthly", ["mes", "candidatas"]),\n        ("palavras_chave", "keywords", ["palavra_chave", "documentos_historico", "documentos_janela", "documentos_janela_anterior", "variacao_documentos", "percentual_candidatas_janela"]),\n        ("palavras_frequentes", "words", ["palavra", "ocorrencias", "documentos"]),\n        ("coocorrencias", "cooccurrences", ["termo_1", "termo_2", "documentos"])]:\n        write_csv(folder / f"{filename}.csv", data[key], fields)\n\n\ndef charts(folder: Path, data: dict, label: str) -> list[Path]:\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    from matplotlib.ticker import MaxNLocator\n    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,\n                         "axes.spines.top": False, "axes.spines.right": False})\n    paths = []\n\n    def save(fig, filename, note):\n        fig.text(0.02, 0.02, textwrap.fill(note, 120), fontsize=8, color="#526173")\n        fig.tight_layout(rect=(0.02, 0.09, 0.98, 0.98))\n        path = folder / filename\n        fig.savefig(path, dpi=160, facecolor="white")\n        plt.close(fig)\n        paths.append(path)\n\n    def bar(rows, xkey, value, title, filename, note):\n        selected = [r for r in rows if r[value] > 0][:15]\n        fig, ax = plt.subplots(figsize=(10.2, max(4.5, 0.38 * len(selected) + 2.3)))\n        ax.set_title(title, loc="left", pad=18, fontweight="bold")\n        if selected:\n            labels = [textwrap.fill(str(r[xkey]), 42) for r in selected][::-1]\n            values = [r[value] for r in selected][::-1]\n            bars = ax.barh(labels, values, color="#287d8e")\n            ax.bar_label(bars, padding=4)\n            ax.set_xlim(0, max(values) * 1.2 + 0.5)\n            ax.xaxis.set_major_locator(MaxNLocator(integer=True))\n            ax.set_xlabel("Documentos")\n        else:\n            ax.text(0.5, 0.5, "Sem documentos disponíveis neste recorte", ha="center", transform=ax.transAxes)\n            ax.set_axis_off()\n        save(fig, filename, note)\n\n    window = f"{data[\'start\']} a {data[\'end\']}"\n    caution = "Contagem de documentos observados; cobertura depende dos scrapers. Zero observado não prova ausência de publicações."\n    bar(sorted(data["sources"], key=lambda r: -r["candidatas_janela"]), "fonte", "candidatas_janela",\n        f"{label} · candidatas por fonte\\n{window}", "fontes_janela.png", caution)\n    bar(data["topics"], "tema", "candidatas_historico", f"{label} · eixos de pesquisa no histórico",\n        "temas_historico.png", "Classificação lexical dos títulos. Um documento pode integrar vários eixos; as barras não somam o total de documentos.")\n    bar(data["keywords"], "palavra_chave", "documentos_historico", f"{label} · palavras-chave no histórico",\n        "palavras_chave_historico.png", "Sinônimos em português, inglês e espanhol são agrupados. Cada termo conta uma vez por documento; somente candidatas lexicais.")\n    bar(sorted(data["words"], key=lambda r: -r["documentos"]), "palavra", "documentos",\n        f"{label} · palavras frequentes nos títulos", "palavras_frequentes_historico.png",\n        "Número de documentos com cada palavra após remoção de palavras funcionais. Grafia normalizada; frequências brutas estão no CSV.")\n    fig, ax = plt.subplots(figsize=(10.2, 4.7))\n    ax.plot(range(len(data["daily"])), [r["publicadas"] for r in data["daily"]], label="Todas as publicações observadas", marker="o", color="#99a8b5")\n    ax.plot(range(len(data["daily"])), [r["candidatas"] for r in data["daily"]], label="Candidatas para leitura", marker="o", color="#287d8e")\n    step = max(1, len(data["daily"]) // 10)\n    ticks = list(range(0, len(data["daily"]), step))\n    ax.set_xticks(ticks, [data["daily"][i]["data"][5:] for i in ticks], rotation=30)\n    ax.yaxis.set_major_locator(MaxNLocator(integer=True)); ax.set_ylim(bottom=0)\n    ax.set_title(f"{label} · data de publicação\\n{window}", loc="left", fontweight="bold", pad=18)\n    ax.set_ylabel("Documentos"); ax.legend(frameon=False)\n    save(fig, "publicacoes_janela.png", caution + " Datas ausentes ou apenas mensais ficam fora da janela diária.")\n    fig, ax = plt.subplots(figsize=(10.2, 4.7))\n    if data["monthly"]:\n        months = [r["mes"] for r in data["monthly"]]\n        ax.bar(range(len(months)), [r["candidatas"] for r in data["monthly"]], color="#287d8e")\n        step = max(1, len(months) // 12)\n        ticks = list(range(0, len(months), step))\n        ax.set_xticks(ticks, [months[i] for i in ticks], rotation=45, ha="right")\n        ax.yaxis.set_major_locator(MaxNLocator(integer=True)); ax.set_ylabel("Candidatas")\n    else:\n        ax.text(0.5, 0.5, "Sem publicações com mês identificável", ha="center", transform=ax.transAxes)\n        ax.set_axis_off()\n    ax.set_title(f"{label} · publicações candidatas por mês", loc="left", fontweight="bold", pad=18)\n    save(fig, "publicacoes_mensais.png", "Agrupamento pela publicação: aceita datas com precisão de dia ou mês. Datas desconhecidas e futuras ficam fora. Meses exibidos são os observados na base.")\n    labels = [r["palavra_chave"] for r in data["keywords"] if r["documentos_historico"] > 0][:10]\n    fig, ax = plt.subplots(figsize=(10.4, 8.4))\n    ax.set_title(f"{label} · coocorrência de palavras-chave nos títulos", loc="left", fontweight="bold", pad=18)\n    if labels:\n        matrix = [[sum(a in r["keywords"] and b in r["keywords"] for r in data["eligible"])\n                   for b in labels] for a in labels]\n        im = ax.imshow(matrix, cmap="Blues", vmin=0)\n        ax.set_xticks(range(len(labels)), [textwrap.fill(x, 20) for x in labels], rotation=60, ha="right")\n        ax.set_yticks(range(len(labels)), [textwrap.fill(x, 23) for x in labels])\n        maximum = max(max(row) for row in matrix)\n        for i, row in enumerate(matrix):\n            for j, n in enumerate(row):\n                ax.text(j, i, str(n), ha="center", va="center", color="white" if n > maximum / 2 else "#163047", fontsize=9)\n        fig.colorbar(im, ax=ax, shrink=0.65, label="Documentos")\n    else:\n        ax.text(0.5, 0.5, "Sem palavras-chave identificadas", ha="center", transform=ax.transAxes); ax.set_axis_off()\n    save(fig, "coocorrencias_historico.png", "Termos presentes no mesmo título; não indica causalidade ou influência regulatória. A diagonal mostra documentos com o próprio termo.")\n    return paths\n\n\ndef html_table(rows: list[dict], fields: list[tuple[str, str]], searchable: bool = False) -> str:\n    headers = "".join(f"<th>{html.escape(label)}</th>" for _, label in fields)\n    body = []\n    for row in rows:\n        cells = []\n        for key, _ in fields:\n            value = row.get(key, "")\n            if key == "candidate":\n                value = "Sim" if value else "Sem correspondência"\n            elif key == "date_precision":\n                value = {"day": "Dia", "month": "Mês", "year": "Ano", "unknown": "Não informada"}.get(value, value)\n            if isinstance(value, list):\n                value = "; ".join(value)\n            if key == "title" and safe_url(row.get("url", "")):\n                cell = f\'<a href="{html.escape(row["url"], quote=True)}" target="_blank" rel="noopener noreferrer">{html.escape(str(value))}</a>\'\n            else:\n                cell = html.escape(str(value))\n            cells.append(f"<td>{cell}</td>")\n        body.append("<tr>" + "".join(cells) + "</tr>")\n    empty = f\'<tr><td colspan="{len(fields)}">Sem documentos neste recorte.</td></tr>\'\n    return f\'<div class="table-wrap"><table class="{"searchable" if searchable else "summary-table"}"><thead><tr>{headers}</tr></thead><tbody>{"".join(body) or empty}</tbody></table></div>\'\n\n\ndef render_report(output: Path, corpus_data: dict, image_paths: dict, metadata: dict) -> None:\n    blocks = []\n    esc = html.escape\n    for corpus, data in corpus_data.items():\n        label = "Notícias dos scrapers" if corpus == "scrapers" else "Fontes selecionadas no catálogo"\n        counts = data["counts"]\n        cards = "".join(f\'<div class="card"><strong>{n}</strong><span>{esc(text)}</span></div>\' for n, text in [\n            (counts["documents"], "Documentos na base"), (counts["candidates"], "Candidatas lexicais no histórico"),\n            (counts["published_window"], "Publicadas na janela"), (counts["candidates_window"], "Candidatas na janela")])\n        images = []\n        for path in image_paths[corpus]:\n            encoded = base64.b64encode(path.read_bytes()).decode("ascii")\n            images.append(f\'<figure><img src="data:image/png;base64,{encoded}" alt="{esc(path.stem)}" loading="lazy"></figure>\')\n        source_table = html_table(data["sources"], [("fonte", "Fonte"), ("total_base", "Base"),\n            ("publicadas_janela", "Publicadas na janela"), ("candidatas_janela", "Candidatas na janela"),\n            ("links_detectados_janela", "Primeira detecção na janela"), ("status_ultima_coleta", "Última coleta"), ("observacao", "Observação")])\n        keywords = html_table(data["keywords"], [("palavra_chave", "Palavra-chave"), ("documentos_historico", "Histórico"),\n            ("documentos_janela", "Janela atual"), ("documentos_janela_anterior", "Janela anterior"), ("variacao_documentos", "Variação observada")])\n        documents = html_table(data["rows"], [("title", "Título e link"), ("source", "Fonte"),\n            ("publication_type", "Tipo"), ("published_date", "Publicação"), ("date_precision", "Precisão"),\n            ("candidate", "Candidata lexical"), ("topics", "Eixos"), ("keywords", "Palavras-chave"),\n            ("match_evidence", "Termos encontrados")], searchable=True)\n        downloads = " · ".join(f\'<a href="{corpus}/{name}">{esc(name)}</a>\' for name in [\n            "documentos.csv", "candidatas_janela.csv", "por_fonte.csv", "por_tema.csv", "por_dia.csv", "por_mes.csv", "palavras_chave.csv", "palavras_frequentes.csv", "coocorrencias.csv"])\n        blocks.append(f\'<section><h2>{label}</h2><div class="cards">{cards}</div><p>Sem data exata: {counts["without_exact_date"]}. \'\n                      f\'Publicações futuras excluídas do histórico e da janela: {counts["future_publications"]}. Candidatas sem eixo correspondente: {counts["candidates_without_axis"]}.</p>\'\n                      f\'<h3>Cobertura e fontes</h3>{source_table}<div class="charts">{"".join(images)}</div>\'\n                      f\'<h3>Palavras-chave e comparação com a janela anterior</h3><p>{data["prior_start"]} a {data["prior_end"]}; as diferenças dependem também da cobertura da coleta.</p>{keywords}\'\n                      f\'<h3>Documentos e evidências da classificação</h3>{documents}<details><summary>Baixar tabelas CSV</summary><p>{downloads}</p></details></section>\')\n    warnings = "".join(f"<li>{esc(w)}</li>" for w in metadata["warnings"])\n    document = \'\'\'<!doctype html><html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Curadoria FAPESP — tabelas e gráficos</title><style>\n*{box-sizing:border-box}body{margin:0;background:#f4f6f9;color:#203146;font:15px/1.55 system-ui,sans-serif}main{max-width:1250px;margin:auto;padding:32px 24px}h1{font-size:32px;line-height:1.2}h2{font-size:24px}h3{margin-top:28px}a{color:#17677e}header,section{background:white;border:1px solid #e0e6ee;border-radius:14px;padding:28px;margin-bottom:24px}.eyebrow{color:#287d8e;font-weight:700;letter-spacing:.12em;text-transform:uppercase;font-size:12px}.cards{display:grid;grid-template-columns:repeat(4,1fr);gap:12px}.card{background:#eff6f7;border-radius:9px;padding:16px}.card strong{font-size:28px;display:block}.card span{font-size:13px}.table-wrap{overflow:auto;max-height:520px;border:1px solid #e0e6ee;border-radius:8px}table{border-collapse:collapse;width:100%;font-size:13px}th,td{padding:10px 12px;border-bottom:1px solid #e0e6ee;text-align:left;vertical-align:top;min-width:100px}td:first-child{min-width:250px}th{background:#edf2f7;position:sticky;top:0}tbody tr:hover{background:#f6f9fc}.charts{display:grid;grid-template-columns:1fr 1fr;gap:12px}figure{margin:18px 0 0}img{width:100%;height:auto}input{padding:12px;border:1px solid #a6b7c7;border-radius:7px;width:100%;font:inherit}.warning{background:#fff7e5;padding:14px;border-radius:8px}summary{cursor:pointer;color:#17677e}footer{font-size:13px;color:#526173}@media(max-width:800px){main{padding:16px 10px}header,section{padding:18px}.cards{grid-template-columns:1fr 1fr}.charts{grid-template-columns:1fr}}\n</style></head><body><main><header><div class="eyebrow">Pesquisa FAPESP · Governança digital</div><h1>Curadoria: notícias, temas e palavras-chave</h1>\'\'\'\n    document += f\'<p>Janela de publicação: <strong>{metadata["window_start"]} a {metadata["as_of"]}</strong> · horário de Brasília. Gerado em {metadata["generated_at"]}.</p>\'\n    scope = "títulos" if metadata["analysis_text"] == "title" else "títulos e notas editoriais fornecidas (não o texto integral)"\n    document += f\'<p>Os gráficos analisam os <strong>{scope}</strong>. “Candidata” significa correspondência lexical para leitura; a pertinência acadêmica requer revisão. Os cinco eixos são multilabel. Os corpus dos scrapers e do catálogo aparecem separadamente.</p>\'\n    document += \'<p><a href="painel_interativo.html">Abrir painel com filtros e gráficos interativos</a> · <a href="resumo_semanal.md">Resumo automático e sugestões de leitura</a></p>\'\n    document += \'<p>Palavras-chave usam um dicionário editável com sinônimos em português, inglês e espanhol. Coocorrência mostra termos no mesmo título e não demonstra influência regulatória.</p>\'\n    if warnings:\n        document += f\'<div class="warning"><strong>Limites da base</strong><ul>{warnings}</ul></div>\'\n    document += \'<p><label for="search">Buscar nos títulos, fontes, temas ou palavras-chave das tabelas de documentos</label></p><input id="search" type="search" placeholder="Ex.: LGPD, inteligência artificial, Mercosul…"></header>\'\n    document += "".join(blocks)\n    document += f\'<footer>Entradas e versões de configuração estão registradas em <a href="resumo.json">resumo.json</a>. A base é uma amostra das fontes monitoradas, sem pretensão de cobertura universal. Catálogo: metadados fornecidos no arquivo; o agente não verifica o conteúdo dos links.</footer>\'\n    document += \'\'\'</main><script>document.getElementById(\'search\').addEventListener(\'input\',e=>{const norm=s=>s.normalize(\'NFD\').replace(/[\\\\u0300-\\\\u036f]/g,\'\').toLowerCase();const q=norm(e.target.value);document.querySelectorAll(\'table.searchable tbody tr\').forEach(r=>{r.hidden=!norm(r.textContent).includes(q)})});</script></body></html>\'\'\'\n    (output / "index.html").write_text(document, encoding="utf-8")\n\n\ndef run(news: Path, output: Path, config_path: Path, as_of: date, days: int = 7,\n        catalog: Path | None = None, report_path: Path | None = None,\n        analysis_text: str = "title") -> dict:\n    if not 1 <= days <= 366:\n        raise ValueError("days deve estar entre 1 e 366")\n    config = load_config(config_path)\n    diagnostics, warnings = {}, []\n    if report_path and report_path.exists():\n        diagnostics = json.loads(report_path.read_text(encoding="utf-8"))\n        generated = seen_date(diagnostics.get("generated_at", ""))\n        if not generated:\n            warnings.append("Diagnóstico da coleta sem data válida.")\n        elif (as_of - date.fromisoformat(generated)).days > 1:\n            warnings.append(f"A última coleta registrada é de {generated}; a janela atual pode estar incompleta.")\n        for source in diagnostics.get("sources", []):\n            if source.get("status") != "ok":\n                warnings.append(f"Fonte {source[\'source_id\']}: {source.get(\'status\', \'unknown\')}; {source.get(\'error\', source.get(\'coverage_note\', \'cobertura incompleta\'))}")\n            elif source.get("coverage_note"):\n                warnings.append(f"{source[\'source_id\']}: {source[\'coverage_note\']}")\n    else:\n        warnings.append("Diagnóstico da coleta não fornecido; funcionamento e cobertura das fontes não confirmados.")\n    inputs = {"scrapers": news}\n    raw = {"scrapers": read_news(news)}\n    if catalog:\n        inputs["catalogo"] = catalog\n        raw["catalogo"] = read_catalog(catalog)\n    output.mkdir(parents=True, exist_ok=True)\n    corpus_data, image_paths, quality = {}, {}, {}\n    for corpus, rows in raw.items():\n        clean, stats = prepare(rows, config, analysis_text)\n        data = summarize(clean, config, as_of, days, diagnostics if corpus == "scrapers" else None)\n        stats.update(data["counts"])\n        quality[corpus] = stats\n        folder = output / corpus\n        folder.mkdir(parents=True, exist_ok=True)\n        save_tables(folder, data)\n        label = "Scrapers" if corpus == "scrapers" else "Catálogo"\n        image_paths[corpus] = charts(folder, data, label)\n        corpus_data[corpus] = data\n    if not catalog:\n        warnings.append("O catálogo da curadoria não foi fornecido: estas saídas cobrem somente as notícias dos scrapers.")\n    source_metadata = {corpus: {"path": str(path), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}\n                       for corpus, path in inputs.items()}\n    metadata = {"generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),\n                "as_of": as_of.isoformat(), "window_start": (as_of - timedelta(days=days - 1)).isoformat(),\n                "days": days, "timezone": "America/Sao_Paulo", "analysis_text": analysis_text,\n                "config_version": config["version"], "config_sha256": hashlib.sha256(config_path.read_bytes()).hexdigest(),\n                "inputs": source_metadata, "corpus": quality, "warnings": warnings,\n                "collection_generated_at": diagnostics.get("generated_at"),\n                "method": "Correspondências lexicais com sinônimos; multilabel; URL canônica deduplicada por corpus; publicação distinta de primeira detecção."}\n    render_report(output, corpus_data, image_paths, metadata)\n    # The offline interactive dashboard also supplies a weekly reading shortlist.\n    if str(Path(__file__).resolve().parent) not in sys.path:\n        sys.path.insert(0, str(Path(__file__).resolve().parent))\n    from curadoria_dashboard import export_dashboard\n    export_dashboard(output, corpus_data, config, metadata)\n    (output / "resumo.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    return metadata\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--news", type=Path, default=ROOT / "data/news.csv")\n    saved_catalog = ROOT / "data/curadoria/Catalogo_de_fontes_FAPESP.md"\n    parser.add_argument("--catalog", type=Path, default=saved_catalog if saved_catalog.exists() else None)\n    parser.add_argument("--collection-report", type=Path, default=ROOT / "data/runs/latest.json")\n    parser.add_argument("--config", type=Path, default=ROOT / "config/curadoria.json")\n    parser.add_argument("--output", type=Path, default=ROOT / "reports/curadoria/latest")\n    parser.add_argument("--as-of", type=date.fromisoformat, default=datetime.now(ZoneInfo("America/Sao_Paulo")).date())\n    parser.add_argument("--days", type=int, default=7)\n    parser.add_argument("--analysis-text", choices=("title", "title_summary"), default="title",\n                        help="title_summary inclui notas editoriais; não representa texto integral")\n    args = parser.parse_args()\n    metadata = run(args.news, args.output, args.config, args.as_of, args.days, args.catalog,\n                   args.collection_report, args.analysis_text)\n    print(f"Relatório: {args.output / \'index.html\'}")\n    for corpus, counts in metadata["corpus"].items():\n        print(f"{corpus}: {counts[\'documents\']} documentos, {counts[\'candidates\']} candidatas no histórico, {counts[\'candidates_window\']} na janela")\n    for warning in metadata["warnings"]:\n        print(f"ATENÇÃO: {warning}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'scripts/curadoria_dashboard.py': '"""Offline dashboard, reading shortlist and additional CSV import for the curator."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport html\nimport io\nimport json\nfrom collections import Counter\nfrom datetime import date, timedelta\nfrom itertools import combinations\nfrom pathlib import Path\n\nfrom curadoria_agent import contains, in_window, normalize, safe_url, write_csv\n\n\ndef import_csv(path: Path) -> list[dict]:\n    """Accept ordinary exports from the existing notebooks without modifying them."""\n    aliases = {\n        "title": ("title", "titulo", "manchete", "news_title"),\n        "url": ("url", "link", "permalink", "href"),\n        "published_date": ("published_date", "data", "date", "fecha", "data_publicacao", "data_de_publicacao", "published"),\n        "source_id": ("source_id", "fonte", "source", "origem", "instituicao"),\n        "summary": ("summary", "resumo", "descricao", "abstract"),\n    }\n    text = path.read_text(encoding="utf-8-sig")\n    try:\n        dialect = csv.Sniffer().sniff(text[:8192], delimiters=",;\\t")\n    except csv.Error:\n        dialect = csv.excel\n    reader = csv.DictReader(io.StringIO(text), dialect=dialect)\n    fields = {normalize(k).replace(" ", "_"): k for k in (reader.fieldnames or [])}\n    columns = {key: next((fields[a] for a in names if a in fields), None)\n               for key, names in aliases.items()}\n    if not columns["title"] or not columns["url"]:\n        raise ValueError(f"{path.name}: CSV precisa de colunas título e URL/link")\n    rows = []\n    for raw in reader:\n        row = {key: (raw.get(column) or "").strip() if column else ""\n               for key, column in columns.items()}\n        row["source_id"] = row["source_id"] or path.stem\n        row.update(publication_type="CSV importado", corpus="scrapers", manually_selected=False,\n                   first_seen="", last_seen="", input_file=path.name)\n        rows.append(row)\n    return rows\n\n\ndef merge_documents(corpus_data: dict) -> list[dict]:\n    """Deduplicate across inputs; keep scraper title and preserve catalog provenance."""\n    merged = {}\n    for corpus, data in corpus_data.items():\n        for raw in data["rows"]:\n            row = dict(raw, corpora=[corpus])\n            url = row["canonical_url"]\n            if url not in merged:\n                merged[url] = row\n                continue\n            old = merged[url]\n            old["corpora"] = sorted(set(old["corpora"]) | {corpus})\n            old["manually_selected"] = bool(old.get("manually_selected") or row.get("manually_selected"))\n            if corpus == "catalogo":\n                old["catalog_title"] = row["title"]\n                old["catalog_summary"] = row.get("summary", "")\n                old["catalog_source"] = row["source"]\n            # Do not merge keyword sets from different titles: evidence stays reproducible.\n    return list(merged.values())\n\n\ndef filter_documents(rows: list[dict], as_of: str, start: str = "", end: str = "",\n                     source: str = "", topic: str = "", keyword: str = "",\n                     query: str = "", include_uncertain: bool = True,\n                     candidates_only: bool = True) -> list[dict]:\n    if start and end and start > end:\n        raise ValueError("Data inicial deve ser anterior ou igual à final")\n    result = []\n    end = min(end, as_of) if end else as_of\n    for row in rows:\n        precision, published = row["date_precision"], row["published_date"]\n        if published and published > as_of:\n            continue\n        if candidates_only and not (row["candidate"] or row.get("manually_selected")):\n            continue\n        if source and row["source"] != source:\n            continue\n        if topic and topic not in row["topics"]:\n            continue\n        if keyword and keyword not in row["keywords"]:\n            continue\n        if query and normalize(query) not in normalize(" ".join([\n                row["title"], row["source"], " ".join(row["keywords"]), " ".join(row["topics"])])):\n            continue\n        if precision == "day":\n            if (start and published < start) or published > end:\n                continue\n        elif not include_uncertain:\n            continue\n        elif precision in {"month", "year"}:\n            size = 7 if precision == "month" else 4\n            if (start and published < start[:size]) or published > end[:size]:\n                continue\n        result.append(row)\n    return sorted(result, key=lambda r: (r["published_date"], r["title"]), reverse=True)\n\n\ndef shortlist(rows: list[dict], as_of: date, days: int, limit: int = 5) -> list[dict]:\n    start, end = (as_of - timedelta(days=days - 1)).isoformat(), as_of.isoformat()\n    pool = [dict(r) for r in rows if in_window(r, start, end)\n            and (r["candidate"] or r.get("manually_selected"))]\n    for row in pool:\n        row["priority_score"] = (min(5, len(row["keywords"])) + 2 * min(3, len(row["topics"]))\n                                 + 3 * bool(row.get("manually_selected")))\n    chosen, covered = [], set()\n    while pool and len(chosen) < limit:\n        row = max(pool, key=lambda r: (r["priority_score"] + 2 * len(set(r["topics"]) - covered),\n                                      r["published_date"], r["title"]))\n        pool.remove(row)\n        covered.update(row["topics"])\n        row["selection_reason"] = ("Termos identificados: " + (", ".join(row["keywords"]) or "nenhum no título")\n                                   + ". Eixos: " + (", ".join(row["topics"]) or "revisão temática necessária")\n                                   + (". Já selecionada no catálogo." if row.get("manually_selected") else "."))\n        chosen.append(row)\n    return chosen\n\n\ndef measures(rows: list[dict], config: dict, as_of: date, days: int = 7) -> dict:\n    """Counts always use documents; word occurrences are separately recorded."""\n    terms = Counter(k for r in rows for k in set(r["keywords"]))\n    topics = Counter(t for r in rows for t in set(r["topics"]))\n    sources = Counter(r["source"] for r in rows)\n    stop = set(normalize(config["stopwords"]).split())\n    words, occurrences = Counter(), Counter()\n    pairs = Counter()\n    for row in rows:\n        tokens = [w for w in normalize(row.get("analysis_text", row["title"])).split()\n                  if len(w) >= 3 and not w.isdigit() and w not in stop]\n        words.update(set(tokens))\n        occurrences.update(tokens)\n        pairs.update(combinations(sorted(set(row["keywords"])), 2))\n    start = (as_of - timedelta(days=days - 1)).isoformat()\n    old_start = (as_of - timedelta(days=2 * days - 1)).isoformat()\n    old_end = (as_of - timedelta(days=days)).isoformat()\n    comparison = []\n    for label in config["keywords"]:\n        current = sum(label in r["keywords"] and in_window(r, start, as_of.isoformat()) for r in rows)\n        previous = sum(label in r["keywords"] and in_window(r, old_start, old_end) for r in rows)\n        comparison.append(dict(palavra_chave=label, atual=current, anterior=previous, diferenca=current - previous))\n    return dict(\n        keywords=[dict(palavra_chave=k, documentos=n) for k, n in terms.most_common()],\n        topics=[dict(tema=k, documentos=topics[k]) for k in config["axes"]],\n        sources=[dict(fonte=k, documentos=n) for k, n in sources.most_common()],\n        words=[dict(palavra=k, documentos=n, ocorrencias=occurrences[k]) for k, n in words.most_common()],\n        pairs=[dict(termo_1=a, termo_2=b, documentos=n,\n                    jaccard=round(n / (terms[a] + terms[b] - n), 4))\n               for (a, b), n in pairs.most_common()],\n        comparison=comparison,\n        months=[dict(mes=k, documentos=n) for k, n in sorted(Counter(\n            r["published_date"][:7] for r in rows if r["date_precision"] in {"day", "month"}).items())],\n        days=[dict(data=k, documentos=n) for k, n in sorted(Counter(\n            r["published_date"] for r in rows if r["date_precision"] == "day").items())])\n\n\ndef notebook_figures(rows: list[dict], config: dict, as_of: date, days: int = 7,\n                     comparison_rows: list[dict] | None = None) -> dict:\n    import plotly.graph_objects as go\n    data, figures = measures(rows, config, as_of, days), {}\n\n    def bar(key, label, title, color="#237c83"):\n        selected = data[key][:15][::-1]\n        fig = go.Figure(go.Bar(x=[r["documentos"] for r in selected],\n                              y=[html.escape(r[label]) for r in selected], orientation="h",\n                              marker_color=color, text=[r["documentos"] for r in selected],\n                              textposition="auto"))\n        fig.update_layout(title=title, template="plotly_white", height=max(380, 24 * len(selected) + 170),\n                          xaxis_title="Documentos", xaxis=dict(\n                              rangemode="tozero", tickformat=",d", nticks=6,\n                              dtick=1 if max((r["documentos"] for r in selected), default=0) <= 10 else None),\n                          margin=dict(l=220, r=35, t=70, b=70))\n        return fig\n    figures["palavras_chave"] = bar("keywords", "palavra_chave", "Palavras-chave • uma contagem por documento")\n    figures["temas"] = bar("topics", "tema", "Eixos da pesquisa • um documento pode ter vários eixos")\n    figures["fontes"] = bar("sources", "fonte", "Publicações por fonte no recorte observado")\n    figures["palavras_livres"] = bar("words", "palavra", "Palavras frequentes • documentos com cada palavra")\n    monthly_counts = {r["mes"]: r["documentos"] for r in data["months"]}\n    month_labels = []\n    if monthly_counts:\n        cursor = date.fromisoformat(min(monthly_counts) + "-01")\n        last = max(monthly_counts)\n        while cursor.strftime("%Y-%m") <= last:\n            month_labels.append(cursor.strftime("%Y-%m"))\n            cursor = date(cursor.year + (cursor.month == 12), cursor.month % 12 + 1, 1)\n    monthly = go.Figure(go.Scatter(x=month_labels,\n                                  y=[monthly_counts.get(m) for m in month_labels], mode="lines+markers",\n                                  line_color="#237c83", connectgaps=False))\n    monthly.update_layout(title="Publicações por mês • meses não observados não são zeros verificados",\n                          template="plotly_white", yaxis_title="Documentos", yaxis=dict(rangemode="tozero"))\n    figures["evolucao"] = monthly\n    labels = [r["palavra_chave"] for r in data["keywords"][:10]]\n    matrix = [[sum(a in r["keywords"] and b in r["keywords"] for r in rows) for b in labels] for a in labels]\n    figures["coocorrencias"] = go.Figure(go.Heatmap(\n        z=matrix, x=labels, y=labels, colorscale="Blues", colorbar=dict(title="Documentos"),\n        text=matrix, texttemplate="%{text}", hovertemplate="%{y} + %{x}: %{z} documentos<extra></extra>"))\n    figures["coocorrencias"].update_layout(title="Coocorrência de termos • presença no mesmo documento",\n                                          template="plotly_white", height=620, margin=dict(l=170, b=170))\n    comparative = measures(comparison_rows, config, as_of, days)["comparison"] if comparison_rows is not None else data["comparison"]\n    comparison = sorted(comparative, key=lambda r: -(r["atual"] + r["anterior"]))[:12]\n    figures["comparacao"] = go.Figure([\n        go.Bar(name="Janela anterior", x=[r["palavra_chave"] for r in comparison],\n               y=[r["anterior"] for r in comparison], marker_color="#aabcc6"),\n        go.Bar(name="Janela atual", x=[r["palavra_chave"] for r in comparison],\n               y=[r["atual"] for r in comparison], marker_color="#237c83")])\n    figures["comparacao"].update_layout(title=f"Comparação de duas janelas de {days} dias",\n                                        template="plotly_white", barmode="group", height=470,\n                                        xaxis=dict(tickangle=-35), yaxis=dict(rangemode="tozero"))\n    return figures\n\n\ndef json_for_html(value: object) -> str:\n    return json.dumps(value, ensure_ascii=False, separators=(",", ":")).replace(\n        "<", r"\\u003c").replace(">", r"\\u003e").replace("&", r"\\u0026")\n\n\ndef export_dashboard(output: Path, corpus_data: dict, config: dict, metadata: dict) -> None:\n    from plotly.offline import get_plotlyjs\n    rows = merge_documents(corpus_data)\n    as_of = date.fromisoformat(metadata["as_of"])\n    observed = filter_documents(rows, metadata["as_of"])\n    chosen = shortlist(observed, as_of, metadata["days"])\n    write_csv(output / "selecionadas_semana.csv", chosen, [\n        "title", "source", "published_date", "url", "priority_score", "selection_reason", "keywords", "topics"])\n    data = measures(observed, config, as_of, metadata["days"])\n    write_csv(output / "coocorrencias_jaccard.csv", data["pairs"],\n              ["termo_1", "termo_2", "documentos", "jaccard"])\n    template = Path(__file__).with_name("curadoria_dashboard.html").read_text(encoding="utf-8")\n    payload = dict(documents=rows, corpora={k: d["rows"] for k, d in corpus_data.items()},\n                   config=config, metadata=metadata)\n    page = template.replace("__CURADORIA_DATA__", json_for_html(payload)).replace("__PLOTLY_JS__", get_plotlyjs())\n    (output / "painel_interativo.html").write_text(page, encoding="utf-8")\n    lines = [\n        "# Curadoria FAPESP — resumo automático",\n        "",\n        f"Janela: {metadata[\'window_start\']} a {metadata[\'as_of\']} (America/Sao_Paulo).",\n        f"Escopo textual: {metadata[\'analysis_text\']}. Regras lexicais editáveis; sem API de IA.",\n        "",\n        "## Base e cobertura",\n        "",\n    ]\n    for corpus, counts in metadata["corpus"].items():\n        lines.append(f"- {corpus}: {counts[\'documents\']} registros; {counts[\'published_window\']} publicações "\n                     f"datadas na janela; {counts[\'candidates_window\']} candidatas lexicais na janela.")\n    lines += ["", "## Até cinco publicações para leitura", "",\n              "Ordem: termos (até 5 pontos), eixos (2 por eixo, até 6), seleção prévia no catálogo (3) "\n              "e diversidade de eixos (2 por eixo ainda não coberto). A pontuação orienta a leitura.", ""]\n    if not chosen:\n        lines.append("Não há candidatas com data exata nesta janela. Não foram preenchidas vagas com notícias antigas.")\n    for index, row in enumerate(chosen, 1):\n        lines += [f"{index}. {row[\'title\']}", f"   - Publicação: {row[\'published_date\']} · {row[\'source\']}",\n                  f"   - Fonte: {row[\'url\']}", f"   - Critério: {row[\'selection_reason\']}", ""]\n    lines += ["", "## Cuidados de interpretação", "",\n              "- Volume observado depende da cobertura dos coletores. O corpus não representa toda a agenda pública.",\n              "- Coocorrência não demonstra causalidade, influência europeia ou mediação brasileira.",\n              "- Os textos completos não foram analisados. O modo title_summary também usa notas editoriais do catálogo.",\n              "- Datas apenas mensais ou anuais e datas ausentes não entram na seleção semanal.",\n              "- Todas as fontes do catálogo permanecem na base, mesmo sem correspondência lexical no título.", ""]\n    lines += ["- " + warning for warning in metadata["warnings"]]\n    (output / "resumo_semanal.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    metadata["interactive_dashboard"] = "painel_interativo.html"\n    metadata["shortlist_count"] = len(chosen)\n    metadata["combined_unique_documents"] = len(rows)\n    metadata["dashboard_sha256"] = hashlib.sha256(page.encode("utf-8")).hexdigest()\n', 'scripts/curadoria_dashboard.html': '<!doctype html>\n<html lang="pt-BR">\n<head>\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Observatório FAPESP · Curadoria semanal</title>\n<style>\n:root{--ink:#21394b;--muted:#5d7180;--teal:#237c83;--line:#dce5ea;--bg:#f2f6f8}\n*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.55 system-ui,sans-serif}\nmain{max-width:1480px;margin:auto;padding:30px}header{padding:34px;background:#153b4d;color:#fff;border-radius:18px}\n.eyebrow{text-transform:uppercase;letter-spacing:.14em;font-size:11px;color:#b9e5e2}h1{font-size:35px;margin:9px 0;line-height:1.2}h2{font-size:22px;margin:0 0 10px}h3{margin:0 0 8px;font-size:18px}p{margin:8px 0}.muted,.note{color:var(--muted);font-size:13px}\nsection{background:#fff;border:1px solid var(--line);border-radius:13px;padding:22px;margin-top:22px}\n.controls{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:15px}label{display:block;font-size:12px;font-weight:650;margin-bottom:5px}\nselect,input{width:100%;padding:10px;border:1px solid #b9ccd4;border-radius:7px;background:white;color:var(--ink);font:inherit;font-size:13px}\n.check{display:flex;align-items:center;gap:7px;margin-top:20px}.check input{width:auto}.check label{margin:0;font-size:13px;font-weight:400}\n.cards{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-top:18px}.card{background:#edf5f6;border:1px solid #dce9eb;border-radius:10px;padding:15px}.card b{font-size:31px;display:block}.card span{font-size:12px}\n.grid{display:grid;grid-template-columns:1fr 1fr;gap:20px}.panel{min-width:0}.chart{width:100%;min-height:390px}.warning{background:#fff8e8;border:1px solid #eee1bd;border-radius:9px;padding:13px;margin-top:16px;font-size:13px}\n.table-wrap{overflow:auto;max-height:600px;border:1px solid var(--line);border-radius:9px}table{border-collapse:collapse;width:100%;font-size:12px}th{background:#edf4f6;position:sticky;top:0;text-align:left}th,td{padding:12px;vertical-align:top;border-bottom:1px solid var(--line)}td:first-child{min-width:270px}td{min-width:130px}a{color:#176c81}tbody tr:hover{background:#f5f9fa}\nbutton{border:0;padding:11px 16px;border-radius:7px;background:var(--teal);color:white;font:inherit;cursor:pointer;font-size:13px}.toolbar{display:flex;justify-content:space-between;align-items:center;gap:12px;margin:13px 0}\n.reading{display:grid;gap:12px}.reading article{border-left:4px solid var(--teal);background:#f4f8fa;padding:15px;border-radius:6px}.reading a{font-weight:650}.pill{display:inline-block;background:#e4f0f1;border-radius:5px;padding:3px 6px;font-size:11px;margin:2px}\nfooter{font-size:12px;color:var(--muted);margin:20px 0}details summary{cursor:pointer;font-weight:600}#busy{font-size:12px}\n@media(max-width:1050px){.controls{grid-template-columns:repeat(2,1fr)}.grid{grid-template-columns:1fr}}\n@media(max-width:620px){main{padding:12px}header{padding:22px}h1{font-size:27px}section{padding:16px}.cards{grid-template-columns:1fr 1fr}.controls{grid-template-columns:1fr}.toolbar{align-items:flex-start;flex-direction:column}}\n</style>\n</head>\n<body>\n<main>\n<header><div class="eyebrow">Pesquisa FAPESP · Brasil e governança digital</div>\n<h1>Curadoria semanal</h1><p>Notícias, fontes, palavras-chave e relações entre temas.</p>\n<p id="reference"></p><p id="scope"></p></header>\n<section>\n<h2>Explore a base</h2><p class="muted">Os filtros atualizam as tabelas, os gráficos e as sugestões de leitura. O arquivo funciona sem conexão à internet.</p>\n<div class="controls">\n<div><label for="period">Período de publicação</label><select id="period"><option value="all">Histórico até a data de referência</option><option value="7">Últimos 7 dias</option><option value="30">Últimos 30 dias</option><option value="90">Últimos 90 dias</option><option value="custom">Intervalo personalizado</option></select></div>\n<div><label for="corpus">Conjunto de fontes</label><select id="corpus"><option value="todos">Scrapers e catálogo, sem duplicar URLs</option><option value="scrapers">Notícias dos scrapers</option><option value="catalogo">Catálogo da curadoria</option></select></div>\n<div><label for="start">Publicado desde</label><input id="start" type="date"></div>\n<div><label for="end">Publicado até</label><input id="end" type="date"></div>\n<div><label for="source">Fonte</label><select id="source"></select></div>\n<div><label for="topic">Eixo da pesquisa</label><select id="topic"></select></div>\n<div><label for="keyword">Palavra-chave</label><select id="keyword"></select></div>\n<div><label for="query">Buscar no título, fonte ou termos</label><input id="query" type="search" placeholder="Ex.: LGPD, Mercosul, privacidade"></div>\n</div>\n<div class="controls">\n<div class="check"><input id="candidates" type="checkbox" checked><label for="candidates">Candidatas lexicais ou já selecionadas no catálogo</label></div>\n<div class="check"><input id="uncertain" type="checkbox" checked><label for="uncertain">Incluir datas imprecisas como contexto</label></div>\n<div><label for="metric">Métrica de coocorrência</label><select id="metric"><option value="count">Número de documentos</option><option value="jaccard">Jaccard: proporção da união dos termos</option></select></div>\n<div class="check"><button id="reset" type="button">Limpar filtros</button><span id="busy" aria-live="polite"></span></div>\n</div>\n<div class="cards">\n<div class="card"><b id="n-docs">0</b><span>Documentos no recorte</span></div>\n<div class="card"><b id="n-sources">0</b><span>Fontes observadas</span></div>\n<div class="card"><b id="n-terms">0</b><span>Palavras-chave identificadas</span></div>\n<div class="card"><b id="n-uncertain">0</b><span>Sem data exata no recorte</span></div>\n</div><p id="summary"></p><div id="warnings" class="warning"></div>\n</section>\n<div class="grid">\n<section class="panel"><h2>Palavras-chave</h2><p class="note">Até 15 termos. Sinônimos em português, inglês e espanhol; cada termo conta uma vez por documento.</p><div id="chart-keywords" class="chart"></div></section>\n<section class="panel"><h2>Eixos da pesquisa</h2><p class="note">Um documento pode integrar vários eixos. As barras não somam o total de publicações.</p><div id="chart-topics" class="chart"></div></section>\n<section class="panel"><h2>Fontes monitoradas</h2><p class="note">Até 12 fontes. Volume nesta amostra, sujeito à cobertura e ao método dos coletores.</p><div id="chart-sources" class="chart"></div></section>\n<section class="panel"><h2>Evolução das publicações</h2><p class="note">Datas com dia ou mês. Meses sem registros são lacunas, sem confirmação de ausência de publicações.</p><div id="chart-months" class="chart"></div></section>\n<section class="panel"><h2>Termos que aparecem juntos</h2><p class="note">Coocorrência no texto analisado. A diagonal é a frequência do próprio termo; Jaccard normaliza pela união dos documentos.</p><div id="chart-pairs" class="chart"></div></section>\n<section class="panel"><h2>Palavras frequentes</h2><p class="note">Até 15 palavras após remover palavras funcionais. Valores indicam documentos, não repetições no mesmo texto.</p><div id="chart-words" class="chart"></div></section>\n<section class="panel"><h2>Palavras-chave por fonte</h2><p class="note">Percentual dos documentos filtrados de cada fonte com o termo. Não mede prevalência no universo de notícias.</p><div id="chart-source-terms" class="chart"></div></section>\n<section class="panel"><h2>Comparação entre períodos</h2><p id="comparison-note" class="note"></p><div id="chart-comparison" class="chart"></div></section>\n</div>\n<section><h2>Até cinco publicações para leitura</h2><p id="reading-note" class="note"></p><div id="reading" class="reading"></div></section>\n<section><h2>Documentos e fontes</h2><div class="toolbar"><p id="table-note" class="muted"></p><button id="download" type="button">Baixar o recorte em CSV</button></div>\n<div class="table-wrap"><table id="documents"><thead><tr><th>Título e link</th><th>Fonte</th><th>Publicação</th><th>Eixos e palavras-chave</th><th>Termos encontrados</th><th>Contexto do catálogo</th></tr></thead><tbody></tbody></table></div></section>\n<section><details><summary>Como interpretar esta análise</summary>\n<p>“Candidata” indica correspondência com o dicionário da pesquisa. A seleção prévia no catálogo é preservada mesmo sem palavras-chave no título. A confirmação de pertinência exige leitura da fonte.</p>\n<p>O modo padrão analisa os títulos. O modo com contexto acrescenta resumos ou notas editoriais fornecidas, não o texto integral das publicações. As frequências desse segundo modo também refletem a linguagem das notas.</p>\n<p>Data de publicação e primeira detecção são distintas. Datas imprecisas podem aparecer no histórico, mas não na seleção semanal nem na comparação de janelas exatas.</p>\n<p>A relação entre termos não demonstra causalidade, difusão do GDPR ou atuação do Brasil como ponte regulatória. Essas relações são perguntas para investigação documental.</p>\n<p>A ordem de leitura atribui até 5 pontos por termos, até 6 por eixos, 3 por seleção prévia e 2 por eixo ainda não coberto nas sugestões. A pontuação é uma regra de triagem, sem avaliação automática da qualidade científica.</p>\n</details></section><footer id="footer"></footer>\n</main>\n<script id="payload" type="application/json">__CURADORIA_DATA__</script>\n<script>__PLOTLY_JS__</script>\n<script>\n"use strict";\nconst D=JSON.parse(document.getElementById("payload").textContent);\nconst ref=D.metadata.as_of, windowDays=D.metadata.days, WSTART=D.metadata.window_start;\nconst $=id=>document.getElementById(id);\nconst norm=s=>String(s||"").normalize("NFD").replace(/[\\u0300-\\u036f]/g,"").toLowerCase().replace(/[^\\p{L}\\p{N}]+/gu," ").trim().replace(/\\s+/g," ");\nconst esc=s=>String(s).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;").replace(/"/g,"&quot;");\nconst isoShift=(s,n)=>{let x=new Date(s+"T12:00:00Z");x.setUTCDate(x.getUTCDate()+n);return x.toISOString().slice(0,10)};\nconst oldEnd=isoShift(ref,-windowDays),oldStart=isoShift(ref,-2*windowDays+1);\nconst stop=new Set(norm(D.config.stopwords).split(" "));\nconst rank=m=>Object.entries(m).sort((a,b)=>b[1]-a[1]||a[0].localeCompare(b[0]));\nconst count=(rows,fn)=>{let c={};rows.forEach(r=>Array.from(new Set(fn(r))).forEach(k=>{c[k]=(c[k]||0)+1}));return c};\nconst exact=(r,a,b)=>r.date_precision==="day"&&r.published_date>=a&&r.published_date<=b;\nconst periodLabel=s=>s.split("-").reverse().join("/");\nfunction options(id,values){const el=$(id);el.replaceChildren();const all=document.createElement("option");all.value="";all.textContent="Todas";el.append(all);values.forEach(v=>{const o=document.createElement("option");o.value=v;o.textContent=v;el.append(o)})}\noptions("source",[...new Set(D.documents.map(r=>r.source))].sort());\noptions("topic",Object.keys(D.config.axes));options("keyword",Object.keys(D.config.keywords));\n$("end").value=ref;$("end").max=ref;$("start").max=ref;\n$("reference").textContent="Data de referência: "+periodLabel(ref)+" · Brasília · comparação de janelas de "+windowDays+" dias";\n$("scope").textContent=D.metadata.analysis_text==="title"?"Análise dos títulos · dicionário editável · fontes rastreáveis":"Análise de títulos e notas editoriais · sem leitura automática do texto integral";\n$("comparison-note").textContent=periodLabel(oldStart)+"–"+periodLabel(oldEnd)+" versus "+periodLabel(WSTART)+"–"+periodLabel(ref)+". Usa os mesmos filtros temáticos e de fonte; o filtro de datas é desconsiderado apenas nesta comparação.";\n$("reading-note").textContent="Publicadas de "+periodLabel(WSTART)+" a "+periodLabel(ref)+". A lista respeita os filtros e pode ter menos de cinco itens. Requer leitura e avaliação acadêmica.";\n$("footer").textContent="Gerado em "+D.metadata.generated_at+" · entradas, configuração e hashes no resumo.json · catálogo e scrapers preservados.";\nconst warningList=document.createElement("ul");\nD.metadata.warnings.forEach(w=>{const li=document.createElement("li");li.textContent=w;warningList.append(li)});\nconst warningIntro=document.createElement("strong");warningIntro.textContent="Cobertura observada";\n$("warnings").append(warningIntro,warningList);\nfunction dataset(){return $("corpus").value==="todos"?D.documents:(D.corpora[$("corpus").value]||[])}\nfunction selected(ignoreDates=false){\n const start=$("start").value,end=$("end").value||ref,source=$("source").value,topic=$("topic").value,keyword=$("keyword").value,q=norm($("query").value);\n if(!ignoreDates&&start&&start>end)return [];\n return dataset().filter(r=>{\n   if(r.published_date&&r.published_date>ref)return false;\n   if($("candidates").checked&&!r.candidate&&!r.manually_selected)return false;\n   if(source&&r.source!==source)return false;if(topic&&!r.topics.includes(topic))return false;\n   if(keyword&&!r.keywords.includes(keyword))return false;\n   if(q&&!norm([r.title,r.source,...r.keywords,...r.topics].join(" ")).includes(q))return false;\n   if(ignoreDates)return true;\n   if(r.date_precision==="day")return (!start||r.published_date>=start)&&r.published_date<=end;\n   if(!$("uncertain").checked)return false;\n   if(["month","year"].includes(r.date_precision)){const size=r.date_precision==="month"?7:4;\n     return (!start||r.published_date>=start.slice(0,size))&&r.published_date<=end.slice(0,size)}\n   return true;\n }).sort((a,b)=>b.published_date.localeCompare(a.published_date)||a.title.localeCompare(b.title));\n}\nconst plotConfig={responsive:true,displaylogo:false,toImageButtonOptions:{format:"png",scale:2}};\nfunction layout(height=440){return {paper_bgcolor:"#fff",plot_bgcolor:"#fff",font:{family:"Arial, sans-serif",size:11,color:"#21394b"},height:height,margin:{l:35,r:25,t:30,b:65},xaxis:{automargin:true},yaxis:{automargin:true,zeroline:false,gridcolor:"#edf2f4"}}}\nfunction wrap(s,width=27){let lines=[],line="";String(s).split(" ").forEach(w=>{if(line&&line.length+w.length+1>width){lines.push(line);line=w}else{line+=(line?" ":"")+w}});if(line)lines.push(line);return lines.map(esc).join("<br>")}\nfunction bar(id,pairs,limit=15){const r=pairs.slice(0,limit).reverse(),labels=r.map(x=>wrap(x[0])),counts=r.map(x=>x[1]);\n let l=layout(Math.max(400,r.length*36+100));l.margin.l=170;l.xaxis={title:"Documentos",rangemode:"tozero",tickformat:",d",gridcolor:"#edf2f4",automargin:true};\n if(!r.length)l.annotations=[{text:"Sem registros neste recorte",showarrow:false,x:.5,y:.5,xref:"paper",yref:"paper"}];\n return Plotly.react(id,[{type:"bar",orientation:"h",y:labels,x:counts,marker:{color:"#237c83"},text:counts,textposition:"auto",hovertemplate:"%{y}<br>%{x} documentos<extra></extra>"}],l,plotConfig)}\nfunction readList(rows){\n const pool=rows.filter(r=>exact(r,WSTART,ref)&&(r.candidate||r.manually_selected)).map(r=>Object.assign({},r,{score:Math.min(5,r.keywords.length)+2*Math.min(3,r.topics.length)+(r.manually_selected?3:0)}));\n const chosen=[],covered=new Set();while(pool.length&&chosen.length<5){\n   pool.sort((a,b)=>(b.score+2*b.topics.filter(t=>!covered.has(t)).length)-(a.score+2*a.topics.filter(t=>!covered.has(t)).length)||b.published_date.localeCompare(a.published_date)||b.title.localeCompare(a.title));\n   const row=pool.shift();row.topics.forEach(t=>covered.add(t));chosen.push(row)}\n const out=$("reading");out.replaceChildren();\n if(!chosen.length){const p=document.createElement("p");p.textContent="Nenhuma candidata com data exata nesta janela e nestes filtros.";out.append(p)}\n chosen.forEach((r,i)=>{const article=document.createElement("article"),a=document.createElement("a"),p=document.createElement("p"),why=document.createElement("p");\n a.textContent=(i+1)+". "+r.title;a.href=r.url;a.target="_blank";a.rel="noopener noreferrer";\n p.className="muted";p.textContent=periodLabel(r.published_date)+" · "+r.source+" · pontuação base "+r.score;\n why.textContent="Termos: "+(r.keywords.join(", ")||"nenhum no título")+". Eixos: "+(r.topics.join(", ")||"revisão necessária")+(r.manually_selected?". Selecionada no catálogo.":".");\n article.append(a,p,why);out.append(article)});\n}\nfunction documents(rows){\n const tbody=$("documents").querySelector("tbody");tbody.replaceChildren();\n rows.forEach(r=>{let tr=document.createElement("tr");\n const values=[r.title,r.source,r.published_date||"Não informada",r.topics.join("; ")+"\\n"+r.keywords.join("; "),r.match_evidence||"Sem correspondência lexical",r.catalog_summary||r.summary||""];\n values.forEach((v,i)=>{const td=document.createElement("td");if(i===0){const a=document.createElement("a");a.textContent=v;a.href=r.url;a.target="_blank";a.rel="noopener noreferrer";td.append(a)}else{td.textContent=v;td.style.whiteSpace="pre-line"}\n if(i===2){const note=document.createElement("div");note.className="muted";note.textContent="Precisão: "+({day:"dia",month:"mês",year:"ano",unknown:"desconhecida"}[r.date_precision]||r.date_precision);td.append(note)}tr.append(td)});tbody.append(tr)});\n if(!rows.length){let tr=document.createElement("tr"),td=document.createElement("td");td.colSpan=6;td.textContent="Sem documentos neste recorte.";tr.append(td);tbody.append(tr)}\n $("table-note").textContent=rows.length+" documentos · links das fontes originais · uma linha por URL no conjunto selecionado";\n}\nasync function render(){\n $("busy").textContent="Atualizando…";const rows=selected(),base=selected(true);\n window.curadoriaView=rows;\n const terms=count(rows,r=>r.keywords),topics=count(rows,r=>r.topics),sources=count(rows,r=>[r.source]);\n const words=count(rows,r=>norm(r.analysis_text||r.title).split(" ").filter(w=>w.length>=3&&!/^\\d+$/.test(w)&&!stop.has(w)));\n $("n-docs").textContent=rows.length;$("n-sources").textContent=Object.keys(sources).length;\n $("n-terms").textContent=Object.keys(terms).length;$("n-uncertain").textContent=rows.filter(r=>r.date_precision!=="day").length;\n const top=rank(terms).slice(0,3).map(t=>t[0]+" ("+t[1]+")").join(", ");\n $("summary").textContent=rows.length?"Termos mais presentes no recorte: "+(top||"sem correspondências no dicionário")+". A frequência descreve a amostra observada.":"Nenhum registro atende a estes filtros.";\n const jobs=[bar("chart-keywords",rank(terms)),bar("chart-topics",Object.keys(D.config.axes).map(k=>[k,topics[k]||0])),bar("chart-sources",rank(sources),12),bar("chart-words",rank(words))];\n const monthCounts=count(rows.filter(r=>["day","month"].includes(r.date_precision)),r=>[r.published_date.slice(0,7)]),months=Object.keys(monthCounts).sort();\n let mx=[],my=[];if(months.length){let d=new Date(months[0]+"-01T12:00:00Z");const last=months[months.length-1];\n   while(d.toISOString().slice(0,7)<=last){let s=d.toISOString().slice(0,7);mx.push(s);my.push(monthCounts[s]||null);d.setUTCMonth(d.getUTCMonth()+1)}}\n let ml=layout();ml.yaxis.title="Documentos";ml.yaxis.rangemode="tozero";\n jobs.push(Plotly.react("chart-months",[{type:"scatter",mode:"lines+markers",x:mx,y:my,connectgaps:false,line:{color:"#237c83"},hovertemplate:"%{x}: %{y} documentos<extra></extra>"}],ml,plotConfig));\n const keys=rank(terms).slice(0,10).map(x=>x[0]),metric=$("metric").value;\n const z=keys.map(a=>keys.map(b=>{const n=rows.filter(r=>r.keywords.includes(a)&&r.keywords.includes(b)).length;return metric==="count"?n:(terms[a]+terms[b]-n?n/(terms[a]+terms[b]-n):0)}));\n let hl=layout(590);hl.margin={l:160,r:30,t:25,b:150};hl.xaxis.tickangle=-45;\n jobs.push(Plotly.react("chart-pairs",keys.length?[{type:"heatmap",z:z,x:keys.map(esc),y:keys.map(esc),colorscale:"Blues",zmin:0,zmax:metric==="jaccard"?1:undefined,text:z.map(r=>r.map(v=>metric==="count"?String(v):v.toFixed(2))),texttemplate:"%{text}",colorbar:{title:{text:metric==="count"?"Docs":"Jaccard"}},hovertemplate:"%{y} + %{x}: %{z}<extra></extra>"}]:[],hl,plotConfig));\n const src=rank(sources).slice(0,10).map(x=>x[0]),pct=src.map(s=>keys.map(k=>100*rows.filter(r=>r.source===s&&r.keywords.includes(k)).length/sources[s]));\n let sl=layout(Math.max(480,src.length*48+170));sl.margin={l:190,r:40,t:25,b:130};sl.xaxis.tickangle=-45;\n jobs.push(Plotly.react("chart-source-terms",src.length&&keys.length?[{type:"heatmap",z:pct,x:keys.map(esc),y:src.map(s=>wrap(s)),colorscale:"Blues",zmin:0,zmax:100,text:pct.map(r=>r.map(v=>v.toFixed(0)+"%")),texttemplate:"%{text}",customdata:src.map(s=>keys.map(()=>sources[s])),colorbar:{title:{text:"%"}},hovertemplate:"%{y}<br>%{x}: %{z:.1f}%<br>Base: %{customdata} documentos<extra></extra>"}]:[],sl,plotConfig));\n const cmp=Object.keys(D.config.keywords).map(k=>({term:k,current:base.filter(r=>exact(r,WSTART,ref)&&r.keywords.includes(k)).length,prior:base.filter(r=>exact(r,oldStart,oldEnd)&&r.keywords.includes(k)).length})).sort((a,b)=>(b.current+b.prior)-(a.current+a.prior)||a.term.localeCompare(b.term));\n window.curadoriaComparison=cmp;const shown=cmp.filter(r=>r.current+r.prior>0).slice(0,12);let cl=layout(490);cl.xaxis.tickangle=-40;cl.yaxis.title="Documentos";cl.yaxis.rangemode="tozero";cl.barmode="group";cl.legend={orientation:"h",y:1.15};\n jobs.push(Plotly.react("chart-comparison",[{type:"bar",name:"Janela anterior",x:shown.map(r=>esc(r.term)),y:shown.map(r=>r.prior),marker:{color:"#b0c0cb"}},{type:"bar",name:"Janela atual",x:shown.map(r=>esc(r.term)),y:shown.map(r=>r.current),marker:{color:"#237c83"}}],cl,plotConfig));\n readList(rows);documents(rows);await Promise.all(jobs);$("busy").textContent="Atualizado";\n}\nfunction changePeriod(){const value=$("period").value;if(value==="custom")return;\n $("end").value=ref;$("start").value=value==="all"?"":isoShift(ref,1-Number(value));$("uncertain").checked=value==="all";render()}\n$("period").addEventListener("change",changePeriod);\n["corpus","source","topic","keyword","uncertain","candidates","metric"].forEach(id=>$(id).addEventListener("change",render));\n["start","end"].forEach(id=>$(id).addEventListener("change",()=>{$("period").value="custom";render()}));\nlet timer;$("query").addEventListener("input",()=>{clearTimeout(timer);timer=setTimeout(render,150)});\n$("reset").addEventListener("click",()=>{["source","topic","keyword","query","start"].forEach(id=>$(id).value="");$("corpus").value="todos";$("period").value="all";$("end").value=ref;$("candidates").checked=true;$("uncertain").checked=true;render()});\nfunction csvCell(v){let s=Array.isArray(v)?v.join("; "):String(v==null?"":v);if(/^\\s*[=+\\-@]/.test(s))s="\'"+s;return \'"\'+s.replace(/"/g,\'""\')+\'"\'}\n$("download").addEventListener("click",()=>{const fields=["title","source","published_date","date_precision","url","topics","keywords","match_evidence","summary"],lines=[fields.map(csvCell).join(",")];\n selected().forEach(r=>lines.push(fields.map(f=>csvCell(r[f])).join(",")));const blob=new Blob(["\\ufeff"+lines.join("\\r\\n")],{type:"text/csv;charset=utf-8"}),a=document.createElement("a"),u=URL.createObjectURL(blob);a.href=u;a.download="curadoria_recorte.csv";a.click();setTimeout(()=>URL.revokeObjectURL(u),1000)});\nrender().catch(e=>{$("busy").textContent="Erro: "+e.message;console.error(e)});\n</script>\n</body>\n</html>\n', 'scripts/fapesp_weekly.py': '"""Collect recent public news and check the existing FAPESP notebooks.\n\nThe CSV is cumulative. A failed source never deletes previously collected rows.\nThis script does not execute Colab notebooks or claim that they run end to end.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport re\nfrom datetime import datetime, timedelta, timezone\nfrom email.utils import parsedate_to_datetime\nfrom pathlib import Path\nfrom urllib.parse import urljoin, urlsplit, urlunsplit\nfrom xml.etree import ElementTree\n\nimport requests\nfrom bs4 import BeautifulSoup\nfrom IPython.core.interactiveshell import InteractiveShell\nfrom requests.adapters import HTTPAdapter\nfrom urllib3.util.retry import Retry\n\n\nROOT = Path(__file__).resolve().parents[1]\nFIELDS = ("source_id", "title", "published_date", "published_raw", "url", "first_seen", "last_seen")\nSOURCES = ("nic", "edpb", "mercociudades", "senado_federal")\nHEADERS = {"User-Agent": "Mozilla/5.0 (compatible; FAPESP research news monitor; +https://github.com/anamacao/FAPESP-PIBIC-scrapping)"}\nMONTHS = {\n    "janeiro": 1, "fevereiro": 2, "março": 3, "marco": 3, "abril": 4,\n    "maio": 5, "junho": 6, "julho": 7, "agosto": 8, "setembro": 9,\n    "outubro": 10, "novembro": 11, "dezembro": 12,\n    "enero": 1, "febrero": 2, "marzo": 3, "mayo": 5, "junio": 6,\n    "julio": 7, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12,\n}\n\n\ndef session() -> requests.Session:\n    client = requests.Session()\n    client.headers.update(HEADERS)\n    retry = Retry(total=2, backoff_factor=0.5, status_forcelist=(429, 500, 502, 503, 504))\n    client.mount("https://", HTTPAdapter(max_retries=retry))\n    return client\n\n\ndef fetch(client: requests.Session, url: str) -> requests.Response:\n    response = client.get(url, timeout=25)\n    response.raise_for_status()\n    return response\n\n\ndef date_iso(raw: str) -> str:\n    raw = re.sub(r"\\s+", " ", raw or "").strip()\n    match = re.search(r"\\b(\\d{4}-\\d{2}-\\d{2})(?:\\b|T)", raw)\n    if match:\n        try:\n            return datetime.strptime(match.group(1), "%Y-%m-%d").date().isoformat()\n        except ValueError:\n            return ""\n    match = re.search(r"\\b(\\d{1,2})[/.](\\d{1,2})[/.](\\d{4})\\b", raw)\n    if match:\n        try:\n            return datetime(int(match.group(3)), int(match.group(2)), int(match.group(1))).date().isoformat()\n        except ValueError:\n            return ""\n    match = re.search(r"\\b(\\d{1,2})\\s+de\\s+([a-zç]+)\\s+de\\s+(\\d{4})\\b", raw.lower())\n    if match and match.group(2) in MONTHS:\n        try:\n            return datetime(int(match.group(3)), MONTHS[match.group(2)], int(match.group(1))).date().isoformat()\n        except ValueError:\n            return ""\n    for fmt in ("%d %B %Y", "%B %d, %Y", "%d %b %Y"):\n        try:\n            return datetime.strptime(raw, fmt).date().isoformat()\n        except ValueError:\n            pass\n    return ""\n\n\ndef record(source_id: str, title: str, raw_date: str, url: str, base: str) -> dict | None:\n    title = re.sub(r"\\s+", " ", BeautifulSoup(title or "", "html.parser").get_text(" ", strip=True)).strip()\n    absolute = urljoin(base, url or "")\n    parsed = urlsplit(absolute)\n    if not title or not parsed.hostname or parsed.scheme != "https":\n        return None\n    canonical = urlunsplit((parsed.scheme, parsed.netloc, parsed.path, parsed.query, ""))\n    return {"source_id": source_id, "title": title, "published_date": date_iso(raw_date),\n            "published_raw": (raw_date or "").strip()[:100], "url": canonical}\n\n\ndef collect_nic(client: requests.Session) -> list[dict]:\n    base = "https://www.nic.br"\n    data = fetch(client, base + "/noticias/indice/todos/indice-1.json").json()\n    if not isinstance(data, list):\n        raise ValueError("O índice NIC.br não retornou uma lista JSON")\n    return [item for news in data[:600]\n            if (item := record("nic", news.get("titulo"), news.get("data"), news.get("permalink"), base))]\n\n\ndef collect_edpb(client: requests.Session) -> list[dict]:\n    base = "https://www.edpb.europa.eu/news_en?type%5B1%5D=1"\n    soup = BeautifulSoup(fetch(client, base).text, "html.parser")\n    rows = []\n    for card in soup.select(".node-article-card"):\n        title = card.select_one(".node-article-card__title")\n        link = card.select_one("a.node-article-card__link[href]")\n        if not title or not link:\n            continue\n        date = card.select_one("time, .views-field-created, .date-display-single")\n        raw = date.get("datetime") or date.get_text(" ", strip=True) if date else ""\n        item = record("edpb", title.get_text(" ", strip=True), raw, link["href"], base)\n        if item:\n            rows.append(item)\n    return rows\n\n\ndef collect_mercociudades(client: requests.Session) -> list[dict]:\n    base = "https://mercociudades.org/pt-br/noticias/"\n    api = ("https://mercociudades.org/wp-json/wp/v2/posts"\n           "?per_page=50&lang=pt-br&_fields=date,link,title")\n    try:\n        posts = fetch(client, api).json()\n    except requests.HTTPError as exc:\n        if exc.response is None or exc.response.status_code != 403:\n            raise\n        feed = "https://mercociudades.org/feed/"\n        try:\n            root = ElementTree.fromstring(fetch(client, feed).text)\n        except ElementTree.ParseError as parse_error:\n            raise ValueError("O RSS oficial de Mercociudades não contém XML válido") from parse_error\n        rows = []\n        dates = []\n        for entry in root.findall("./channel/item"):\n            raw = entry.findtext("pubDate", "")\n            try:\n                published = parsedate_to_datetime(raw).date().isoformat()\n            except (TypeError, ValueError, IndexError):\n                continue\n            link = entry.findtext("link", "")\n            if urlsplit(link).hostname != "mercociudades.org":\n                continue\n            item = record("mercociudades", entry.findtext("title", ""), published, link, feed)\n            if item:\n                rows.append(item)\n                dates.append(published)\n        if not rows:\n            raise ValueError("O RSS oficial de Mercociudades não retornou notícias datadas")\n        complete_week = min(dates) <= (datetime.now(timezone.utc) - timedelta(days=7)).date().isoformat()\n        return CollectedRows(rows, method="rss", complete_week=complete_week)\n    if not isinstance(posts, list):\n        raise ValueError("A API pública de Mercociudades não retornou uma lista")\n    rows = []\n    for post in posts:\n        title = post.get("title", {}).get("rendered", "")\n        item = record("mercociudades", title, post.get("date", ""), post.get("link", ""), base)\n        if item:\n            rows.append(item)\n    return rows\n\n\nclass CollectedRows(list):\n    """Attach source coverage to records without changing the CSV schema."""\n\n    def __init__(self, rows: list[dict], method: str, complete_week: bool):\n        super().__init__(rows)\n        self.method = method\n        self.complete_week = complete_week\n\n\ndef collect_senado_federal(client: requests.Session) -> list[dict]:\n    base = "https://www12.senado.leg.br/noticias/ultimas"\n    soup = BeautifulSoup(fetch(client, base).text, "html.parser")\n    rows = []\n    for article in soup.select("ol.lista-resultados > li"):\n        link = article.select_one(\'a[href*="/noticias/materias/"]\')\n        if not link:\n            continue\n        date = article.select_one(".text-muted.normalis.hidden-xs, .text-muted.normalis-xs")\n        item = record("senado_federal", link.get_text(" ", strip=True),\n                      date.get_text(" ", strip=True) if date else "", link["href"], base)\n        if item:\n            rows.append(item)\n    return rows\n\n\nCOLLECTORS = {\n    "nic": collect_nic, "edpb": collect_edpb,\n    "mercociudades": collect_mercociudades, "senado_federal": collect_senado_federal,\n}\n\n\ndef check_notebooks() -> dict:\n    shell = InteractiveShell.instance()\n    results = []\n    for path in sorted(ROOT.rglob("*.ipynb")):\n        relative = path.relative_to(ROOT).as_posix()\n        errors = []\n        code_count = 0\n        try:\n            notebook = json.loads(path.read_text(encoding="utf-8"))\n            for index, cell in enumerate(notebook["cells"]):\n                if cell.get("cell_type") != "code":\n                    continue\n                code_count += 1\n                code = shell.input_transformer_manager.transform_cell("".join(cell.get("source", [])))\n                try:\n                    compile(code, f"{relative}:cell{index}", "exec")\n                except SyntaxError as exc:\n                    errors.append(f"Célula {index}: {exc.msg}")\n        except (OSError, ValueError, KeyError, TypeError) as exc:\n            errors.append(f"Estrutura do notebook: {exc}")\n        results.append({"path": relative, "code_cells": code_count,\n                        "status": "syntax_ok" if not errors else "invalid", "errors": errors,\n                        "runtime_status": "not_executed"})\n    return {"count": len(results), "syntax_ok": sum(r["status"] == "syntax_ok" for r in results),\n            "results": results}\n\n\ndef read_history(path: Path) -> dict[tuple[str, str], dict]:\n    if not path.exists():\n        return {}\n    with path.open(newline="", encoding="utf-8") as handle:\n        rows = list(csv.DictReader(handle))\n    return {(row["source_id"], row["url"]): row for row in rows}\n\n\ndef run(output: Path, collectors: dict = COLLECTORS, client: requests.Session | None = None) -> dict:\n    output.mkdir(parents=True, exist_ok=True)\n    csv_path = output / "news.csv"\n    previous = read_history(csv_path)\n    current = dict(previous)\n    now = datetime.now(timezone.utc).isoformat(timespec="seconds")\n    source_reports = []\n    client = client or session()\n    for source_id, collector in collectors.items():\n        try:\n            records = collector(client)\n            unique = {(r["source_id"], r["url"]): r for r in records}\n            if not unique:\n                raise ValueError("Nenhuma notícia válida encontrada; verificar URL e seletores")\n            added = 0\n            for key, item in unique.items():\n                if key in current:\n                    item["first_seen"] = current[key]["first_seen"]\n                    if not item["published_date"]:\n                        item["published_date"] = current[key]["published_date"]\n                        item["published_raw"] = current[key]["published_raw"]\n                else:\n                    item["first_seen"] = now\n                    added += 1\n                item["last_seen"] = now\n                current[key] = item\n            method = getattr(records, "method", "api_or_html")\n            complete_week = getattr(records, "complete_week", True)\n            source_reports.append({"source_id": source_id,\n                                   "status": "ok" if complete_week else "partial",\n                                   "seen": len(unique), "new": added,\n                                   **({"method": "official_rss_after_api_403",\n                                       "coverage_note": "Feed oficial com as últimas publicações; notícias anteriores ao item mais antigo não foram verificadas."}\n                                      if method == "rss" else {})})\n        except (requests.RequestException, ValueError, TypeError, KeyError) as exc:\n            source_reports.append({"source_id": source_id, "status": "failed", "seen": 0, "new": 0,\n                                   "error": f"{type(exc).__name__}: {exc}"[:400]})\n    with csv_path.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=FIELDS)\n        writer.writeheader()\n        writer.writerows(current[key] for key in sorted(current))\n    notebooks = check_notebooks()\n    week_ago = (datetime.now(timezone.utc) - timedelta(days=6)).date().isoformat()\n    recent = [r for r in current.values() if r["published_date"] and r["published_date"] >= week_ago]\n    report = {"generated_at": now, "dataset": "data/news.csv", "total_records": len(current),\n              "published_last_7_days": len(recent), "sources": source_reports, "notebooks": notebooks,\n              "scope": "Coleta recente de 4 fontes; verificação sintática de todos os notebooks. Não executa Colab."}\n    reports = output / "runs"\n    reports.mkdir(exist_ok=True)\n    text = json.dumps(report, ensure_ascii=False, indent=2) + "\\n"\n    (reports / "latest.json").write_text(text, encoding="utf-8")\n    (reports / f"run-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.json").write_text(text, encoding="utf-8")\n    return report\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", type=Path, default=ROOT / "data")\n    parser.add_argument("--check-report", type=Path)\n    args = parser.parse_args()\n    if args.check_report:\n        report = json.loads(args.check_report.read_text(encoding="utf-8"))\n    else:\n        report = run(args.output)\n    print(f"Notebooks válidos: {report[\'notebooks\'][\'syntax_ok\']}/{report[\'notebooks\'][\'count\']}")\n    for item in report["sources"]:\n        detail = f" — {item[\'error\']}" if item.get("error") else ""\n        print(f"{item[\'source_id\']}: {item[\'status\']} ({item[\'seen\']} vistos, {item[\'new\']} novos){detail}")\n    return int(report["notebooks"]["count"] == 0 or\n               report["notebooks"]["syntax_ok"] != report["notebooks"]["count"] or\n               any(item["status"] != "ok" for item in report["sources"]))\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'config/curadoria.json': '{\n  "version": "1.1",\n  "source_names": {\n    "nic": "NIC.br",\n    "edpb": "EDPB",\n    "mercociudades": "Mercociudades",\n    "senado_federal": "Senado Federal"\n  },\n  "keywords": {\n    "GDPR": [\n      "gdpr",\n      "rgpd",\n      "general data protection regulation"\n    ],\n    "LGPD": [\n      "lgpd",\n      "lei geral de proteção de dados"\n    ],\n    "Proteção de dados": [\n      "proteção de dados",\n      "protecao de dados",\n      "data protection",\n      "protección de datos",\n      "datos personales",\n      "personal data",\n      "dados pessoais"\n    ],\n    "Privacidade": [\n      "privacidade",\n      "privacy",\n      "privacidad"\n    ],\n    "Inteligência artificial": [\n      "inteligência artificial",\n      "inteligencia artificial",\n      "artificial intelligence",\n      "generative ai",\n      "ia",\n      "ai"\n    ],\n    "Algoritmos": [\n      "algoritmo",\n      "algoritmos",\n      "algorithm",\n      "algorithms",\n      "algorítmica",\n      "algorithmic"\n    ],\n    "Governança digital": [\n      "governança digital",\n      "digital governance",\n      "gobernanza digital",\n      "governança da internet",\n      "internet governance",\n      "gobernanza de internet"\n    ],\n    "Plataformas": [\n      "plataforma digital",\n      "plataformas digitais",\n      "digital platform",\n      "digital platforms",\n      "plataformas digitales",\n      "redes sociais",\n      "social media",\n      "redes sociales"\n    ],\n    "Moderação": [\n      "moderação de conteúdo",\n      "content moderation",\n      "moderación de contenidos",\n      "desinformação",\n      "disinformation",\n      "misinformation",\n      "desinformación",\n      "fake news"\n    ],\n    "Transferências de dados": [\n      "transferência internacional de dados",\n      "transferências internacionais",\n      "data transfers",\n      "international data transfer",\n      "cross border data",\n      "fluxos de dados",\n      "data transfer",\n      "data flows",\n      "data flow",\n      "transferencia de datos",\n      "transferencias de datos"\n    ],\n    "Adequação": [\n      "adequação",\n      "adequacy",\n      "adecuación"\n    ],\n    "Soberania digital": [\n      "soberania digital",\n      "digital sovereignty",\n      "soberanía digital"\n    ],\n    "Proteção infantil": [\n      "proteção infantil",\n      "children online",\n      "child protection",\n      "verificação de idade",\n      "age verification",\n      "aferição de idade",\n      "e ca digital",\n      "eca digital",\n      "crianças e adolescentes na internet",\n      "crianças na internet",\n      "proteger crianças",\n      "child safety",\n      "children and adolescents online",\n      "niños y adolescentes en internet"\n    ],\n    "DSA": [\n      "dsa",\n      "digital services act",\n      "regulamento dos serviços digitais"\n    ],\n    "ANPD": [\n      "anpd",\n      "autoridade nacional de proteção de dados",\n      "agência nacional de proteção de dados"\n    ],\n    "EDPB": [\n      "edpb",\n      "european data protection board",\n      "comitê europeu para a proteção de dados"\n    ],\n    "Mercosul": [\n      "mercosul",\n      "mercosur"\n    ],\n    "União Europeia": [\n      "união europeia",\n      "european union",\n      "unión europea",\n      "europa",\n      "european",\n      "europeu",\n      "europeia",\n      "ue",\n      "eu"\n    ],\n    "Sul Global": [\n      "sul global",\n      "global south",\n      "sur global",\n      "países em desenvolvimento",\n      "developing countries",\n      "países en desarrollo"\n    ],\n    "Cooperação digital": [\n      "parceria digital",\n      "digital partnership",\n      "aliança digital",\n      "digital alliance",\n      "cooperação digital",\n      "digital cooperation"\n    ],\n    "Internet": [\n      "internet",\n      "cibersegurança",\n      "cybersecurity",\n      "ciberseguridad"\n    ],\n    "AI Act": [\n      "ai act",\n      "artificial intelligence act",\n      "regulamento de inteligência artificial"\n    ],\n    "Comércio digital": [\n      "comércio digital",\n      "digital trade",\n      "comercio digital",\n      "comércio eletrônico",\n      "comercio electrónico",\n      "e commerce"\n    ]\n  },\n  "relevance_keywords": [\n    "GDPR",\n    "LGPD",\n    "Proteção de dados",\n    "Privacidade",\n    "Inteligência artificial",\n    "Algoritmos",\n    "Governança digital",\n    "Plataformas",\n    "Moderação",\n    "Transferências de dados",\n    "Soberania digital",\n    "Proteção infantil",\n    "DSA",\n    "ANPD",\n    "EDPB",\n    "Cooperação digital",\n    "Internet",\n    "AI Act",\n    "Comércio digital"\n  ],\n  "axes": {\n    "GDPR/LGPD e proteção de dados": {\n      "any_keywords": [\n        "GDPR",\n        "LGPD",\n        "Proteção de dados",\n        "Privacidade",\n        "Transferências de dados",\n        "Adequação",\n        "ANPD",\n        "EDPB"\n      ]\n    },\n    "IA e proteção de dados": {\n      "any_keywords": [\n        "Inteligência artificial",\n        "Algoritmos",\n        "AI Act"\n      ]\n    },\n    "Governança digital no Mercosul": {\n      "any_keywords": [\n        "Mercosul"\n      ]\n    },\n    "UE–Mercosul": {\n      "all_groups": [\n        [\n          "União Europeia",\n          "Cooperação digital"\n        ],\n        [\n          "Mercosul"\n        ]\n      ]\n    },\n    "Sul Global": {\n      "any_keywords": [\n        "Sul Global",\n        "Soberania digital"\n      ]\n    }\n  },\n  "stopwords": "a ao aos as o os um uma umas uns de da das do dos e em no nos na nas para por com sem sob sobre entre que quem como se sua suas seu seus não nao mais muito já ja também tambem esta este estas estes isso esse essa foi são sao ser tem ter há ha dos das à às ou contra até apos após antes durante ainda pela pelo pelos pelas agora todos todas vai pode the a an and or of to in on for from by with without about at into is are was were be been being this that these those it its their our your they you we not new more also will can has have had el la los las un una unos unas y o de del al en por para con sin sobre entre que quien como se su sus no es son ser fue han hay este esta estos estas lo las le les más mas nuevos nuevas da news noticia noticias notícia notícias report reports relatório relatorio brazil brasil brazilian brasileiro brasileira announces announcement anuncia apresenta diz afirma abre publica publicado publicação publicacao latest último última semana week"\n}\n'}
BASE_INCLUIDA = {'data/news.csv': 'source_id,title,published_date,published_raw,url,first_seen,last_seen\nedpb,Coordinated Supervision Committee extends scope to include Eurodac,2026-06-12,2026-06-12T12:00:00Z,https://www.edpb.europa.eu/news/coordinated-supervision-committee-extends-scope-to-include-eurodac_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,EDPB and AMLA to develop Joint Guidelines on partnerships for information sharing,2026-07-01,2026-07-01T12:00:00Z,https://www.edpb.europa.eu/news/edpb-and-amla-to-develop-joint-guidelines-on-partnerships-for-information-sharing_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,EDPB and EDPS support strengthening EU’s cybersecurity and easing compliance while protecting individuals’ personal data,2026-05-19,2026-05-19T12:00:00Z,https://www.edpb.europa.eu/news/edpb-and-edps-support-strengthening-eus-cybersecurity-and-easing-compliance-while-protecting_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,EDPB annual report 2025: supporting stakeholders through guidance and dialogue,2026-05-04,2026-05-04T12:00:00Z,https://www.edpb.europa.eu/news/edpb-annual-report-2025-supporting-stakeholders-through-guidance-and-dialogue_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,"EDPB brings clarity to data processing for scientific research, speeds up the finalisation of the anonymisation guidelines and approves first European data protection seal as a tool for transfers",2026-04-16,2026-04-16T12:00:00Z,https://www.edpb.europa.eu/news/edpb-brings-clarity-to-data-processing-for-scientific-research-speeds-up-the-finalisation-of_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,EDPB calls for legal basis for cross-regulatory information sharing,2026-07-17,2026-07-17T12:00:00Z,https://www.edpb.europa.eu/news/edpb-calls-for-legal-basis-for-cross-regulatory-information-sharing_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,EDPB gets a new look: discover the new website and brand identity,2026-06-22,2026-06-22T12:00:00Z,https://www.edpb.europa.eu/news/edpb-gets-a-new-look-discover-the-new-website-and-brand-identity_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,EDPB harmonises fining methodology and adopts final DSA-GDPR guidelines,2026-09-21,2026-09-21T12:00:00Z,https://www.edpb.europa.eu/news/edpb-harmonises-fining-methodology-and-adopts-final-dsa-gdpr-guidelines_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,EDPB meets with EU Commissioner McGrath and adopts common data breach notification template,2026-06-10,2026-06-10T12:00:00Z,https://www.edpb.europa.eu/news/edpb-meets-with-eu-commissioner-mcgrath-and-adopts-common-data-breach-notification-template_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,EDPB requires Belgian DPA to handle the merits of NOYB cookie banner complaint,2026-07-14,2026-07-14T12:00:00Z,https://www.edpb.europa.eu/news/edpb-requires-belgian-dpa-to-handle-the-merits-of-noyb-cookie-banner-complaint_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,EDPB sheds light on anonymisation and web scraping for generative AI and adopts final version of guidelines on blockchain,2026-07-08,2026-07-08T12:00:00Z,https://www.edpb.europa.eu/news/edpb-sheds-light-on-anonymisation-and-web-scraping-for-generative-ai-and-adopts-final-version_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,Marking 10 years of the GDPR: the evolution of the European data protection landscape,2026-04-27,2026-04-27T12:00:00Z,https://www.edpb.europa.eu/news/marking-10-years-of-the-gdpr-the-evolution-of-the-european-data-protection-landscape_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,One-Stop-Shop case digest on right to object and right to erasure updated,2026-06-25,2026-06-25T12:00:00Z,https://www.edpb.europa.eu/news/one-stop-shop-case-digest-on-right-to-object-and-right-to-erasure-updated_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,Stakeholder event on guidelines on the interplay between data protection and competition law: agenda now available,2026-07-30,2026-07-30T12:00:00Z,https://www.edpb.europa.eu/news/stakeholder-event-on-guidelines-on-the-interplay-between-data-protection-and-competition-law-0_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,Stakeholder event on guidelines on the interplay between data protection and competition law: save the date,2026-07-23,2026-07-23T12:00:00Z,https://www.edpb.europa.eu/news/stakeholder-event-on-guidelines-on-the-interplay-between-data-protection-and-competition-law_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nedpb,Supporting GDPR consistency: EDPB launches dedicated form,2026-06-24,2026-06-24T12:00:00Z,https://www.edpb.europa.eu/news/supporting-gdpr-consistency-edpb-launches-dedicated-form_en,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,"Convocatoria a la Comunidad de Aprendizaje «Experiencias, realidades, retos y oportunidades de los Cuidados Comunitarios»",2026-09-24,2026-09-24,https://mercociudades.org/convocatoria-a-la-comunidad-de-aprendizaje-experiencias-realidades-retos-y-oportunidades-de-los-cuidados-comunitarios/,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nmercociudades,"Convocatoria a la reunión de la Unidad Temática Autonomía, Gestión y Participación",2026-09-28,2026-09-28,https://mercociudades.org/convocatoria-a-la-reunion-de-la-unidad-tematica-autonomia-gestion-y-participacion/,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nmercociudades,Convocatoria abierta: Encuentro virtual de Discapacidad e Inclusión,2026-09-18,2026-09-18,https://mercociudades.org/convocatoria-abierta-encuentro-virtual-de-discapacidad-e-inclusion/,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nmercociudades,Convocatoria abierta: Reconocimiento a buenas prácticas en innovación educativa local,2026-09-23,2026-09-23,https://mercociudades.org/convocatoria-abierta-reconocimiento-buenas-practicas-innovacion-educativa-local/,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nmercociudades,Deportes invita a participar de su próximo encuentro y abre llamado para presentar experiencias en Deportes y Cuidados,2026-10-01,2026-10-01,https://mercociudades.org/deportes-invita-a-participar-de-su-proximo-encuentro-y-abre-llamado-para-presentar-experiencias-en-deportes-y-cuidados/,2026-10-01T16:45:22+00:00,2026-10-01T16:53:19+00:00\nmercociudades,Encuentro de Mercociudades en la Feria Internacional de Turismo de América Latina,2026-09-21,2026-09-21,https://mercociudades.org/encuentro-de-mercociudades-en-la-feria-internacional-de-turismo-de-america-latina-2/,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nmercociudades,Encuentro virtual de la UT de Infancias: “Ciudades que Cuidan de las Infancias”,2026-09-09,2026-09-09,https://mercociudades.org/encuentro-virtual-de-la-ut-de-infancias-ciudades-que-cuidan-de-las-infancias/,2026-09-29T23:31:45+00:00,2026-09-29T23:31:45+00:00\nmercociudades,Finaliza la Escuela de Resiliencia de Mercociudades en Juiz de Fora,2026-09-11,2026-09-11,https://mercociudades.org/finaliza-la-escuela-de-resiliencia-de-mercociudades-en-juiz-de-fora/,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nmercociudades,Mercociudades e ICLEI convocan a las ciudades a contribuir a los posicionamientos regionales rumbo a la COP17 y la COP31,2026-09-21,2026-09-21,https://mercociudades.org/mercociudades-iclei-convocan-ciudades-contribuir-posicionamientos-regionales-rumbo-cop17-cop31/,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nmercociudades,Mercociudades e ICLEI convocan a un diálogo regional rumbo a la COP17 y la COP31,2026-09-14,2026-09-14,https://mercociudades.org/mercociudades-iclei-convocan-dialogo-regional-rumbo-cop17-biodiversidad/,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nmercociudades,6 de agosto: Sessão sobre os Objetivos de Desenvolvimento Sustentável no ciclo orçamentário,2026-07-31,2026-07-31T16:18:46,https://mercociudades.org/pt-br/6-de-agosto-sesion-sobre-ods-en-el-ciclo-presupuestario/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,8 de setembro / Encontro de Cultura da Mercocidades,2026-09-04,2026-09-04T11:09:07,https://mercociudades.org/pt-br/8-de-septiembre-reunion-de-cultura-de-mercociudades/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Conselho da Mercocidades em Quilmes,2026-08-11,2026-08-11T14:06:18,https://mercociudades.org/pt-br/consejo-de-mercociudades-en-quilmes/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Consulta para o fortalecimento das áreas de Relações Internacionais e Comércio Exterior das “mercocidades”,2026-07-07,2026-07-07T14:11:38,https://mercociudades.org/pt-br/consulta-para-el-fortalecimiento-de-las-areas-de-relaciones-internacionales-y-comercio-exterior-de-las-mercociudades/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,"Chamada para a Comunidade de Aprendizagem “Experiências, realidades, desafios e oportunidades dos Cuidados Comunitários”",2026-09-24,2026-09-24T12:45:32,https://mercociudades.org/pt-br/convocatoria-a-la-comunidad-de-aprendizaje-experiencias-realidades-retos-y-oportunidades-de-los-cuidados-comunitarios/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,"Convite para a reunião da Unidade Temática sobre Autonomia, Gestão e Participação",2026-09-28,2026-09-28T13:22:03,https://mercociudades.org/pt-br/convocatoria-a-la-reunion-de-la-unidad-tematica-autonomia-gestion-y-participacion/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada para as “mercocidades”: IX Encontro da Rede Argentina de Cidades Educadoras,2026-09-01,2026-09-01T11:39:32,https://mercociudades.org/pt-br/convocatoria-a-las-mercociudades-ix-encuentro-de-la-red-argentina-de-ciudades-educadoras/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada aberta: 9ª Escola de Resiliência da Mercocidades,2026-07-01,2026-07-01T11:24:23,https://mercociudades.org/pt-br/convocatoria-abierta-9a-escuela-de-resiliencia-de-mercociudades/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada aberta para integrar o Boletim de Igualdade de Gênero de 2026,2026-08-17,2026-08-17T14:15:06,https://mercociudades.org/pt-br/convocatoria-abierta-boletin-equidad-genero-politicas-cuidados/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada aberta: Encontro da Unidade Temática da Juventude “Cidades que Cuidam”,2026-09-07,2026-09-07T14:45:34,https://mercociudades.org/pt-br/convocatoria-abierta-encuentro-de-la-unidad-tematica-de-juventudes-ciudades-que-cuidan/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada aberta: Encontro virtual sobre deficiência e inclusão,2026-09-18,2026-09-18T16:14:44,https://mercociudades.org/pt-br/convocatoria-abierta-encuentro-virtual-de-discapacidad-e-inclusion/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada aberta para contribuir ao posicionamento diante da COP17 da UNCCD,2026-08-04,2026-08-04T14:47:50,https://mercociudades.org/pt-br/convocatoria-abierta-para-contribuir-al-posicionamiento-ante-la-cop17-de-la-unccd/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada aberta: Reconhecimento de boas práticas em inovação educacional local,2026-09-23,2026-09-23T14:24:38,https://mercociudades.org/pt-br/convocatoria-abierta-reconocimiento-buenas-practicas-innovacion-educativa-local/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada aberta para candidaturas à Presidência da Mercocidades 2027-2028,2026-08-03,2026-08-03T14:51:09,https://mercociudades.org/pt-br/convocatoria-candidaturas-presidencia-mercociudades-2027-2028/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada aberta para o primeiro seminário do ciclo sobre dados e políticas culturais ligadas à mobilidade humana,2026-07-30,2026-07-30T15:38:33,https://mercociudades.org/pt-br/convocatoria-primer-seminario-ciclo-datos-politicas-culturales-vinculadas-movilidad-humana/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,"A Mercocidades reafirma seu compromisso com o Cuidado, a Justiça Social e Ambiental e defende o financiamento acessível e justo para enfrentar esses desafios na América Latina",2026-08-14,2026-08-14T16:11:51,https://mercociudades.org/pt-br/declaracion-consejo-quilmes/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Deportes invita a participar de su próximo encuentro y abre llamado para presentar experiencias en Deportes y Cuidados,2026-10-01,2026-10-01T12:44:46,https://mercociudades.org/pt-br/deportes-invita-a-participar-de-su-proximo-encuentro-y-abre-llamado-para-presentar-experiencias-en-deportes-y-cuidados/,2026-10-01T16:58:06+00:00,2026-10-01T16:58:06+00:00\nmercociudades,O Grupo de Esportes da Mercocidades realizará um debate sobre esporte e cuidado,2026-09-08,2026-09-08T11:21:10,https://mercociudades.org/pt-br/deportes-mercociudades-desarrollara-conversatorio-deporte-cuidados/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,"Dia Internacional das Vítimas de Desaparecimentos Forçados / Compromisso com a verdade, a justiça e a reparação para todas as vítimas e suas famílias",2026-08-31,2026-08-31T14:26:08,https://mercociudades.org/pt-br/dia-internacional-del-detenido-desaparecido-compromiso-con-la-verdad-la-justicia-y-la-reparacion-para-todas-las-victimas-y-sus-familias/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Dia Internacional da Juventude: chamado para compartilhar políticas e iniciativas para as juventudes,2026-08-12,2026-08-12T14:16:37,https://mercociudades.org/pt-br/dia-internacional-juventud-convocatoria-compartir-politicas-iniciativas-juventudes/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,XXXI Cúpula da Mercocidades em Canelones,2026-08-07,2026-08-07T16:08:26,https://mercociudades.org/pt-br/diciembre-de-2026-cumbre-de-mercociudades-en-canelones/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Encontro de Infâncias da Mercocidades sobre proteção integral dos direitos digitais,2026-07-14,2026-07-14T14:58:07,https://mercociudades.org/pt-br/encuentro-de-infancias-de-mercociudades-sobre-proteccion-integral-de-derechos-digitales/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Encontro da Mercocidades na Feira Internacional de Turismo da América Latina,2026-09-21,2026-09-21T11:45:21,https://mercociudades.org/pt-br/encuentro-de-mercociudades-en-la-feria-internacional-de-turismo-de-america-latina-2/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Encontro sobre desenvolvimento regional e turismo transfronteiriço,2026-07-31,2026-07-31T14:54:26,https://mercociudades.org/pt-br/encuentro-desarrollo-regional-binacional-turismo-transfronterizo-frontera-uruguay-brasil/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Encontro Virtual da UT de Infâncias: “Cidades que Cuidam das Infâncias”,2026-09-09,2026-09-09T12:02:01,https://mercociudades.org/pt-br/encuentro-virtual-de-la-ut-de-infancias-ciudades-que-cuidan-de-las-infancias/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Escola de Resiliência da Mercocidades chega ao fim em Juiz de Fora,2026-09-11,2026-09-11T13:59:21,https://mercociudades.org/pt-br/finaliza-la-escuela-de-resiliencia-de-mercociudades-en-juiz-de-fora/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,O Grupo de Esportes da Mercocidades convida a participar de sua próxima reunião virtual,2026-06-25,2026-06-25T14:44:16,https://mercociudades.org/pt-br/grupo-deportes-mercociudades-invita-participar-proximo-encuentro-virtual/,2026-09-28T21:36:56+00:00,2026-09-28T21:36:56+00:00\nmercociudades,Inscrições abertas: Encontro sobre ordenamento territorial e planejamento urbano,2026-09-02,2026-09-02T12:35:28,https://mercociudades.org/pt-br/inscripciones-abiertas-encuentro-sobre-ordenamiento-territorial-y-planificacion-urbana/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Encontro da Unidade Temática de Infâncias “Mobilidade do cuidado: urbanismo sensível e cidades que caminham com as infâncias”,2026-08-11,2026-08-11T14:53:09,https://mercociudades.org/pt-br/inscripciones-abiertas-encuentro-unidad-tematica-infancias-movilidad-cuidado-urbanismo-sensible-ciudades-caminan-infancias/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,"Inscrições abertas: Evento internacional sobre regulamentação, governança e indicadores de Direitos Culturais",2026-08-04,2026-08-04T14:29:12,https://mercociudades.org/pt-br/inscripciones-abiertas-evento-internacional-sobre-normativa-gobernanza-e-indicadores-de-los-derechos-culturales-en-quilmes/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Inscrições abertas: Rodadas internacionais de negócios sobre economia circular,2026-07-20,2026-07-20T15:17:34,https://mercociudades.org/pt-br/inscripciones-abiertas-rondas-internacionales-negocios-economia-circular/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,"Inscrições abertas para a oficina “Liderar sem limites: mulheres, IA e novos desafios para os governos locais”",2026-09-01,2026-09-01T13:44:20,https://mercociudades.org/pt-br/inscripciones-abiertas-taller-liderar-sin-limites-mujeres-ia-nuevos-desafios-gobiernos-locales/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Convidamos as mercocidades a participar da Pesquisa sobre Pobreza Energética e Acesso à Energia do GCoM Latam,2026-07-08,2026-07-08T13:31:27,https://mercociudades.org/pt-br/invitamos-a-las-mercociudades-a-participar-de-la-encuesta-sobre-pobreza-energetica-y-acceso-a-la-energia-de-gcom-latam/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Juiz de Fora sediará a 9ª edição da Escola de Resiliência,2026-09-04,2026-09-04T11:20:11,https://mercociudades.org/pt-br/juiz-de-fora-sera-sede-9a-edicion-escuela-resiliencia/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,"As juventudes da Mercocidades publicam o manifesto “Semeando o futuro, transformando territórios”",2026-07-09,2026-07-09T14:37:51,https://mercociudades.org/pt-br/las-juventudes-de-mercociudades-emiten-el-manifiesto-ambiental-sembrando-futuro-transformando-territorios/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Mensagem de Solidariedade ao povo da irmã República Bolivariana da Venezuela,2026-06-25,2026-06-25T15:10:29,https://mercociudades.org/pt-br/mensaje-de-solidaridad-con-el-pueblo-de-la-hermana-republica-bolivariana-de-venezuela/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Mercocidades contribui para o diálogo regional sobre governança e desenvolvimento na América Latina e no Caribe,2026-09-01,2026-09-01T15:26:53,https://mercociudades.org/pt-br/mercociudades-contribuye-al-dialogo-regional-sobre-gobernanza-y-desarrollo-en-america-latina-y-el-caribe/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Mercocidades assina três novos acordos estratégicos durante reunião do Conselho em Quilmes,2026-08-14,2026-08-14T13:56:02,https://mercociudades.org/pt-br/mercociudades-firma-tres-nuevos-acuerdos-estrategicos-consejo-quilmes/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Mercocidades e ICLEI convidam as cidades a contribuírem para as posições regionais da COP17 e da COP31,2026-09-21,2026-09-21T14:28:20,https://mercociudades.org/pt-br/mercociudades-iclei-convocan-ciudades-contribuir-posicionamientos-regionales-rumbo-cop17-cop31/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Mercocidades e ICLEI promovem diálogo regional rumo à COP17 e à COP31,2026-09-14,2026-09-14T16:11:13,https://mercociudades.org/pt-br/mercociudades-iclei-convocan-dialogo-regional-rumbo-cop17-biodiversidad/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Mercocidades e ICLEI apresentam o posicionamento regional de Governos Locais na COP17 sobre Desertificação,2026-08-31,2026-08-31T10:33:50,https://mercociudades.org/pt-br/mercociudades-iclei-presentan-posicionamiento-regional-gobiernos-locales-cop17-desertificacion/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Chamada: Mercocidades e OIM lançam laboratório de inovação para fortalecer a governança migratória local na América Latina,2026-09-03,2026-09-03T11:49:37,https://mercociudades.org/pt-br/mercociudades-oim-lanzan-laboratorio-innovacion-fortalecer-gobernanza-migratoria-local-america-latina/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Mercocidades apresenta a publicação “Comunidades e Cidades que Cuidam”,2026-07-13,2026-07-13T14:06:58,https://mercociudades.org/pt-br/mercociudades-presenta-la-publicacion-comunidades-y-ciudades-que-cuidan/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Mercocidades adere à Aliança Global de Cuidados,2026-08-27,2026-08-27T16:42:23,https://mercociudades.org/pt-br/mercociudades-se-suma-a-la-alianza-global-por-los-cuidados/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Oportunidade de treinamento em financiamento climático para governos subnacionais,2026-08-20,2026-08-20T14:03:28,https://mercociudades.org/pt-br/oportunidad-capacitacion-finanzas-climaticas-gobiernos-subnacionales/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Planejamento Estratégico e Áreas Metropolitanas convida a compartilhar experiências locais,2026-07-21,2026-07-21T12:47:23,https://mercociudades.org/pt-br/planificacion-estrategica-areas-metropolitanas-invita-compartir-experiencias-locales/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,San José sediou a 20ª Capacitação da Mercocidades em formulação de projetos,2026-08-26,2026-08-26T15:08:53,https://mercociudades.org/pt-br/san-jose-recibio-a-la-20a-capacitacion-de-mercociudades-en-formulacion-de-proyectos/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Semana de integração em San José: capacitação da Mercocidades em formulação de projetos,2026-08-07,2026-08-07T14:07:30,https://mercociudades.org/pt-br/semana-de-la-integracion-en-san-jose-capacitacion-en-formulacion-de-proyectos-de-mercociudades/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Unidade Temática de Cultura: Dois dias de trabalho diagnóstico participativo definiram as bases do Inventário Ibero-Americano de Normativas Locais,2026-08-31,2026-08-31T15:14:10,https://mercociudades.org/pt-br/unidad-tematica-cultura-dos-jornadas-diagnostico-participativo-definieron-bases-inventario-iberoamericano-normatividades-locales/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Unidade Temática de Educação lançará chamada de propostas sobre inovação educacional local,2026-09-16,2026-09-16T14:08:43,https://mercociudades.org/pt-br/unidad-tematica-educacion-presentara-convocatoria-innovacion-educativa-local/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nmercociudades,Unidade Temática de Turismo realizará reunião conjunta com o Instituto de Turismo da Mercocidades,2026-08-11,2026-08-11T12:08:54,https://mercociudades.org/pt-br/unidad-tematica-turismo-reunira-junto-instituto-turismo-mercociudades/,2026-09-28T21:36:56+00:00,2026-10-01T16:58:06+00:00\nmercociudades,La Unidad Temática de Educación presentará una convocatoria sobre innovación educativa local,2026-09-16,2026-09-16,https://mercociudades.org/unidad-tematica-educacion-presentara-convocatoria-innovacion-educativa-local/,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nnic,17º Seminário de Proteção à Privacidade e aos Dados Pessoais reúne especialistas em São Paulo,2026-08-20,2026-08-20,https://www.nic.br/noticia/na-midia/17-seminario-de-protecao-a-privacidade-e-aos-dados-pessoais-reune-especialistas-em-sao-paulo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"18% das unidades de saúde usam IA no Brasil, diz pesquisa",2026-05-13,2026-05-13,https://www.nic.br/noticia/na-midia/18-das-unidades-de-saude-usam-ia-no-brasil-diz-pesquisa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,19% dos internautas brasileiros fizeram apostas online em 2025,2026-02-27,2026-02-27,https://www.nic.br/noticia/na-midia/19-dos-internautas-brasileiros-fizeram-apostas-i-online-i-em-2025/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"2º Workshop da África Lusófona sobre Governação de Dados da APD reuniu UNESCO, Banco Mundial e Cetic.br",2026-07-28,2026-07-28,https://www.nic.br/noticia/na-midia/2-i-workshop-i-da-africa-lusofona-sobre-governacao-de-dados-da-apd-reuniu-unesco-banco-mundial-e-cetic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,2º Workshop de Credenciais Verificáveis debate transformação digital brasileira,2026-06-18,2026-06-18,https://www.nic.br/noticia/na-midia/2-i-workshop-i-de-credenciais-verificaveis-debate-transformacao-digital-brasileira/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"2026, a eleição da inteligência artificial",2026-04-22,2026-04-22,https://www.nic.br/noticia/na-midia/2026-a-eleicao-da-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,3º Encontro da Rede Conexão Povos da Floresta reúne comunidades e define novos compromissos para os territórios,2026-09-10,2026-09-10,https://www.nic.br/noticia/na-midia/3-encontro-da-rede-conexao-povos-da-floresta-reune-comunidades-e-define-novos-compromissos-para-os-territorios/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IGF Lusófono aconteceu em Angola e refletiu os desafios para a diversidade linguística da lusofonia,2026-09-29,2026-09-29,https://www.nic.br/noticia/na-midia/4-igf-lusofono-aconteceu-na-angola-e-abordou-ciberseguranca-protecao-de-dados-e-a-promocao-de-conteudos-em-lingua-portuguesa/,2026-09-29T23:31:45+00:00,2026-10-01T16:58:06+00:00\nnic,4ª Semana Nacional de Jornalismo ABI começa com debate sobre desinformação,2026-04-09,2026-04-09,https://www.nic.br/noticia/na-midia/4-semana-nacional-de-jornalismo-abi-comeca-com-debate-sobre-desinformacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,60% dos órgãos públicos utilizam IA em gestão pública,2026-07-31,2026-07-31,https://www.nic.br/noticia/na-midia/60-dos-orgaos-publicos-utilizam-ia-em-gestao-publica/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,60% dos órgãos públicos utilizam inteligência artificial,2026-07-29,2026-07-29,https://www.nic.br/noticia/na-midia/60-dos-orgaos-publicos-utilizam-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,65% das crianças e adolescentes já usam IA generativa,2026-09-17,2026-09-17,https://www.nic.br/noticia/na-midia/65-das-criancas-e-adolescentes-ja-usam-ia-generativa-aulas-de-robotica-ampliam-o-contato-com-programacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,78% das escolas que permitem uso de IA não têm guia para orientar alunos e gestores,2026-08-06,2026-08-06,https://www.nic.br/noticia/na-midia/78-das-escolas-que-permitem-uso-de-ia-nao-tem-guia-para-orientar-alunos-e-gestores/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,80% das escolas brasileiras já discutem saúde mental e tecnologia,2026-08-07,2026-08-07,https://www.nic.br/noticia/na-midia/80-das-escolas-brasileiras-ja-discutem-saude-mental-e-tecnologia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,9º BlogProg reúne lideranças da comunicação e da política em São Paulo,2026-04-08,2026-04-08,https://www.nic.br/noticia/na-midia/9-blogprog-reune-liderancas-da-comunicacao-e-da-politica-em-sao-paulo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A arquitetura da mentira,2026-07-01,2026-07-01,https://www.nic.br/noticia/na-midia/a-arquitetura-da-mentira/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A confiança do cliente entrou na era da inteligência artificial,2026-08-14,2026-08-14,https://www.nic.br/noticia/na-midia/a-confianca-do-cliente-entrou-na-era-da-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A Copa dos múltiplos dispositivos,2026-06-29,2026-06-29,https://www.nic.br/noticia/na-midia/a-copa-dos-multiplos-dispositivos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A digitalização como aliada da inclusão educacional,2026-02-13,2026-02-13,https://www.nic.br/noticia/na-midia/a-digitalizacao-como-aliada-da-inclusao-educacional/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A economia digital e a sociedade hiperconectada,2026-08-27,2026-08-27,https://www.nic.br/noticia/na-midia/a-economia-digital-e-a-sociedade-hiperconectada/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A IA chegou às empresas antes da transformação,2026-08-19,2026-08-19,https://www.nic.br/noticia/na-midia/a-ia-chegou-as-empresas-antes-da-transformacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"A IA foi um sucesso, mas o time voltou para a planilha",2026-08-28,2026-08-28,https://www.nic.br/noticia/na-midia/a-ia-foi-um-sucesso-mas-o-time-voltou-para-a-planilha/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"A IA já chegou ao setor público, mas parou nas capitais",2026-07-24,2026-07-24,https://www.nic.br/noticia/na-midia/a-ia-ja-chegou-ao-setor-publico-mas-parou-nas-capitais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"A IA já entrou nas empresas brasileiras, mas ainda de forma concentrada e padronizada",2026-06-26,2026-06-26,https://www.nic.br/noticia/na-midia/a-ia-ja-entrou-nas-empresas-brasileiras-mas-ainda-de-forma-concentrada-e-padronizada/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O uso de inteligência artificial e a inclusão digital,2026-09-15,2026-09-15,https://www.nic.br/noticia/na-midia/a-ia-tem-cep/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A IA vai avançar até onde a infraestrutura permitir,2026-05-20,2026-05-20,https://www.nic.br/noticia/na-midia/a-ia-vai-avancar-ate-onde-a-infraestrutura-permitir/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A IA está aprendendo a dizer \'não\' e a impor limites,2026-06-29,2026-06-29,https://www.nic.br/noticia/na-midia/a-inteligencia-artificial-esta-aprendendo-a-dizer-nao-e-a-impor-limites/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A nova geração se interessa por política ou apenas consome conteúdo?,2026-06-03,2026-06-03,https://www.nic.br/noticia/na-midia/a-nova-geracao-se-interessa-por-politica-ou-apenas-consome-conteudo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"A revolução da IA na medicina: inovação, prática clínica e diretrizes éticas",2026-03-26,2026-03-26,https://www.nic.br/noticia/na-midia/a-revolucao-da-ia-na-medicina-inovacao-pratica-clinica-e-diretrizes-eticas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A saúde digital cresceu e agora precisa amadurecer,2026-04-24,2026-04-24,https://www.nic.br/noticia/na-midia/a-sau-de-digital-cresceu-e-agora-precisa-amadurecer/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A tecnologia chega cada vez mais cedo à infância,2026-09-09,2026-09-09,https://www.nic.br/noticia/na-midia/a-tecnologia-chega-cada-vez-mais-cedo-a-infancia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A tecnologia e a saúde mental de crianças e adolescentes,2026-08-24,2026-08-24,https://www.nic.br/noticia/na-midia/a-tecnologia-e-a-saude-mental-de-criancas-e-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A tecnologia mudou quem você é?,2026-04-27,2026-04-27,https://www.nic.br/noticia/na-midia/a-tecnologia-mudou-quem-voce-e/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Abrint leva aos presidenciáveis agenda para conectividade e economia digital,2026-09-15,2026-09-15,https://www.nic.br/noticia/na-midia/abrint-leva-aos-presidenciaveis-agenda-para-conectividade-e-economia-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Acordos e rotina para limitação das telas durante as férias,2026-07-06,2026-07-06,https://www.nic.br/noticia/na-midia/acordos-e-rotina-para-limitacao-das-telas-durante-as-ferias/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Adeus ao ""maior de 18"": sites reforçam verificação após ECA Digital",2026-03-24,2026-03-24,https://www.nic.br/noticia/na-midia/adeus-ao-maior-de-18-sites-reforcam-verificacao-apos-eca-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Adolescência e os desafios na construção da identidade,2026-02-26,2026-02-26,https://www.nic.br/noticia/na-midia/adolescencia-e-os-desafios-na-construcao-da-identidade/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Adolescentes buscam tratamentos contra o vício em redes sociais,2026-02-19,2026-02-19,https://www.nic.br/noticia/na-midia/adolescentes-buscam-tratamentos-contra-o-vicio-em-redes-sociais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Agentes de IA precisam de maior controle e segurança,2026-07-21,2026-07-21,https://www.nic.br/noticia/na-midia/agentes-de-ia-precisam-de-maior-controle-e-seguranca/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Alana participa do FIB16, principal fórum de Internet do país",2026-05-22,2026-05-22,https://www.nic.br/noticia/na-midia/alana-participa-do-fib16-principal-forum-de-internet-do-pais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Amazônia ganha protagonismo nos debates do Fórum da Internet no Brasil,2026-06-03,2026-06-03,https://www.nic.br/noticia/na-midia/amazonia-ganha-protagonismo-nos-debates-do-forum-da-internet-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Anatel e Ancine articulam ações globais de combate à pirataria,2026-08-13,2026-08-13,https://www.nic.br/noticia/na-midia/anatel-e-ancine-articulam-acoes-globais-de-combate-a-pirataria/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Anatel e UNESCO iniciam mapeamento sobre competências digitais de crianças e adolescentes,2026-06-25,2026-06-25,https://www.nic.br/noticia/na-midia/anatel-e-unesco-iniciam-mapeamento-sobre-competencias-digitais-de-criancas-e-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Revogação da Norma 4 e o futuro da Internet no Brasil,2026-06-09,2026-06-09,https://www.nic.br/noticia/na-midia/anatel-quer-revogar-norma-de-1995-e-abre-disputa-sobre-futuro-da-internet-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ANJL vai ao STF tentar impedir bloqueio de bets pela Anatel,2026-09-29,2026-09-29,https://www.nic.br/noticia/na-midia/anjl-vai-ao-stf-tentar-impedir-bloqueio-de-i-bets-i-pela-anatel/,2026-09-29T23:31:45+00:00,2026-10-01T16:58:06+00:00\nnic,ANPD amplia pressão por proteção de dados,2026-08-28,2026-08-28,https://www.nic.br/noticia/na-midia/anpd-amplia-pressao-por-protecao-de-dados/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ANPD apresenta ECA Digital em seminário do CGI.br e NIC.br,2026-03-19,2026-03-19,https://www.nic.br/noticia/na-midia/anpd-apresenta-eca-digital-em-evento-do-nic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ANPD e Cade renovam Acordo de Cooperação Técnica por mais cinco anos,2026-07-03,2026-07-03,https://www.nic.br/noticia/na-midia/anpd-e-cade-renovam-acordo-de-cooperacao-tecnica-por-mais-cinco-anos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ANPD intensifica monitoramento de plataformas com foco no ECA Digital,2026-02-11,2026-02-11,https://www.nic.br/noticia/na-midia/anpd-intensifica-monitoramento-de-plataformas-com-foco-no-eca-digital/,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nnic,ANPD participa do 1º Seminário de Serviços Públicos Datificados,2026-04-23,2026-04-23,https://www.nic.br/noticia/na-midia/anpd-participa-do-1-seminario-de-servicos-publicos-datificados/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Ao menos 19% dos usuários de Internet no Brasil fizeram apostas online em 2025,2026-02-24,2026-02-24,https://www.nic.br/noticia/na-midia/ao-menos-19-dos-usuarios-de-internet-no-brasil-fizeram-apostas-i-online-i-em-2025/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Aos 14 anos, jogando Minecraft, alunos dessa escola dominam a ferramenta que o mercado procura",2026-07-17,2026-07-17,https://www.nic.br/noticia/na-midia/aos-14-anos-jogando-minecraft-alunos-dessa-escola-dominam-a-ferramenta-que-o-mercado-procura/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Apatia para checar informações desafia a democracia,2026-04-23,2026-04-23,https://www.nic.br/noticia/na-midia/apatia-para-checar-informacoes-desafia-a-democracia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Após ECA Digital: Congresso avança com leis para proteger crianças e adolescentes,2026-04-07,2026-04-07,https://www.nic.br/noticia/na-midia/apos-eca-digital-congresso-avanca-com-leis-para-proteger-criancas-e-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Os limites da IA na educação,2026-08-13,2026-08-13,https://www.nic.br/noticia/na-midia/aprender-antes-de-automatizar-os-limites-da-ia-na-educacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Aprender Conectado abre edital para levar Internet a novas 5.734 escolas,2026-08-27,2026-08-27,https://www.nic.br/noticia/na-midia/aprender-conectado-abre-edital-para-levar-internet-a-novas-5-734-escolas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Aprender Conectado chega à marca de 30 mil escolas; legado é principal preocupação,2026-09-14,2026-09-14,https://www.nic.br/noticia/na-midia/aprender-conectado-chega-a-marca-de-30-mil-escolas-legado-e-principal-preocupacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Arquiteturas digitais e seus impactos na proteção da infância,2026-06-08,2026-06-08,https://www.nic.br/noticia/na-midia/arquiteturas-digitais-e-seus-impactos-na-protecao-da-infancia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,As crianças precisam aprender a ficar entediadas,2026-06-09,2026-06-09,https://www.nic.br/noticia/na-midia/as-criancas-precisam-aprender-a-ficar-entediadas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"As novas gerações são, de fato, mais violentas?",2026-09-22,2026-09-22,https://www.nic.br/noticia/na-midia/as-novas-geracoes-sao-de-fato-mais-violentas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,As transmissões de futebol no Brasil em 2025,2026-05-04,2026-05-04,https://www.nic.br/noticia/na-midia/as-transmissoes-de-futebol-no-brasil-em-2025/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Assuntos relevantes que devem ser objeto de gestão pública,2026-03-05,2026-03-05,https://www.nic.br/noticia/na-midia/assuntos-relevantes-que-devem-ser-objeto-de-gestao-publica/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Atendimento via telefone ainda supera WhatsApp e Telegram, aponta TIC Governo",2026-07-27,2026-07-27,https://www.nic.br/noticia/na-midia/atendimento-via-telefone-ainda-supera-whatsapp-e-telegram-com-83-aponta-pesquisa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Avanço da IA e fake news acendem alerta sobre moderação das redes,2026-09-24,2026-09-24,https://www.nic.br/noticia/na-midia/avanco-da-ia-e-i-fake-news-i-acendem-alerta-sobre-moderacao-das-redes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Belém volta a sediar o FIB após 13 anos,2026-05-08,2026-05-08,https://www.nic.br/noticia/na-midia/belem-volta-a-sediar-o-fib-apos-13-anos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Bets e Copa do Mundo: combinação pode aumentar o risco de dependência?,2026-06-15,2026-06-15,https://www.nic.br/noticia/na-midia/bets-e-copa-do-mundo-combinacao-pode-aumentar-o-risco-de-dependencia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Big techs: STF julga últimos recursos sobre responsabilidade,2026-08-17,2026-08-17,https://www.nic.br/noticia/na-midia/big-techs-stf-julga-ultimos-recursos-sobre-responsabilidade/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Bloquear redes sociais até os 16 anos protege adolescentes? Entenda prós e contras,2026-03-06,2026-03-06,https://www.nic.br/noticia/na-midia/bloquear-redes-sociais-ate-os-16-anos-protege-adolescentes-entenda-pros-e-contras/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Bloqueios de IP e DNS expõem risco de dano colateral na Internet,2026-03-26,2026-03-26,https://www.nic.br/noticia/na-midia/bloqueios-de-ip-e-dns-expoem-risco-de-dano-colateral-na-internet/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Brasil 2050: especialistas defendem definição de papel estratégico na transição digital,2026-07-06,2026-07-06,https://www.nic.br/noticia/na-midia/brasil-2050-brasil-precisa-definir-seu-papel-na-transicao-digital-para-alcancar-soberania-defendem-especialistas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Brasil 2050: Fórum debate soluções para o futuro do país,2026-07-01,2026-07-01,https://www.nic.br/noticia/na-midia/brasil-2050-forum-reune-especialistas-para-buscar-solucoes-para-o-futuro-do-pais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Brasil celebra primeiro Dia Nacional da Proteção de Dados nesta sexta-feira (17),2026-07-17,2026-07-17,https://www.nic.br/noticia/na-midia/brasil-celebra-primeiro-dia-nacional-da-protecao-de-dados-nesta-sexta-feira-17/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Brasileiros compartilham emoções com IA, aponta pesquisa do Cetic.br",2026-09-01,2026-09-01,https://www.nic.br/noticia/na-midia/brasileiros-compartilham-emocoes-e-dados-de-saude-com-ia-aponta-pesquisa-do-cetic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Brasileiros desconfiam mais de jornais do que de conteúdo de amigos,2026-04-16,2026-04-16,https://www.nic.br/noticia/na-midia/brasileiros-desconfiam-mais-de-jornais-do-que-de-conteudo-de-amigos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Brasileiros desconfiam mais de notícias de veículos jornalísticos que de conteúdo de amigos em redes sociais e mensagens,2026-04-10,2026-04-10,https://www.nic.br/noticia/na-midia/brasileiros-desconfiam-mais-de-noticias-de-veiculos-jornalisticos-que-de-conteudo-de-amigos-em-redes-sociais-e-mensagens/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Brasileiros desconfiam, mas têm dificuldades de identificar fakes , aponta pesquisa",2026-04-13,2026-04-13,https://www.nic.br/noticia/na-midia/brasileiros-desconfiam-mas-tem-dificuldades-de-identificar-i-fakes-i-aponta-pesquisa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Brasileiros revelam vidas pessoais à IA, mas temem uso de dados",2026-09-10,2026-09-10,https://www.nic.br/noticia/na-midia/brasileiros-revelam-vidas-pessoais-a-ia-mas-temem-uso-de-dados/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Câmara temática do CGI.br amplia debate sobre soberania digital,2026-07-30,2026-07-30,https://www.nic.br/noticia/na-midia/camara-tematica-do-cgi-br-amplia-debate-sobre-soberania-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Carlos Afonso (C.A.) inaugura o Diálogos EGI, novo projeto de memória da governança da Internet",2026-08-20,2026-08-20,https://www.nic.br/noticia/na-midia/carlos-afonso-c-a-inaugura-o-dialogos-egi-novo-projeto-de-memoria-da-governanca-da-internet/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Cartografias das desigualdades no Brasil: do território agrário ao digital,2026-04-30,2026-04-30,https://www.nic.br/noticia/na-midia/cartografias-das-desigualdades-no-brasil-do-territorio-agrario-ao-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Caso de suposto aliciamento no Roblox revela riscos online,2026-05-04,2026-05-04,https://www.nic.br/noticia/na-midia/caso-de-suposto-aliciamento-no-roblox-revela-riscos-i-online-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Caso Meta e Discord: quem deve responder pelos efeitos das redes em crianças e adolescentes?,2026-08-21,2026-08-21,https://www.nic.br/noticia/na-midia/caso-meta-e-discord-quem-deve-responder-pelos-efeitos-das-redes-em-criancas-e-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Caso Orelha e a cultura de violência entre adolescentes,2026-02-25,2026-02-25,https://www.nic.br/noticia/na-midia/caso-orelha-e-a-cultura-de-violencia-entre-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Catorze anos de debates: como a pauta de inclusão digital caminhou durante o FIB entre 2013 e 2026,2026-08-27,2026-08-27,https://www.nic.br/noticia/na-midia/catorze-anos-de-debates-como-a-pauta-de-inclusao-digital-caminhou-durante-o-fib-entre-2013-e-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CDR promove evento sobre regulação de plataformas,2026-02-27,2026-02-27,https://www.nic.br/noticia/na-midia/cdr-promove-evento-sobre-regulacao-de-plataformas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Centro de Ciência de Dados para a Indústria Inteligente abre vaga de pós-doutorado,2026-07-30,2026-07-30,https://www.nic.br/noticia/na-midia/centro-de-ciencia-de-dados-para-a-industria-inteligente-abre-vaga-de-pos-doutorado/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Centro de Pesquisa Aplicada em Inteligência Artificial abre duas vagas de pós-doutorado,2026-02-23,2026-02-23,https://www.nic.br/noticia/na-midia/centro-de-pesquisa-aplicada-em-inteligencia-artificial-abre-duas-vagas-de-pos-doutorado/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Ceptro.br|NIC.br promove 13ª edição online da Semana de Capacitação,2026-09-14,2026-09-14,https://www.nic.br/noticia/na-midia/ceptro-br-nic-br-promove-13-edicao-i-online-i-da-semana-de-capacitacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"CERT.br alerta para riscos no uso de IA e orienta sobre privacidade, fraudes e segurança",2026-09-04,2026-09-04,https://www.nic.br/noticia/na-midia/cert-br-alerta-para-riscos-no-uso-de-ia-e-orienta-sobre-privacidade-fraudes-e-seguranca/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CERT.br atualiza cartilha contra golpes online,2026-04-01,2026-04-01,https://www.nic.br/noticia/na-midia/cert-br-atualiza-cartilha-contra-golpes-i-online-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CERT.br atualiza cartilha sobre golpes e fraudes online,2026-04-07,2026-04-07,https://www.nic.br/noticia/na-midia/cert-br-atualiza-cartilha-sobre-golpes-e-fraudes-i-online-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CERT.br lança novos fascículos da Cartilha de Segurança para Internet com foco na prevenção de golpes e fraudes online,2026-04-06,2026-04-06,https://www.nic.br/noticia/na-midia/cert-br-lanca-novos-fasciculos-da-cartilha-de-seguranca-para-internet-com-foco-na-prevencao-de-golpes-e-fraudes-i-online-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CERT.br vai lançar novos guias para proteger crianças na Internet,2026-10-01,2026-10-01,https://www.nic.br/noticia/na-midia/cert-br-vai-lancar-novos-guias-para-proteger-criancas-na-internet/,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br alerta para impacto de IA e algoritmos nas eleições de 2026,2026-09-23,2026-09-23,https://www.nic.br/noticia/na-midia/cgi-br-alerta-para-impacto-de-ia-e-algoritmos-nas-eleicoes-de-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br apoia decretos do MCI e defende dever de cuidado das plataformas,2026-05-26,2026-05-26,https://www.nic.br/noticia/na-midia/cgi-br-apoia-decretos-do-mci-e-defende-dever-de-cuidado-das-plataformas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Comissão da Câmara debate princípios para regulação de plataformas digitais elaborados pelo CGI.br,2026-05-14,2026-05-14,https://www.nic.br/noticia/na-midia/cgi-br-apresenta-principios-para-regulacao-de-i-big-techs-i-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br defende normas da ABNT para acessibilidade digital,2026-06-30,2026-06-30,https://www.nic.br/noticia/na-midia/cgi-br-defende-normas-da-abnt-para-acessibilidade-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br defende transparência e modelo de risco na regulação da IA no Brasil,2026-03-12,2026-03-12,https://www.nic.br/noticia/na-midia/cgi-br-defende-transparencia-e-modelo-de-risco-na-regulacao-da-ia-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br destaca avanços e aponta aprimoramentos necessários no Marco Regulatório da Inteligência Artificial,2026-03-09,2026-03-09,https://www.nic.br/noticia/na-midia/cgi-br-destaca-avancos-e-aponta-aprimoramentos-necessarios-no-marco-regulatorio-da-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br divulga nota técnica sobre recomendação de normas técnicas para acessibilidade digital,2026-07-03,2026-07-03,https://www.nic.br/noticia/na-midia/cgi-br-divulga-nota-tecnica-sobre-recomendacao-de-normas-tecnicas-para-acessibilidade-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br pede ajustes no PL do marco regulatório de inteligência artificial,2026-03-09,2026-03-09,https://www.nic.br/noticia/na-midia/cgi-br-pede-ajustes-no-pl-do-marco-regulatorio-de-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br propõe responsabilização gradual de redes sociais por conteúdo de terceiros,2026-09-17,2026-09-17,https://www.nic.br/noticia/na-midia/cgi-br-propoe-responsabilizacao-gradual-de-redes-sociais-por-conteudo-de-terceiros/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br quer acesso ao domínio .jus.br sem franquia como regra,2026-09-30,2026-09-30,https://www.nic.br/noticia/na-midia/cgi-br-quer-acesso-ao-dominio-jus-br-sem-franquia-como-regra/,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br se manifesta a favor das regras mais duras do Governo Lula às big techs,2026-05-25,2026-05-25,https://www.nic.br/noticia/na-midia/cgi-br-se-manifesta-a-favor-das-regras-mais-duras-do-governo-lula-as-i-big-techs-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ChatGPT para adolescentes: OpenAI lança versão com mais segurança,2026-08-20,2026-08-20,https://www.nic.br/noticia/na-midia/chatgpt-para-adolescentes-openai-lanca-versao-com-mais-seguranca/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ChatGPT pode ser um grande apoio de estudos para concursos e Enem,2026-07-20,2026-07-20,https://www.nic.br/noticia/na-midia/chatgpt-pode-ser-um-grande-apoio-de-estudos-para-concursos-e-enem/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Checar antes de compartilhar? Brasileiro diz que não,2026-04-22,2026-04-22,https://www.nic.br/noticia/na-midia/checar-antes-de-compartilhar-brasileiro-diz-que-nao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Claude amplia atuação no Brasil com parceiros especializados,2026-06-18,2026-06-18,https://www.nic.br/noticia/na-midia/claude-amplia-atuacao-no-brasil-com-parceiros-especializados/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"CNCiber publica proposta de lei geral para cibersegurança, com apoio multissetorial e Anatel como autoridade sugerida",2026-04-06,2026-04-06,https://www.nic.br/noticia/na-midia/cnciber-publica-proposta-de-lei-geral-para-ciberseguranca-com-apoio-multissetorial-e-anatel-como-autoridade-sugerida/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CNMP destaca acordo que garante acesso a dados estratégicos para proteção de direitos humanos na Internet,2026-02-20,2026-02-20,https://www.nic.br/noticia/na-midia/cnmp-destaca-acordo-que-garante-acesso-a-dados-estrategicos-para-protecao-de-direitos-humanos-na-internet/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Com alunos adotando IA em massa, CNE vota diretrizes para uso da tecnologia nas escolas",2026-03-11,2026-03-11,https://www.nic.br/noticia/na-midia/com-alunos-adotando-ia-em-massa-cne-vota-diretrizes-para-uso-da-tecnologia-nas-escolas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Com apoio do NIC.br e do CGI.br, MGI realiza o XI PARCOM",2026-06-09,2026-06-09,https://www.nic.br/noticia/na-midia/com-apoio-do-nic-br-e-do-cgi-br-mgi-realiza-o-xi-parcom/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Com inscrições gratuitas, 16º Fórum da Internet no Brasil discute governança digital",2026-05-21,2026-05-21,https://www.nic.br/noticia/na-midia/com-inscricoes-gratuitas-16-forum-da-internet-no-brasil-discute-governanca-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Com prefácio de Demi Getschko, livro discute efeitos da IA sobre democracia, desigualdade e sustentabilidade",2026-07-14,2026-07-14,https://www.nic.br/noticia/na-midia/com-prefacio-de-demi-getschko-e-coordenacao-de-integrantes-do-obia-o-sexto-volume-da-obra-direito-e-inteligencia-artificial-ja-esta-disponivel/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Com tradução conduzida pelo Cetic.br|NIC.br, curso da UNESCO fica disponível em português",2026-07-08,2026-07-08,https://www.nic.br/noticia/na-midia/com-traducao-conduzida-pelo-cetic-br-nic-br-curso-da-unesco-fica-disponivel-em-portugues/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Comitê Gestor da Internet lança diretrizes para regulação de redes sociais,2026-09-18,2026-09-18,https://www.nic.br/noticia/na-midia/comite-gestor-da-internet-lanca-diretrizes-para-regulacao-de-redes-sociais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Comitê Gestor da Internet propõe diretrizes para regular redes sociais,2026-09-18,2026-09-18,https://www.nic.br/noticia/na-midia/comite-gestor-da-internet-propoe-diretrizes-para-regular-redes-sociais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Comitê Gestor da Internet publica diretrizes para regular redes sociais,2026-09-21,2026-09-21,https://www.nic.br/noticia/na-midia/comite-gestor-da-internet-publica-diretrizes-para-regular-redes-sociais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br divulga 46 diretrizes para orientar o debate e a regulação de redes sociais no país,2026-09-21,2026-09-21,https://www.nic.br/noticia/na-midia/comite-propoe-freio-a-algoritmos-e-responsabilidade-de-plataformas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Como a inteligência artificial amplia a desigualdade digital,2026-03-25,2026-03-25,https://www.nic.br/noticia/na-midia/como-a-inteligencia-artificial-amplia-a-desigualdade-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Como alunos e professores usam IA nas escolas do Ceará,2026-03-03,2026-03-03,https://www.nic.br/noticia/na-midia/como-alunos-e-professores-usam-ia-nas-escolas-do-ceara/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Como as plataformas digitais controlam a opinião pública?,2026-03-04,2026-03-04,https://www.nic.br/noticia/na-midia/como-as-plataformas-digitais-controlam-a-opiniao-publica/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Como documentos digitais evitam atrasos em serviços e processos,2026-02-24,2026-02-24,https://www.nic.br/noticia/na-midia/como-documentos-digitais-evitam-atrasos-em-servicos-e-processos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Como era a Internet na última vez em que o Brasil ganhou a Copa?,2026-06-10,2026-06-10,https://www.nic.br/noticia/na-midia/como-era-a-internet-na-ultima-vez-em-que-o-brasil-ganhou-a-copa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Como escolas estão repensando o uso da tecnologia na educação,2026-09-04,2026-09-04,https://www.nic.br/noticia/na-midia/como-escolas-estao-repensando-o-uso-da-tecnologia-na-educacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Como o consumo de pornografia na adolescência impacta a saúde mental,2026-05-25,2026-05-25,https://www.nic.br/noticia/na-midia/como-o-consumo-de-pornografia-na-adolescencia-impacta-a-saude-mental/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Como posso evitar ser vítima de deepfakes ?,2026-05-05,2026-05-05,https://www.nic.br/noticia/na-midia/como-posso-evitar-ser-vitima-de-i-deepfakes-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Como transformar o “modo de estudo” das IAs em um aliado na preparação para o Enem e vestibulares,2026-03-13,2026-03-13,https://www.nic.br/noticia/na-midia/como-transformar-o-modo-de-estudo-das-ias-em-um-aliado-na-preparacao-para-o-enem-e-vestibulares/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Conectividade chega a 84% das escolas, mas somente 66% oferecem computadores para uso didático",2026-08-06,2026-08-06,https://www.nic.br/noticia/na-midia/conectividade-chega-a-84-das-escolas-mas-so-22-se-dizem-prontas-para-ia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Evento reúne Anatel, NIC.br e Consulado dos EUA para debater marco regulatório de cibersegurança",2026-07-22,2026-07-22,https://www.nic.br/noticia/na-midia/consulado-dos-eua-gsi-anatel-e-nic-br-debatem-ciberseguranca-no-ts-seguranca-digital-em-agosto/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Consulta pública sugere mais rigidez na aferição de idade na Internet,2026-02-12,2026-02-12,https://www.nic.br/noticia/na-midia/consulta-publica-sugere-mais-rigidez-na-afericao-de-idade-na-internet/,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nnic,"Cooperação de entidades é essencial no combate ao crime na Internet, diz superintendente da Anatel",2026-08-25,2026-08-25,https://www.nic.br/noticia/na-midia/cooperacao-de-entidades-e-essencial-no-combate-ao-crime-na-internet-diz-superintendente-da-anatel/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Coordenadora do CGI.br defende bloqueio de aplicações online apenas como medida extrema,2026-07-03,2026-07-03,https://www.nic.br/noticia/na-midia/coordenadora-do-cgi-br-defende-bloqueio-de-aplicacoes-i-online-i-apenas-como-medida-extrema/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Copa do Mundo feminina de futebol 2027: novas regras, velhos hábitos",2026-02-27,2026-02-27,https://www.nic.br/noticia/na-midia/copa-do-mundo-feminina-de-futebol-2027-novas-regras-velhos-habitos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Credenciais verificáveis avançam em Brasília com foco em segurança, privacidade, ECA Digital e confiança pública",2026-06-19,2026-06-19,https://www.nic.br/noticia/na-midia/credenciais-verificaveis-avancam-em-brasilia-com-foco-em-seguranca-privacidade-eca-digital-e-confianca-publica/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Crianças e adolescentes já têm contato com jogos de apostas,2026-09-11,2026-09-11,https://www.nic.br/noticia/na-midia/criancas-e-adolescentes-ja-tem-contato-com-jogos-de-apostas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Crise no Discord: é hora de banir crianças e jovens das redes sociais?,2026-08-17,2026-08-17,https://www.nic.br/noticia/na-midia/crise-no-discord-e-hora-de-banir-criancas-e-jovens-das-redes-sociais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Cristine Hoepers, gerente do CERT.br, fala sobre phishing em painel na Febraban Tech 2026",2026-09-04,2026-09-04,https://www.nic.br/noticia/na-midia/cristine-hoepers-gerente-geral-do-cert-br-fala-sobre-i-phishing-i-em-painel-na-febraban-tech-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Currículo de educação digital e midiática depende da autoria docente,2026-04-16,2026-04-16,https://www.nic.br/noticia/na-midia/curriculo-de-educacao-digital-e-midiatica-depende-da-autoria-docente/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Curso de governança da Internet debate regulação digital no Brasil,2026-03-30,2026-03-30,https://www.nic.br/noticia/na-midia/curso-de-governanca-da-internet-debate-regulacao-digital-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Dados, IA e qualificação profissional são apontados como pilares para enfrentar transformações do trabalho",2026-06-24,2026-06-24,https://www.nic.br/noticia/na-midia/dados-inteligencia-artificial-e-qualificacao-profissional-sao-apontados-como-pilares-para-enfrentar-transformacoes-do-trabalho/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"De cada 10 serviços digitais, 8 não checam idade na criação de conta",2026-03-23,2026-03-23,https://www.nic.br/noticia/na-midia/de-cada-10-servicos-digitais-8-nao-checam-idade-na-criacao-de-conta/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Debate aponta caminhos para proteger crianças no ambiente digital,2026-04-09,2026-04-09,https://www.nic.br/noticia/na-midia/debate-aponta-caminhos-para-proteger-criancas-no-ambiente-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Decisão automatizada é tema de workshop em Fórum da Internet no Brasil,2026-06-03,2026-06-03,https://www.nic.br/noticia/na-midia/decisao-automatizada-e-tema-de-i-workshop-i-em-forum-da-internet-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Decreto 12.975 e a competência da ANPD à luz da nova interpretação do art. 19,2026-06-02,2026-06-02,https://www.nic.br/noticia/na-midia/decreto-12-975-e-a-competencia-da-anpd-a-luz-da-nova-interpretacao-do-art-19/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Demi Getschko conta como a Internet realmente chegou ao Brasil,2026-03-11,2026-03-11,https://www.nic.br/noticia/na-midia/demi-getschko-conta-como-a-internet-realmente-chegou-ao-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"""Segurança digital não pode ser construída por um único ator"", defende Demi Getschko",2026-07-02,2026-07-02,https://www.nic.br/noticia/na-midia/demi-getschko-defende-colaboracao-no-marco-da-ciberseguranca/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Demi Getschko alerta para os riscos da concentração e da fragmentação da Internet,2026-06-22,2026-06-22,https://www.nic.br/noticia/na-midia/demi-getschko-internet-esta-concentrada-nas-maos-de-poucos-e-isso-preocupa-para-o-futuro/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Denúncias de crimes cibernéticos crescem 28% em 2025, mostra SaferNet",2026-02-10,2026-02-10,https://www.nic.br/noticia/na-midia/denuncias-de-crimes-ciberneticos-crescem-28-em-2025-mostra-safernet/,2026-09-28T21:01:12+00:00,2026-09-28T21:01:12+00:00\nnic,Denúncias de crimes cibernéticos crescem 28% em 2025,2026-02-11,2026-02-11,https://www.nic.br/noticia/na-midia/denuncias-de-crimes-ciberneticos-crescem-28-em-2025/,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nnic,"Em trilha parlamentar do FIB16, deputada alerta para risco de colonialismo de dados",2026-06-01,2026-06-01,https://www.nic.br/noticia/na-midia/deputada-alerta-para-risco-de-colonialismo-de-dados/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Desafios éticos da inteligência artificial: da técnica ao juízo,2026-09-09,2026-09-09,https://www.nic.br/noticia/na-midia/desafios-eticos-da-inteligencia-artificial-da-tecnica-ao-juizo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Dia da Educação: estudantes já usam IA, mas só 32% recebem orientação nas escolas",2026-04-28,2026-04-28,https://www.nic.br/noticia/na-midia/dia-da-educacao-estudantes-ja-usam-ia-mas-so-32-recebem-orientacao-nas-escolas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Digitalização da saúde expõe riscos operacionais,2026-04-16,2026-04-16,https://www.nic.br/noticia/na-midia/digitalizacao-da-saude-expoe-riscos-operacionais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Digitalização na saúde avança e 18% dos estabelecimentos usam IA,2026-05-20,2026-05-20,https://www.nic.br/noticia/na-midia/digitalizacao-na-saude-avanca-e-18-dos-estabelecimentos-usam-ia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Dilema regulatório global do uso de redes sociais pela nova geração: proibir é a resposta?,2026-07-27,2026-07-27,https://www.nic.br/noticia/na-midia/dilema-regulatorio-global-do-uso-de-redes-sociais-pela-nova-geracao-proibir-e-a-resposta/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Diretrizes do CGI.br para regulação das redes sociais,2026-09-22,2026-09-22,https://www.nic.br/noticia/na-midia/diretrizes-do-cgi-br-para-regulacao-das-redes-sociais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Discord acata decisão da ANPD e suspende lives no Brasil,2026-08-18,2026-08-18,https://www.nic.br/noticia/na-midia/discord-acata-decisao-da-anpd-e-suspende-i-lives-i-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Discord implementa novas políticas de restrição de idade,2026-02-19,2026-02-19,https://www.nic.br/noticia/na-midia/discord-implementa-novas-politicas-de-restricao-de-idade/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Disponíveis a adolescentes, ‘namorados’ de IA testam aplicação do ECA Digital",2026-08-03,2026-08-03,https://www.nic.br/noticia/na-midia/disponiveis-a-adolescentes-namorados-de-ia-testam-aplicacao-do-eca-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Do clique ao endividamento: apostas online avançam e colocam famílias em risco,2026-02-25,2026-02-25,https://www.nic.br/noticia/na-midia/do-clique-ao-endividamento-apostas-i-online-i-avancam-e-colocam-familias-em-risco/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ECA Digital: aferição de idade será prioridade da ANPD em 2027,2026-02-13,2026-02-13,https://www.nic.br/noticia/na-midia/eca-digital-afericao-de-idade-sera-prioridade-da-anpd-em-2027/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ECA Digital amplia responsabilidade de escolas e pressiona ensino a formar comportamento online,2026-04-01,2026-04-01,https://www.nic.br/noticia/na-midia/eca-digital-amplia-responsabilidade-de-escolas-e-pressiona-ensino-a-formar-comportamento-i-online-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"ECA Digital começa a valer com pressão sobre grandes plataformas, avalia coordenadora do CGI.br",2026-03-16,2026-03-16,https://www.nic.br/noticia/na-midia/eca-digital-comeca-a-valer-com-pressao-sobre-grandes-plataformas-avalia-renata-mielli-do-cgi-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ECA Digital começa a valer nesta terça; confira principais pontos,2026-03-18,2026-03-18,https://www.nic.br/noticia/na-midia/eca-digital-comeca-a-valer-nesta-terca-confira-principais-pontos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"ECA Digital depende de mudança de cultura para ser efetivo, aponta debate",2026-04-07,2026-04-07,https://www.nic.br/noticia/na-midia/eca-digital-depende-de-mudanca-de-cultura-para-ser-efetivo-aponta-debate/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ECA Digital e os desafios da proteção à infância e adolescência online,2026-08-10,2026-08-10,https://www.nic.br/noticia/na-midia/eca-digital-e-os-desafios-da-protecao-a-infancia-e-adolescencia-i-online-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,NIC.br e CGI.br reúnem autoridades para discutir implementação do ECA Digital,2026-03-17,2026-03-17,https://www.nic.br/noticia/na-midia/eca-digital-entra-em-vigor-com-regulacao-por-etapas-na-anpd/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ECA Digital entra em vigor e inaugura nova fase de proteção de crianças e adolescentes no ambiente online,2026-03-10,2026-03-10,https://www.nic.br/noticia/na-midia/eca-digital-entra-em-vigor-e-inaugura-nova-fase-de-protecao-de-criancas-e-adolescentes-no-ambiente-i-online-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"ECA Digital, lei que cria regras para o acesso de crianças e adolescentes à Internet, entra em vigor",2026-03-18,2026-03-18,https://www.nic.br/noticia/na-midia/eca-digital-estatuto-da-crianca-e-do-adolescente-que-cria-novas-regras-para-o-acesso-de-menores-a-internet-entra-em-vigor/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ECA Digital impulsiona debate sobre credenciais verificáveis para proteção de crianças e adolescentes,2026-06-25,2026-06-25,https://www.nic.br/noticia/na-midia/eca-digital-impulsiona-debate-sobre-credenciais-verificaveis-para-protecao-i-online-i-de-criancas-e-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ECA Digital muda mercado de jogos eletrônicos no Brasil com novas regras de segurança infantil e proibição de loot boxes,2026-08-31,2026-08-31,https://www.nic.br/noticia/na-midia/eca-digital-muda-mercado-de-jogos-eletronicos-no-brasil-com-novas-regras-de-seguranca-infantil-e-proibicao-de-i-loot-boxes-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Nova lei reforça proteção de crianças e adolescentes na Internet e amplia responsabilidades das plataformas,2026-03-16,2026-03-16,https://www.nic.br/noticia/na-midia/eca-digital-nova-lei-reforca-protecao-de-criancas-e-adolescentes-na-internet-e-amplia-responsabilidades-das-plataformas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ECA digital: o que muda na lei para influenciadores mirins?,2026-08-19,2026-08-19,https://www.nic.br/noticia/na-midia/eca-digital-o-que-muda-na-lei-para-influenciadores-mirins/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ECA Digital opõe modelo biométrico ao chamado Zero-Knowledge Proof,2026-02-12,2026-02-12,https://www.nic.br/noticia/na-midia/eca-digital-opoe-modelo-biometrico-ao-chamado-zero-knowledge-proof/,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nnic,Ecossistema digital prepara nova rodada de debate sobre custos de rede,2026-05-08,2026-05-08,https://www.nic.br/noticia/na-midia/ecossistema-digital-prepara-nova-rodada-de-debate-sobre-custos-de-rede/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Edital de 6 GHz: Anatel explica as inovações da proposta,2026-09-10,2026-09-10,https://www.nic.br/noticia/na-midia/edital-de-6-ghz-conselheiro-da-anatel-explica-as-inovacoes-da-proposta/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Educação entra em pleno ciclo de reinvenção,2026-03-02,2026-03-02,https://www.nic.br/noticia/na-midia/educacao-entra-em-pleno-ciclo-de-reinvencao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Novos fascículos do CERT.br alertam para o uso de deepfakes e vazamento de dados em golpes,2026-04-02,2026-04-02,https://www.nic.br/noticia/na-midia/em-atualizacao-novos-fasciculos-do-cert-br-alertam-para-o-uso-de-i-deepfakes-i-e-vazamento-de-dados-em-golpes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Em Belém, FIB16 começa na próxima segunda-feira (25)",2026-05-21,2026-05-21,https://www.nic.br/noticia/na-midia/em-belem-fib16-comeca-na-proxima-segunda-feira-25/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Brasil acelera regulação digital e busca soberania tecnológica diante de desafios da Internet e IA,2026-06-22,2026-06-22,https://www.nic.br/noticia/na-midia/em-entrevista-ao-liberal-renata-mielli-fala-sobre-regulacao-digital-e-soberania-tecnologica/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Em entrevista, Demi Getschko alerta que IA cria risco de \'acomodação intelectual\'",2026-05-18,2026-05-18,https://www.nic.br/noticia/na-midia/em-entrevista-demi-getschko-alerta-que-ia-cria-risco-de-acomodacao-intelectual/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Em legítima defesa de crianças e jovens,2026-08-03,2026-08-03,https://www.nic.br/noticia/na-midia/em-legitima-defesa-de-criancas-e-jovens/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Em meio à crise dos alertas, CNCiber cria grupos para discutir incidentes digitais",2026-06-22,2026-06-22,https://www.nic.br/noticia/na-midia/em-meio-a-crise-dos-alertas-cnciber-cria-grupos-para-discutir-incidentes-digitais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Encontro anual da ISOC Brasil em 2026,2026-04-23,2026-04-23,https://www.nic.br/noticia/na-midia/encontro-anual-da-isoc-brasil-em-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Encontro discute em Belém temas centrais da agenda digital brasileira,2026-05-26,2026-05-26,https://www.nic.br/noticia/na-midia/encontro-discute-em-belem-temas-centrais-da-agenda-digital-brasileira/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Ensino de IA nas escolas públicas já é realidade,2026-03-25,2026-03-25,https://www.nic.br/noticia/na-midia/ensino-de-ia-nas-escolas-publicas-ja-e-realidade/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,EGI abre inscrições para quinta edição de curso EaD gratuito,2026-08-25,2026-08-25,https://www.nic.br/noticia/na-midia/escola-governanca-da-internet-no-brasil-do-cgi-br-abre-inscricoes-para-5-edicao-de-curso-ead-gratuito/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Escolas usam IA na gestão e nas salas de aula, mas faltam regras formais",2026-08-04,2026-08-04,https://www.nic.br/noticia/na-midia/escolas-usam-ia-na-gestao-e-nas-salas-de-aula-mas-faltam-regras-formais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Escrever ainda é humano? Como explosão de textos gerados por IA pode \'padronizar’ a linguagem,2026-03-31,2026-03-31,https://www.nic.br/noticia/na-midia/escrever-ainda-e-humano-como-explosao-de-textos-gerados-por-ia-pode-padronizar-a-linguagem/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,A terceirização da opinião pública para o algoritmo,2026-07-06,2026-07-06,https://www.nic.br/noticia/na-midia/estamos-terceirizando-nossa-opiniao-para-o-algoritmo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Uso de IA no dia a dia pode facilitar estudos para alunos brasileiros,2026-06-30,2026-06-30,https://www.nic.br/noticia/na-midia/estudar-ficou-mais-facil-veja-como-alunos-estao-usando-ia-no-dia-a-dia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Estúdio News aborda as principais inovações e benefícios do ECA Digital,2026-07-17,2026-07-17,https://www.nic.br/noticia/na-midia/estudio-news-aborda-as-principais-inovacoes-e-beneficios-do-eca-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Estudo mostra falta de verificação de idade,2026-03-23,2026-03-23,https://www.nic.br/noticia/na-midia/estudo-mostra-falta-de-verificacao-de-idade/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Etice completa 26 anos durante o maior evento de infraestrutura da Internet da América Latina,2026-03-24,2026-03-24,https://www.nic.br/noticia/na-midia/etice-completa-26-anos-durante-o-maior-evento-de-infraestrutura-da-internet-da-america-latina/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Eventos internacionais debatem gestão de dados em arquivos digitais,2026-05-06,2026-05-06,https://www.nic.br/noticia/na-midia/eventos-internacionais-debatem-gestao-de-dados-em-arquivos-digitais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Fábio Senne, coordenador do Cetic.br, participa de podcast Futuramente, na CBN",2026-09-02,2026-09-02,https://www.nic.br/noticia/na-midia/fabio-senne-coordenador-do-cetic-br-participa-de-i-podcast-i-sobre-algoritmos-e-desinformacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Fantástico destaca dados inéditos da pesquisa TIC Educação 2025,2026-09-14,2026-09-14,https://www.nic.br/noticia/na-midia/fantastico/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Fascículo gratuito do CERT.br ensina como usar a IA com segurança,2026-09-18,2026-09-18,https://www.nic.br/noticia/na-midia/fasciculo-gratuito-do-cert-br-ensina-como-usar-a-ia-com-seguranca/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Febraban Tech 2026 debate o uso da IA na cibersegurança e o avanço das fraudes digitais,2026-07-22,2026-07-22,https://www.nic.br/noticia/na-midia/febraban-tech-2026-debate-o-uso-da-ia-na-ciberseguranca-e-o-avanco-das-fraudes-digitais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Férias na era digital: adolescentes trocam o descanso pela criação de conteúdo e empreendedorismo,2026-07-15,2026-07-15,https://www.nic.br/noticia/na-midia/ferias-na-era-digital-adolescentes-trocam-o-descanso-pela-criacao-de-conteudo-e-empreendedorismo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Ferramenta do NIC.br já mediu a qualidade da conexão em 102 mil escolas,2026-05-12,2026-05-12,https://www.nic.br/noticia/na-midia/ferramenta-do-nic-br-ja-mediu-a-qualidade-da-conexao-em-102-mil-escolas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,FGV Direito SP promove imersão de estudantes na governança da Internet durante visita ao NIC.br,2026-03-30,2026-03-30,https://www.nic.br/noticia/na-midia/fgv-direito-sp-promove-imersao-de-estudantes-na-governanca-da-internet-durante-visita-ao-nic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"FIB 2026: soberania digital, IA e conectividade amazônica marcam debates",2026-05-29,2026-05-29,https://www.nic.br/noticia/na-midia/fib-2026-soberania-digital-ia-e-conectividade-amazonica-marcam-debates/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,FIB retorna a Belém com debates sobre IA e cibersegurança,2026-05-12,2026-05-12,https://www.nic.br/noticia/na-midia/fib-retorna-a-belem-com-debates-sobre-ia-e-ciberseguranca/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"FIB16 acontecerá em Belém, o coração da Amazônia",2026-05-21,2026-05-21,https://www.nic.br/noticia/na-midia/fib16-acontecera-em-belem-o-coracao-da-amazonia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"FIB16 debate IA, cibersegurança e proteção de crianças e adolescentes no ambiente digital",2026-05-28,2026-05-28,https://www.nic.br/noticia/na-midia/fib16-debate-ia-ciberseguranca-e-protecao-de-criancas-e-adolescentes-no-ambiente-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,FIB16 discute soberania linguística indígena,2026-05-29,2026-05-29,https://www.nic.br/noticia/na-midia/fib16-debate-soberania-linguistica-indigena/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Último dia do FIB16 discutiu o papel do Congresso na governança tecnológica,2026-06-01,2026-06-01,https://www.nic.br/noticia/na-midia/fib16-debateu-regulacao-da-ia-soberania-digital-e-papel-do-congresso-na-governanca-tecnologica/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,FIB16: direito dos povos originários precede direitos digitais,2026-05-27,2026-05-27,https://www.nic.br/noticia/na-midia/fib16-direito-dos-povos-originarios-precede-direitos-digitais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,FIB16 em Belém: um passeio visual,2026-06-01,2026-06-01,https://www.nic.br/noticia/na-midia/fib16-em-belem-pa-um-passeio-visual/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,FIB16 fala de IA e cibersegurança em Belém,2026-05-22,2026-05-22,https://www.nic.br/noticia/na-midia/fib16-fala-de-ia-e-ciberseguranca-em-belem/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,FIB16 movimenta Belém entre segunda e sexta-feira,2026-05-27,2026-05-27,https://www.nic.br/noticia/na-midia/fib16-movimenta-belem-entre-segunda-e-sexta-feira/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"FIB16 vai até sexta com debates em Belém sobre IA, ECA Digital e soberania",2026-05-26,2026-05-26,https://www.nic.br/noticia/na-midia/fib16-vai-ate-sexta-com-debates-em-belem-sobre-ia-eca-digital-e-soberania/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,FIB retorna a Belém após 13 anos com debates sobre IA e cibersegurança,2026-05-25,2026-05-25,https://www.nic.br/noticia/na-midia/forum-da-internet-no-brasil-retorna-a-belem-apos-13-anos-com-debates-sobre-ia-e-ciberseguranca/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Fraude migra dos sistemas para o comportamento do cliente,2026-08-31,2026-08-31,https://www.nic.br/noticia/na-midia/fraude-migra-dos-sistemas-para-o-comportamento-do-cliente/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Golpes com dados pessoais atingem 45 milhões de brasileiros,2026-09-02,2026-09-02,https://www.nic.br/noticia/na-midia/golpes-com-dados-pessoais-atingem-45-milhoes-de-brasileiros/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Golpes digitais crescem e guias ensinam como se proteger,2026-04-02,2026-04-02,https://www.nic.br/noticia/na-midia/golpes-digitais-crescem-e-guias-ensinam-como-se-proteger/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Golpes na Internet: quase um terço dos usuários relata tentativas de fraude,2026-09-08,2026-09-08,https://www.nic.br/noticia/na-midia/golpes-na-internet-quase-um-terco-dos-usuarios-relata-tentativas-de-fraude/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Governança digital ainda é gargalo para expansão de provedores no Brasil,2026-03-31,2026-03-31,https://www.nic.br/noticia/na-midia/governanca-digital-ainda-e-gargalo-para-expansao-de-provedores-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Governança em Saúde Digital: Estratégia Nacional de Saúde Digital à luz das transformações do setor,2026-03-11,2026-03-11,https://www.nic.br/noticia/na-midia/governanca-em-saude-digital-estrategia-nacional-de-saude-digital-a-luz-das-transformacoes-do-setor/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Governo defende regulação “cirúrgica” de plataformas digitais na Câmara,2026-05-14,2026-05-14,https://www.nic.br/noticia/na-midia/governo-defende-regulacao-cirurgica-de-plataformas-digitais-na-camara/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Governo Lula avalia ampliar decreto sobre big techs com agência fiscalizando deveres,2026-05-18,2026-05-18,https://www.nic.br/noticia/na-midia/governo-lula-avalia-ampliar-decreto-sobre-i-big-techs-i-com-agencia-fiscalizando-deveres/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Governo publica regras para eleições de 2026 do CGI.br,2026-06-12,2026-06-12,https://www.nic.br/noticia/na-midia/governo-publica-regras-para-eleicoes-de-2026-do-cgi-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Governo vai fiscalizar sites e pagamentos para combater bets clandestinas,2026-09-28,2026-09-28,https://www.nic.br/noticia/na-midia/governo-vai-fiscalizar-i-sites-i-e-pagamentos-para-combater-i-bets-i-clandestinas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,GSMA defende revisão regulatória para equilibrar assimetrias entre teles e grandes usuários de rede,2026-05-11,2026-05-11,https://www.nic.br/noticia/na-midia/gsma-defende-revisao-regulatoria-para-equilibrar-assimetrias-entre-teles-e-grandes-usuarios-de-rede/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Guia para influenciadores orienta sobre regras e boas práticas para atuação nas eleições de 2026,2026-08-12,2026-08-12,https://www.nic.br/noticia/na-midia/guia-para-influenciadores-orienta-sobre-regras-e-boas-praticas-para-atuacao-nas-eleicoes-de-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Cyberbullying em escola de Porto Velho leva Justiça a bloquear perfil,2026-07-13,2026-07-13,https://www.nic.br/noticia/na-midia/i-cyberbullying-i-em-escola-de-porto-velho-leva-justica-a-bloquear-perfil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Cyberbullying , IA e os novos riscos para crianças e adolescentes",2026-08-21,2026-08-21,https://www.nic.br/noticia/na-midia/i-cyberbullying-i-ia-e-os-novos-riscos-para-criancas-e-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Cyberbullying na escola: como proteger crianças e adolescentes,2026-03-05,2026-03-05,https://www.nic.br/noticia/na-midia/i-cyberbullying-i-na-escola-como-proteger-criancas-e-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Deepfakes viram principal ameaça digital às eleições, diz guia",2026-09-25,2026-09-25,https://www.nic.br/noticia/na-midia/i-deepfakes-i-viram-principal-ameaca-digital-as-eleicoes-diz-guia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Dia da Internet Segura 2026: edição brasileira discute ECA Digital, verificação etária online e deepfakes",2026-02-09,2026-02-09,https://www.nic.br/noticia/na-midia/i-dia-da-internet-segura-i-2026-edicao-brasileira-discute-eca-digital-verificacao-etaria-i-online-i-e-i-deepfakes-i/,2026-09-28T21:01:12+00:00,2026-09-28T21:01:12+00:00\nnic,Fair share : CICS analisará inclusão da ABES em audiência,2026-06-02,2026-06-02,https://www.nic.br/noticia/na-midia/i-fair-share-i-cics-analisara-inclusao-da-abes-em-audiencia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Streaming , home office e games : a disputa invisível pela Internet dentro de casa",2026-06-11,2026-06-11,https://www.nic.br/noticia/na-midia/i-streaming-i-i-home-office-i-e-i-games-i-a-disputa-invisivel-pela-internet-dentro-de-casa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Streaming , redes sociais e serviços em nuvem fazem IX.br bater recorde de 50 Tbit/s de tráfego Internet",2026-03-23,2026-03-23,https://www.nic.br/noticia/na-midia/i-streaming-i-redes-sociais-e-servicos-em-nuvem-fazem-ix-br-bater-recorde-de-50-tbit-s-de-trafego-internet/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Workshop em Brasília discute credenciais verificáveis e segurança digital,2026-06-19,2026-06-19,https://www.nic.br/noticia/na-midia/i-workshop-i-em-brasilia-discute-credenciais-verificaveis-e-seguranca-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA acelera mudança na forma como empresas desenvolvem seus profissionais,2026-09-11,2026-09-11,https://www.nic.br/noticia/na-midia/ia-acelera-mudanca-na-forma-como-empresas-desenvolvem-seus-profissionais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA avança nas escolas, mas só 22% têm diretrizes de uso",2026-08-04,2026-08-04,https://www.nic.br/noticia/na-midia/ia-avanca-nas-escolas-mas-so-22-tem-diretrizes-de-uso/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA chega a 18% dos estabelecimentos de saúde, mas uso ainda é operacional",2026-05-13,2026-05-13,https://www.nic.br/noticia/na-midia/ia-chega-a-18-dos-estabelecimentos-de-saude-mas-uso-ainda-e-operacional/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA chega a 39% das organizações sem fins lucrativos,2026-07-08,2026-07-08,https://www.nic.br/noticia/na-midia/ia-chega-a-39-das-organizacoes-sem-fins-lucrativos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA chega à sala de aula e muda a forma de ensinar e aprender; veja dilemas de escolas e alunos,2026-09-16,2026-09-16,https://www.nic.br/noticia/na-midia/ia-chega-a-sala-de-aula-e-muda-a-forma-de-ensinar-e-aprender-veja-dilemas-de-escolas-e-alunos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA chega às escolas, mas 78% ainda não possuem guia de uso",2026-08-11,2026-08-11,https://www.nic.br/noticia/na-midia/ia-chega-as-escolas-mas-78-ainda-nao-possuem-guia-de-uso/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA e vídeos falsos superam fake news como maior ameaça à democracia,2026-09-23,2026-09-23,https://www.nic.br/noticia/na-midia/ia-e-videos-falsos-superam-i-fake-news-i-como-maior-ameaca-a-democracia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA está presente em seis de cada dez órgãos públicos federais e estaduais,2026-07-23,2026-07-23,https://www.nic.br/noticia/na-midia/ia-esta-presente-em-6-de-cada-10-orgaos-publicos-federais-e-estaduais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA está presente em seis de cada dez órgãos públicos federais e estaduais e chega a 94% no Judiciário, aponta TIC Governo",2026-07-28,2026-07-28,https://www.nic.br/noticia/na-midia/ia-esta-presente-em-seis-de-cada-dez-orgaos-publicos-federais-e-estaduais-e-chega-a-94-no-judiciario-aponta-tic-governo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA está presente em seis de cada dez órgãos públicos federais e estaduais,2026-07-24,2026-07-24,https://www.nic.br/noticia/na-midia/ia-esta-presente-em-seis-de-cada-dez-orgaos-publicos-federais-e-estaduais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA exige salto na capacidade energética e desafia conectividade dos data centers,2026-05-07,2026-05-07,https://www.nic.br/noticia/na-midia/ia-exige-salto-na-capacidade-energetica-e-desafia-conectividade-dos-i-data-centers-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA fica entre o risco ambiental e a urgência de respostas,2026-09-08,2026-09-08,https://www.nic.br/noticia/na-midia/ia-fica-entre-o-risco-ambiental-e-a-urgencia-de-respostas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA Generativa: o que é, como funciona e aplicações",2026-07-15,2026-07-15,https://www.nic.br/noticia/na-midia/ia-generativa-o-que-e-como-funciona-e-aplicacoes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA influencia e molda cada vez mais a reputação de marcas,2026-06-17,2026-06-17,https://www.nic.br/noticia/na-midia/ia-influencia-e-molda-cada-vez-mais-a-reputac-a-o-de-marcas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA já está em 59% dos órgãos públicos brasileiros e alcança 94% no Judiciário,2026-07-29,2026-07-29,https://www.nic.br/noticia/na-midia/ia-ja-esta-em-59-dos-orgaos-publicos-brasileiros-e-alcanca-94-no-judiciario/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA já está presente em 59% dos órgãos públicos e alcança 94% do Judiciário, aponta estudo",2026-07-24,2026-07-24,https://www.nic.br/noticia/na-midia/ia-ja-esta-presente-em-59-dos-orgaos-publicos-e-alcanca-94-do-judiciario-aponta-estudo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA muda o papel da escola e desafia educadores,2026-08-24,2026-08-24,https://www.nic.br/noticia/na-midia/ia-muda-o-papel-da-escola-e-desafia-educadores/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA na educação: regulação precisa equilibrar inovação para reduzir desigualdades,2026-04-14,2026-04-14,https://www.nic.br/noticia/na-midia/ia-na-educacao-regulacao-precisa-equilibrar-inovacao-para-reduzir-desigualdades/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA na escola: 70% no Médio, 15% e 39% no Fundamental",2026-06-26,2026-06-26,https://www.nic.br/noticia/na-midia/ia-na-escola-70-no-medio-15-e-39-no-fundamental/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA na sala de aula: o Brasil precisa de um debate à altura do desafio,2026-04-15,2026-04-15,https://www.nic.br/noticia/na-midia/ia-na-sala-de-aula-o-brasil-precisa-de-um-debate-a-altura-do-desafio/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA não substitui o médico, mas já está transformando a medicina",2026-02-25,2026-02-25,https://www.nic.br/noticia/na-midia/ia-nao-substitui-o-medico-mas-ja-esta-transformando-a-medicina/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA nas escolas: a tecnologia pode aumentar as desigualdades?,2026-09-16,2026-09-16,https://www.nic.br/noticia/na-midia/ia-nas-escolas-a-tecnologia-pode-aumentar-as-desigualdades/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA para todos?,2026-05-11,2026-05-11,https://www.nic.br/noticia/na-midia/ia-para-todos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA preditiva e generativa: entenda as diferenças e quando as empresas podem usar cada uma delas,2026-03-12,2026-03-12,https://www.nic.br/noticia/na-midia/ia-preditiva-e-generativa-entenda-as-diferencas-e-quando-as-empresas-podem-usar-cada-uma-delas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IA usa estética infantil para banalizar violência,2026-04-17,2026-04-17,https://www.nic.br/noticia/na-midia/ia-usa-estetica-infantil-para-banalizar-violencia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IDP Debates: lideranças brasileiras analisam presente e futuro do país,2026-08-13,2026-08-13,https://www.nic.br/noticia/na-midia/idp-debates-liderancas-brasileiras-debatem-presente-e-futuro-do-pais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IEA abre inscrições para curso de extensão sobre economia, cultura e poder na Internet",2026-02-19,2026-02-19,https://www.nic.br/noticia/na-midia/iea-abre-inscricoes-para-curso-de-extensao-sobre-economia-cultura-e-poder-na-internet/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Impacto da inteligência artificial na infraestrutura de rede domina debates sobre regulação e taxas de conectividade,2026-05-12,2026-05-12,https://www.nic.br/noticia/na-midia/impacto-da-inteligencia-artificial-na-infraestrutura-de-rede-domina-debates-sobre-regulacao-e-taxas-de-conectividade/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Impacto das telas na saúde mental já é debatido em 80% das escolas,2026-08-05,2026-08-05,https://www.nic.br/noticia/na-midia/impacto-das-telas-na-saude-mental-ja-e-debatido-em-80-das-escolas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Imprensa Jovem participa do Dia da Internet Segura e realiza cobertura sobre cidadania digital,2026-03-04,2026-03-04,https://www.nic.br/noticia/na-midia/imprensa-jovem-participa-do-dia-da-internet-segura-e-realiza-cobertura-sobre-cidadania-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Inclusão digital vira oportunidades em comunidades rurais amazônicas,2026-06-24,2026-06-24,https://www.nic.br/noticia/na-midia/inclusao-digital-vira-oportunidades-em-comunidades-rurais-amazonicas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Inscrições abertas para 17º Seminário de Proteção à Privacidade e para o 3º Seminário do OBIA,2026-08-21,2026-08-21,https://www.nic.br/noticia/na-midia/inscricoes-abertas-para-17-seminario-de-protecao-a-privacidade-e-para-o-3-seminario-do-obia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Inteligência artificial e treino aceleram a busca por hiper personalização no bem-estar digital,2026-03-31,2026-03-31,https://www.nic.br/noticia/na-midia/inteligencia-artificial-e-treino-aceleram-a-busca-por-hiper-personalizacao-no-bem-estar-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Inteligência artificial está presente em 18% dos serviços de saúde do país,2026-05-19,2026-05-19,https://www.nic.br/noticia/na-midia/inteligencia-artificial-esta-presente-em-18-dos-servicos-de-saude-do-pais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Inteligência artificial na escola: proibir ou ensinar a usar?,2026-06-08,2026-06-08,https://www.nic.br/noticia/na-midia/inteligencia-artificial-na-escola-proibir-ou-ensinar-a-usar/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Inteligência emocional: os riscos de crescer com apoio da IA,2026-04-06,2026-04-06,https://www.nic.br/noticia/na-midia/inteligencia-emocional-os-riscos-de-crescer-com-apoio-da-ia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Internet entre crianças e adolescentes exige atenção,2026-03-03,2026-03-03,https://www.nic.br/noticia/na-midia/internet-entre-criancas-e-adolescentes-exige-atencao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Internet supera TV no Brasil, mas deepfakes ameaçam ambiente informacional",2026-04-27,2026-04-27,https://www.nic.br/noticia/na-midia/internet-supera-tv-no-brasil-mas-i-deepfakes-i-ameacam-ambiente-informacional/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IPv6 entra em nova fase e desafia provedores a avançarem na maturidade das redes,2026-09-29,2026-09-29,https://www.nic.br/noticia/na-midia/ipv6-entra-em-nova-fase-e-desafia-provedores-a-avancarem-na-maturidade-das-redes/,2026-09-29T23:31:45+00:00,2026-10-01T16:58:06+00:00\nnic,"IX.br bate novo recorde de troca de tráfego, com 50 Tbps",2026-03-24,2026-03-24,https://www.nic.br/noticia/na-midia/ix-br-bate-novo-recorde-de-troca-de-trafego-com-50-tbps/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IX.br bate recorde de 50 Tbit/s de tráfego Internet agregado,2026-03-25,2026-03-25,https://www.nic.br/noticia/na-midia/ix-br-bate-recorde-de-50-tbit-s-de-trafego-internet-agregado/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IX.br chega ao Amapá e PPTs estão em todos os estados do país,2026-09-24,2026-09-24,https://www.nic.br/noticia/na-midia/ix-br-chega-ao-amapa-e-ppts-estao-em-todos-os-estados-do-pais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IX.br inaugura 40º Ponto de Troca de Tráfego no Amapá,2026-09-25,2026-09-25,https://www.nic.br/noticia/na-midia/ix-br-inaugura-40-ponto-de-troca-de-trafego-no-amapa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IX.br vai hospedar solução para facilitar a interconexão de voz,2026-03-30,2026-03-30,https://www.nic.br/noticia/na-midia/ix-br-vai-hospedar-solucao-para-facilitar-a-interconexao-de-voz/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IX Fórum Fortaleza acontece neste mês de março,2026-03-17,2026-03-17,https://www.nic.br/noticia/na-midia/ix-forum-fortaleza-acontece-nesse-mes-de-marco/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IX Fórum Regional – Edição Norte reúne provedores e especialistas no Tocantins,2026-08-04,2026-08-04,https://www.nic.br/noticia/na-midia/ix-forum-regional-edicao-norte-reune-provedores-e-especialistas-em-tocantins/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Jovens encontram respostas online , mas poucos checam as fontes",2026-08-25,2026-08-25,https://www.nic.br/noticia/na-midia/jovens-encontram-respostas-i-online-i-mas-poucos-checam-as-fontes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Jurista explica por que print de tela não é prova no processo judicial,2026-02-12,2026-02-12,https://www.nic.br/noticia/na-midia/jurista-explica-por-que-i-print-i-de-tela-nao-e-prova-no-processo-judicial/,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nnic,Juventude e cidadania digital ganham destaque no FIB16 em Belém,2026-05-28,2026-05-28,https://www.nic.br/noticia/na-midia/juventude-e-cidadania-digital-ganham-destaque-no-fib16-em-belem/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Legislação penal de cibersegurança está desatualizada, avalia presidente do conselho consultivo da Anatel",2026-05-28,2026-05-28,https://www.nic.br/noticia/na-midia/legislacao-penal-de-ciberseguranca-esta-desatualizada-avalia-presidente-do-conselho-consultivo-da-anatel/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Lei Felca: adolescentes mais protegidos nas redes?,2026-04-01,2026-04-01,https://www.nic.br/noticia/na-midia/lei-felca-adolescentes-mais-protegidos-nas-redes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Levantamento mapeia 24 anos de digitalização desigual nas escolas brasileiras,2026-05-07,2026-05-07,https://www.nic.br/noticia/na-midia/levantamento-mapeia-24-anos-de-digitalizacao-desigual-nas-escolas-brasileiras/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Levantamento revela jogos de cassino online mais populares de 2025,2026-03-13,2026-03-13,https://www.nic.br/noticia/na-midia/levantamento-revela-jogos-de-cassino-i-online-i-mais-populares-de-2025/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Lula assina decretos com novas regras para redes sociais,2026-05-20,2026-05-20,https://www.nic.br/noticia/na-midia/lula-assina-decretos-com-novas-regras-para-redes-sociais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Lula assina medida provisória que proíbe os sites de aposta no Brasil,2026-09-28,2026-09-28,https://www.nic.br/noticia/na-midia/lula-assina-medida-provisoria-que-proibe-os-i-sites-i-de-aposta-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Macaé: ECA Digital mostra disposição, mas transformação não é imediata",2026-03-18,2026-03-18,https://www.nic.br/noticia/na-midia/macae-eca-digital-mostra-disposicao-mas-transformacao-nao-e-imediata/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Maioria das plataformas não protege crianças e adolescentes,2026-07-15,2026-07-15,https://www.nic.br/noticia/na-midia/maioria-das-plataformas-nao-protege-criancas-e-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Maioria dos brasileiros desconhece como funcionam os algoritmos e deepfakes são realidade para 41% dos usuários de Internet,2026-04-13,2026-04-13,https://www.nic.br/noticia/na-midia/maioria-dos-brasileiros-desconhece-como-funcionam-os-algoritmos-e-i-deepfakes-i-sao-realidade-para-41-dos-usuarios-de-internet/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Maioria dos serviços digitais não exige aferição de idade no cadastro, aponta estudo",2026-03-19,2026-03-19,https://www.nic.br/noticia/na-midia/maioria-dos-servicos-digitais-nao-exige-afericao-de-idade-no-cadastro-aponta-estudo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Mais da metade dos brasileiros usa IA para falar sobre emoções,2026-09-03,2026-09-03,https://www.nic.br/noticia/na-midia/mais-da-metade-dos-brasileiros-usa-ia-para-falar-sobre-emocoes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Mais de 90% das crianças e adolescentes utilizam Internet regularmente,2026-07-21,2026-07-21,https://www.nic.br/noticia/na-midia/mais-de-90-das-criancas-e-adolescentes-utilizam-internet-regularmente/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Mais de um terço das escolas já permite uso de IA por alunos, mas só 22% têm regras, diz levantamento",2026-08-07,2026-08-07,https://www.nic.br/noticia/na-midia/mais-de-um-terco-das-escolas-ja-permite-uso-de-ia-por-alunos-mas-so-22-tem-regras-diz-levantamento/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,MCTI participa de seminário para debater o futuro digital do Brasil,2026-04-10,2026-04-10,https://www.nic.br/noticia/na-midia/mcti-participa-de-seminario-para-debater-o-futuro-digital-do-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Meninas e publicidade infantil: entre riscos e novas regras,2026-05-05,2026-05-05,https://www.nic.br/noticia/na-midia/meninas-e-publicidade-infantil-entre-riscos-e-novas-regras/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Menos vagas e mais tecnologia elevam a exigência no mercado de trabalho,2026-08-12,2026-08-12,https://www.nic.br/noticia/na-midia/menos-vagas-e-mais-tecnologia-elevam-a-exigencia-no-mercado-de-trabalho/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Mensagens e ligações fraudulentas já atingiram 45 milhões de brasileiros, aponta Cetic.br",2026-09-03,2026-09-03,https://www.nic.br/noticia/na-midia/mensagens-e-ligacoes-fraudulentas-ja-atingiram-45-milhoes-de-brasileiros-aponta-cetic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Mensageria com IA integrada antecipa novo campo de disputa informacional para as eleições de 2026,2026-03-10,2026-03-10,https://www.nic.br/noticia/na-midia/mensageria-com-ia-integrada-antecipa-novo-campo-de-disputa-informacional-para-as-eleicoes-de-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Mais da metade dos usuários de Internet brasileiros com 16 anos ou mais se informa diariamente por vídeos curtos, aponta pesquisa inédita",2026-04-10,2026-04-10,https://www.nic.br/noticia/na-midia/metade-dos-brasileiros-de-ao-menos-16-anos-se-informam-diariamente-por-videos-curtos-aponta-pesquisa-inedita/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Mídia digital avança e amplia acesso à informação,2026-04-15,2026-04-15,https://www.nic.br/noticia/na-midia/midia-digital-avanca-e-amplia-acesso-a-informacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"""Misantropia"": alerta falso expõe novo desafio da segurança digital",2026-07-01,2026-07-01,https://www.nic.br/noticia/na-midia/misantropia-alerta-falso-expoe-novo-desafio-da-seguranca-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,MPF recomenda que prefeitura de Alcântara (MA) instale medidores de Internet em escolas públicas,2026-03-06,2026-03-06,https://www.nic.br/noticia/na-midia/mpf-recomenda-que-prefeitura-de-alcantara-ma-instale-medidores-de-internet-em-escolas-publicas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Na era da IA, inclusão digital deve ser crítica, defende coordenadora da TIC Educação",2026-08-17,2026-08-17,https://www.nic.br/noticia/na-midia/na-era-da-ia-inclusao-digital-deve-ser-critica-defende-educadora/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,NIC.br abre inscrições para Desafio BCOP 2026,2026-05-18,2026-05-18,https://www.nic.br/noticia/na-midia/nic-br-abre-inscricoes-para-desafio-bcop-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,NIC.br abre inscrições para o Desafio BCOP 2026,2026-05-19,2026-05-19,https://www.nic.br/noticia/na-midia/nic-br-abre-inscricoes-para-o-desafio-bcop-2026-com-novas-categorias-para-provedores-e-usuarios-finais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,NIC.br vê riscos na revogação da Norma 4 e pede debate,2026-04-14,2026-04-14,https://www.nic.br/noticia/na-midia/nic-br-ve-riscos-na-revogacao-da-norma-4-e-pede-debate/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"No ambiente escolar, estudantes relatam \'pacto silencioso\' com docentes sobre uso de IA",2026-09-17,2026-09-17,https://www.nic.br/noticia/na-midia/no-ambiente-escolar-estudantes-relatam-pacto-silencioso-com-discentes-sobre-uso-de-ia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"No Brasil, 30% não estão preocupados com a verificação de informações",2026-04-14,2026-04-14,https://www.nic.br/noticia/na-midia/no-brasil-30-nao-estao-preocupados-com-a-verificacao-de-informacoes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Evento ""Princípios e diretrizes para regulação de plataformas digitais"" tem nova data",2026-05-06,2026-05-06,https://www.nic.br/noticia/na-midia/nova-data-para-debate-sobre-diretrizes-de-regulacao-das-plataformas-digitais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Novos decretos ampliam proteção às mulheres na Internet,2026-08-18,2026-08-18,https://www.nic.br/noticia/na-midia/novos-decretos-ampliam-protecao-as-mulheres-na-internet/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Novos fascículos do CERT.br focam na capacidade crítica e ações práticas do internauta,2026-04-02,2026-04-02,https://www.nic.br/noticia/na-midia/novos-fasciculos-do-cert-br-focam-na-capacidade-critica-e-acoes-praticas-do-internauta/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Nuvem está presente em 36% das empresas brasileiras,2026-06-16,2026-06-16,https://www.nic.br/noticia/na-midia/nuvem-esta-presente-em-36-das-empresas-brasileiras/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O Brasil no centro da revolução global da inteligência artificial,2026-08-28,2026-08-28,https://www.nic.br/noticia/na-midia/o-brasil-no-centro-da-revolucao-global-da-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O desafio da adoção da Inteligência Artificial não está no algoritmo,2026-08-11,2026-08-11,https://www.nic.br/noticia/na-midia/o-desafio-da-inteligencia-artificial-na-saude-nao-esta-no-algoritmo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O dilema das telas: como elas impactam o desenvolvimento infantil,2026-02-20,2026-02-20,https://www.nic.br/noticia/na-midia/o-dilema-das-telas-como-elas-impactam-o-desenvolvimento-infantil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O eleitor entre o real e o fabricado,2026-08-26,2026-08-26,https://www.nic.br/noticia/na-midia/o-eleitor-entre-o-real-e-o-fabricado/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O futuro do jornalismo está nas mãos das gerações que parecem mais ignorá-lo,2026-06-16,2026-06-16,https://www.nic.br/noticia/na-midia/o-futuro-do-jornalismo-esta-nas-maos-das-geracoes-que-parecem-mais-ignora-lo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O lado negligenciado no uso de IA no setor público,2026-09-11,2026-09-11,https://www.nic.br/noticia/na-midia/o-lado-negligenciado-no-uso-de-ia-no-setor-publico/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"O novo funcionário invisível da empresa precisa de crachá, chefe e limite",2026-07-07,2026-07-07,https://www.nic.br/noticia/na-midia/o-novo-funcionario-invisivel-da-empresa-precisa-de-cracha-chefe-e-limite/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O papel do setor privado na dependência tecnológica brasileira,2026-03-04,2026-03-04,https://www.nic.br/noticia/na-midia/o-papel-do-setor-privado-na-dependencia-tecnologica-brasileira/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O potencial tecnológico do Ceará é destaque no IX Fórum Fortaleza,2026-03-27,2026-03-27,https://www.nic.br/noticia/na-midia/o-potencial-tecnologico-do-ceara-e-destaque-no-ix-forum-fortaleza/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O que a obsessão por figurinhas e futebol esconde no cérebro infantil,2026-07-02,2026-07-02,https://www.nic.br/noticia/na-midia/o-que-a-obsessao-por-figurinhas-e-futebol-esconde-no-cerebro-infantil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O que acontece com suas fotos e conversas quando você usa um chatbot,2026-06-19,2026-06-19,https://www.nic.br/noticia/na-midia/o-que-acontece-com-suas-fotos-e-conversas-quando-voce-usa-um-i-chatbot-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O que as empresas estão fazendo para se adequarem ao ECA Digital,2026-02-11,2026-02-11,https://www.nic.br/noticia/na-midia/o-que-as-empresas-estao-fazendo-para-se-adequarem-ao-eca-digital/,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nnic,O que as gírias e os memes revelam sobre como estamos nos comunicando,2026-05-04,2026-05-04,https://www.nic.br/noticia/na-midia/o-que-as-girias-e-os-memes-revelam-sobre-como-estamos-nos-comunicando/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O que muda com o ECA Digital?,2026-03-20,2026-03-20,https://www.nic.br/noticia/na-midia/o-que-muda-com-o-eca-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O que significa a nova lei para as redes sociais,2026-05-22,2026-05-22,https://www.nic.br/noticia/na-midia/o-que-significa-a-nova-lei-para-as-redes-sociais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O risco de modernizar a precariedade na era da Lei de IA,2026-06-26,2026-06-26,https://www.nic.br/noticia/na-midia/o-risco-de-modernizar-a-precariedade-na-era-da-lei-de-ia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O Tecnobrega da regulação de plataformas: reflexões inspiradas por Belém,2026-06-10,2026-06-10,https://www.nic.br/noticia/na-midia/o-tecnobrega-da-regulacao-de-plataformas-reflexoes-inspiradas-por-belem/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"OKBR participa da instalação de nova Câmara do CGI.br sobre tecnologias emergentes, soberania e inovação",2026-03-06,2026-03-06,https://www.nic.br/noticia/na-midia/okbr-integra-nova-camara-do-cgi-br-sobre-tecnologias-emergentes-soberania-e-inovacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,ONGs brasileiras avançam no uso de IA generativa e computação em nuvem,2026-07-08,2026-07-08,https://www.nic.br/noticia/na-midia/ongs-brasileiras-avancam-no-uso-de-ia-generativa-e-computacao-em-nuvem/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,OpenCDN atinge 1 Tbps de tráfego agregado no Brasil,2026-04-29,2026-04-29,https://www.nic.br/noticia/na-midia/opencdn-atinge-1-tbps-de-trafego-agregado-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,OpenCDN atinge marca de 1 Tb/s de tráfego total,2026-04-28,2026-04-28,https://www.nic.br/noticia/na-midia/opencdn-atinge-marca-de-1-tb-s-de-trafego-total/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,OpenCDN bate recorde de dados e atinge marca de 1 Tb/s de tráfego total,2026-04-28,2026-04-28,https://www.nic.br/noticia/na-midia/opencdn-bate-recorde-de-dados-e-atinge-marca-de-1-tb-s-de-trafego-total/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,OpenCDN utiliza infraestrutura do IX.br para descentralizar tráfego,2026-04-29,2026-04-29,https://www.nic.br/noticia/na-midia/opencdn-chega-a-1-tbps-e-avanca-na-descentralizacao-de-trafego-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Os cinco problemas do Discord que colocaram a plataforma na mira das autoridades,2026-08-14,2026-08-14,https://www.nic.br/noticia/na-midia/os-cinco-problemas-do-discord-que-colocaram-a-plataforma-na-mira-das-autoridades/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Páginas de venda de ingressos viram alvo de golpistas,2026-02-26,2026-02-26,https://www.nic.br/noticia/na-midia/paginas-de-venda-de-ingressos-viram-alvo-de-golpistas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Páginas web ganharam mais importância na era da IA,2026-07-31,2026-07-31,https://www.nic.br/noticia/na-midia/paginas-i-web-i-ganharam-mais-importancia-na-era-da-ia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Para ser efetivo, ECA Digital depende de mudança de cultura, aponta debate",2026-04-08,2026-04-08,https://www.nic.br/noticia/na-midia/para-ser-efetivo-eca-digital-depende-de-mudanca-de-cultura-aponta-debate/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Participação de famílias é desafio do ECA Digital, dizem especialistas",2026-03-19,2026-03-19,https://www.nic.br/noticia/na-midia/participacao-de-familias-e-desafio-do-eca-digital-dizem-especialistas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Partido Verde e CNTE pedem que STF determine à Anatel liberação de Internet ilimitada 72 horas antes das eleições,2026-09-28,2026-09-28,https://www.nic.br/noticia/na-midia/partido-verde-e-cnte-pedem-que-stf-determine-a-anatel-liberacao-de-internet-ilimitada-72-horas-antes-das-eleicoes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Pesquisa aponta avanço da IA nas empresas brasileiras,2026-06-29,2026-06-29,https://www.nic.br/noticia/na-midia/pesquisa-aponta-avanco-da-inteligencia-artificial-nas-empresas-brasileiras/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Pesquisa mostra que 1 em cada 5 estabelecimentos de saúde no Brasil usa IA,2026-05-27,2026-05-27,https://www.nic.br/noticia/na-midia/pesquisa-mostra-que-1-em-cada-5-estabelecimentos-de-saude-no-brasil-usa-ia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Oito em cada dez escolas debatem impacto das telas na saúde mental,2026-08-10,2026-08-10,https://www.nic.br/noticia/na-midia/pesquisa-mostra-que-8-em-cada-10-escolas-debatem-impacto-das-telas-na-saude-mental/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Jornal Nacional destaca indicadores da TIC Educação 2025,2026-08-05,2026-08-05,https://www.nic.br/noticia/na-midia/pesquisadores-mediram-o-impacto-da-ia-na-vida-de-alunos-e-professores/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"PL da IA: CGI.br quer análise de risco, controle de biometria e regras mais fortes contra abuso sexual",2026-03-09,2026-03-09,https://www.nic.br/noticia/na-midia/pl-da-ia-cgi-br-quer-analise-de-risco-controle-de-biometria-e-regras-mais-fortes-contra-abuso-sexual/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Planos de governos querem Estado mais digital, falta o brasileiro ter acesso a ele",2026-09-30,2026-09-30,https://www.nic.br/noticia/na-midia/planos-de-governos-querem-estado-mais-digital-falta-o-brasileiro-ter-acesso-a-ele/,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nnic,Plataformas devem verificar idade de usuários a partir desta terça; veja como vai funcionar,2026-03-17,2026-03-17,https://www.nic.br/noticia/na-midia/plataformas-devem-verificar-idade-de-usuarios-a-partir-desta-terca-veja-como-vai-funcionar/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Plataformas digitais vencem TV e rádio na disputa pela atenção informacional do brasileiro,2026-04-13,2026-04-13,https://www.nic.br/noticia/na-midia/plataformas-digitais-vencem-tv-e-radio-na-disputa-pela-atencao-informacional-do-brasileiro/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Plataformas superam TV e rádio como fonte de informação no Brasil,2026-04-15,2026-04-15,https://www.nic.br/noticia/na-midia/plataformas-superam-tv-e-radio-como-fonte-de-informacao-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Por que a Internet cai com frequência no Brasil?,2026-02-13,2026-02-13,https://www.nic.br/noticia/na-midia/por-que-a-internet-cai-com-frequencia-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Por que apagar o foguinho do TikTok antes que cause um incêndio?,2026-02-10,2026-02-10,https://www.nic.br/noticia/na-midia/por-que-apagar-o-foguinho-do-tiktok-antes-que-cause-um-incendio/,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nnic,Pós-doutorado em aprendizado de máquina na Unicamp,2026-03-05,2026-03-05,https://www.nic.br/noticia/na-midia/pos-doutorado-em-aprendizado-de-maquina-na-unicamp/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,36% das empresas brasileiras pagam para ter serviços de nuvem,2026-06-17,2026-06-17,https://www.nic.br/noticia/na-midia/pouco-mais-de-36-das-empresas-brasileiras-pagam-para-ter-servicos-de-nuvem/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Prazo para participação de entidades nas eleições do CGI.br entra na reta final,2026-07-21,2026-07-21,https://www.nic.br/noticia/na-midia/prazo-para-participacao-de-entidades-nas-eleicoes-do-cgi-br-entra-na-reta-final/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Programação do Festival comKids 2026 conta com a participação de especialista do Cetic.br|NIC.br,2026-08-11,2026-08-11,https://www.nic.br/noticia/na-midia/programacao-do-festival-comkids-2026-conta-com-a-participacao-de-especialista-do-cetic-br-nic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Programe-se para as próximas iniciativas gratuitas do NIC.br,2026-06-02,2026-06-02,https://www.nic.br/noticia/na-midia/programe-se-para-as-proximas-iniciativas-gratuitas-do-nic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Proibição das bets : governo diz que já pediu o bloqueio de mais de 5,2 mil sites e retirada de quase 200 apps de lojas",2026-09-30,2026-09-30,https://www.nic.br/noticia/na-midia/proibicao-das-i-bets-i-governo-diz-que-ja-pediu-o-bloqueio-de-mais-de-5-2-mil-i-sites-i-e-retirada-de-quase-200-i-apps-i-de-lojas/,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nnic,Projeto Cidadão na Rede nas Escolas passa a integrar o Mapa Brasileiro da Educação Midiática,2026-07-16,2026-07-16,https://www.nic.br/noticia/na-midia/projeto-coordenado-pelo-ceptro-br-nic-br-agora-integra-o-mapa-brasileiro-da-educacao-midiatica/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Projeto da UFSC lança jogo educativo sobre segurança digital voltado para jovens e adolescentes,2026-02-23,2026-02-23,https://www.nic.br/noticia/na-midia/projeto-da-ufsc-lanca-jogo-educativo-sobre-seguranca-digital-voltado-para-jovens-e-adolescentes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Projeto do NIC.br atinge marca histórica de 1 Tb/s de tráfego total,2026-04-30,2026-04-30,https://www.nic.br/noticia/na-midia/projeto-do-nic-br-atinge-marca-historica-de-1-tb-s-de-trafego-total/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Projeto educacional leva robôs humanoides e IA às salas de aula no Centro-Oeste,2026-03-03,2026-03-03,https://www.nic.br/noticia/na-midia/projeto-educacional-leva-robos-humanoides-e-ia-as-salas-de-aula-no-centro-oeste/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Projeto mobiliza bibliotecas no combate à violência de gênero,2026-07-20,2026-07-20,https://www.nic.br/noticia/na-midia/projeto-mobiliza-bibliotecas-no-combate-a-violencia-de-genero/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Provedores regionais apostam em streaming e TV para ampliar receita e reduzir cancelamentos,2026-06-30,2026-06-30,https://www.nic.br/noticia/na-midia/provedores-regionais-apostam-em-streaming-e-tv-para-ampliar-receita-e-reduzir-cancelamentos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Psiquiatras criam critérios para diferenciar apostador patológico de recreativo,2026-07-31,2026-07-31,https://www.nic.br/noticia/na-midia/psiquiatras-criam-criterios-para-diferenciar-apostador-patologico-de-recreativo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Qualidade da conexão é monitorada em 102 mil escolas da Enec,2026-05-08,2026-05-08,https://www.nic.br/noticia/na-midia/qualidade-da-conexao-e-monitorada-em-102-mil-escolas-da-enec/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Quantas são e onde estão as bibliotecas públicas brasileiras,2026-06-08,2026-06-08,https://www.nic.br/noticia/na-midia/quantas-sao-e-onde-estao-as-bibliotecas-publicas-brasileiras/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Quase 100 mil empresas brasileiras fizeram uso de alguma Inteligência Artificial,2026-06-16,2026-06-16,https://www.nic.br/noticia/na-midia/quase-100-mil-empresas-brasileiras-fizeram-uso-de-alguma-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Quase 60% dos órgãos públicos federais e estaduais utilizam IA,2026-07-23,2026-07-23,https://www.nic.br/noticia/na-midia/quase-60-dos-orgaos-publicos-federais-e-estaduais-utilizam-ia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Quase metade dos brasileiros desconfia da mídia tradicional, aponta Painel TIC",2026-04-17,2026-04-17,https://www.nic.br/noticia/na-midia/quase-metade-dos-brasileiros-desconfia-da-midia-tradicional-aponta-painel-tic/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Quase um terço dos usuários de internet relatam tentativas de golpe,2026-09-01,2026-09-01,https://www.nic.br/noticia/na-midia/quase-um-terco-dos-usuarios-de-internet-relatam-tentativas-de-golpe/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Renata Mielli participa de debate sobre IA, saúde mental e proteção digital de jovens na Rádio Alepa FM",2026-05-29,2026-05-29,https://www.nic.br/noticia/na-midia/radio-alepa-fm-promove-debate-sobre-ia-saude-mental-e-protecao-digital-de-jovens-com-a-doutora-no-assunto-renata-mielli/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,MEC reabre prazos da Piec e reforça uso de medidor desenvolvido pelo NIC.br,2026-06-11,2026-06-11,https://www.nic.br/noticia/na-midia/reaberto-ciclo-2026-da-politica-de-educacao-conectada/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Redata vai à sanção e destrava obras de data centers,2026-09-14,2026-09-14,https://www.nic.br/noticia/na-midia/redata-vai-a-sancao-e-destrava-obras-de-i-data-centers-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Reflexões sobre um ecossistema de interconexão transformador,2026-02-24,2026-02-24,https://www.nic.br/noticia/na-midia/reflexoes-sobre-um-ecossistema-de-interconexao-transformador/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Regulação das redes sociais deveria ser implementada ou não?,2026-08-18,2026-08-18,https://www.nic.br/noticia/na-midia/regulacao-das-redes-sociais-deveria-ser-implementada-ou-nao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Regulação das redes sociais: regras podem variar conforme a empresa,2026-09-21,2026-09-21,https://www.nic.br/noticia/na-midia/regulacao-das-redes-sociais-regras-podem-variar-conforme-a-empresa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Regulação de plataformas volta ao centro do debate legislativo,2026-05-15,2026-05-15,https://www.nic.br/noticia/na-midia/regulacao-de-plataformas-volta-ao-centro-do-debate-legislativo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Regulamentação para acolher nova interpretação do Artigo 19 do MCI fortalecerá ANPD,2026-05-15,2026-05-15,https://www.nic.br/noticia/na-midia/regulamentacao-para-acolher-nova-interpretacao-do-artigo-19-do-mci-fortalecera-anpd/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Regulamentação para IA na educação mira veto à correção de questão dissertativa sem professor,2026-02-23,2026-02-23,https://www.nic.br/noticia/na-midia/regulamentacao-para-ia-na-educacao-mira-veto-a-correcao-de-questao-dissertativa-sem-professor/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Relato de endividamento por bets viraliza e acende alerta para avanço da ludopatia no país,2026-06-15,2026-06-15,https://www.nic.br/noticia/na-midia/relato-de-endividamento-por-i-bets-i-viraliza-e-acende-alerta-para-avanco-da-ludopatia-no-pais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Renata Mielli é a palestrante de abertura do Congresso Acadêmico Unifesp 2026,2026-07-28,2026-07-28,https://www.nic.br/noticia/na-midia/renata-mielli-e-a-palestrante-de-abertura-do-congresso-academico-unifesp-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Resolução que regula uso de IA na medicina entra em vigor,2026-08-26,2026-08-26,https://www.nic.br/noticia/na-midia/resolucao-que-regula-uso-de-ia-na-medicina-entra-em-vigor/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,RNP debate futuro da identidade digital acadêmica no 2º Workshop de Credenciais Verificáveis,2026-06-23,2026-06-23,https://www.nic.br/noticia/na-midia/rnp-debate-futuro-da-identidade-digital-academica-no-2-i-workshop-i-de-credenciais-verificaveis/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Robótica torna-se obrigatória nas escolas a partir de 2026,2026-07-20,2026-07-20,https://www.nic.br/noticia/na-midia/robotica-torna-se-obrigatoria-nas-escolas-a-partir-de-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Saber em quem confiar,2026-07-16,2026-07-16,https://www.nic.br/noticia/na-midia/saber-em-quem-confiar/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,O mito dos nativos digitais,2026-09-22,2026-09-22,https://www.nic.br/noticia/na-midia/saber-usar-o-celular-nao-e-saber-trabalhar-com-ele/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,SaferNet Brasil e Governo do Reino Unido lançam recurso pedagógico inédito sobre Cidadania Digital para crianças,2026-04-24,2026-04-24,https://www.nic.br/noticia/na-midia/safernet-brasil-e-governo-do-reino-unido-lancam-recurso-pedagogico-inedito-sobre-cidadania-digital-para-criancas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Satc participa de capacitação oferecida pelo CERT.br,2026-08-03,2026-08-03,https://www.nic.br/noticia/na-midia/satc-participa-de-capacitacao-oferecida-pelo-cert-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Saúde mental de estudantes: debate cresce em 80% das escolas, aponta TIC Educação",2026-08-05,2026-08-05,https://www.nic.br/noticia/na-midia/saude-mental-de-estudantes-debate-cresce-em-80-das-escolas-aponta-tic-educacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Se perdeu no ECA Digital? Entenda o que muda a partir de agora,2026-03-20,2026-03-20,https://www.nic.br/noticia/na-midia/se-perdeu-no-eca-digital-entenda-o-que-muda-a-partir-de-agora/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Sedigi debate regulação de plataformas digitais na Câmara dos Deputados,2026-05-15,2026-05-15,https://www.nic.br/noticia/na-midia/sedigi-debate-regulacao-de-plataformas-digitais-na-camara-dos-deputados/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Podcast “Nós da Internet” estreia segunda temporada,2026-08-06,2026-08-06,https://www.nic.br/noticia/na-midia/segunda-temporada-do-i-podcast-i-nos-da-internet/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Seis em cada dez órgãos públicos já usam inteligência artificial,2026-07-30,2026-07-30,https://www.nic.br/noticia/na-midia/seis-em-cada-dez-orgaos-publicos-ja-usam-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Sem alvará, sem dinheiro: a nova realidade dos influenciadores mirins",2026-06-24,2026-06-24,https://www.nic.br/noticia/na-midia/sem-alvara-sem-dinheiro-a-nova-realidade-dos-influenciadores-mirins/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Sem investir nos professores, a educação digital vai fracassar",2026-05-06,2026-05-06,https://www.nic.br/noticia/na-midia/sem-investir-nos-professores-a-educacao-digital-vai-fracassar/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Sem lei federal, escolas criam as próprias regras para uso de inteligência artificial",2026-08-26,2026-08-26,https://www.nic.br/noticia/na-midia/sem-lei-federal-escolas-criam-regra-propria-para-ia-na-aprendizagem/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Seminário marca abertura do curso jurídico da EGI,2026-03-27,2026-03-27,https://www.nic.br/noticia/na-midia/seminario-marca-abertura-do-curso-juridico-da-egi/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Seminários discutem privacidade e inteligência artificial,2026-08-24,2026-08-24,https://www.nic.br/noticia/na-midia/seminarios-discutem-privacidade-e-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Série “Eleições”: do algoritmo ao santinho eleitoral,2026-09-16,2026-09-16,https://www.nic.br/noticia/na-midia/serie-eleicoes-do-algoritmo-ao-santinho-eleitoral/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Serviços digitais não checam idade no momento de criação da conta,2026-03-20,2026-03-20,https://www.nic.br/noticia/na-midia/servicos-digitais-nao-checam-idade-no-momento-de-criacao-da-conta/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Serviços na nuvem e IA aumentam a importância da rede de longa distância gerenciada por software,2026-04-17,2026-04-17,https://www.nic.br/noticia/na-midia/servicos-na-nuvem-e-ia-aumentam-a-importancia-da-rede-de-longa-distancia-gerenciada-por-i-software-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,SMS de João Pessoa alerta para os riscos do uso excessivo de telas na infância,2026-07-13,2026-07-13,https://www.nic.br/noticia/na-midia/sms-de-joao-pessoa-alerta-para-os-riscos-do-uso-excessivo-de-telas-na-infancia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Apenas 10% dos jovens adotam postura ativa para checar informações na Internet, diz pesquisa",2026-09-08,2026-09-08,https://www.nic.br/noticia/na-midia/so-10-dos-jovens-checam-informacoes-que-veem-na-internet-diz-pesquisa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Sob pressão dos EUA, STF julga nesta quarta recursos contra decisão que aumentou responsabilidade das big techs",2026-06-10,2026-06-10,https://www.nic.br/noticia/na-midia/sob-pressao-dos-eua-stf-julga-nesta-quarta-recursos-contra-decisao-que-aumentou-responsabilidade-das-i-big-techs-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Soberania digital: as camadas invisíveis que organizam sua vida online,2026-07-14,2026-07-14,https://www.nic.br/noticia/na-midia/soberania-digital-as-camadas-invisiveis-que-organizam-sua-vida-i-online-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Soberania digital, eficiência e sustentabilidade: o novo marco dos data centers no Brasil",2026-07-16,2026-07-16,https://www.nic.br/noticia/na-midia/soberania-digital-eficiencia-e-sustentabilidade-o-novo-marco-dos-i-data-centers-i-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Sobre o desenvolvimento e uso responsáveis de Inteligência Artificial na educação,2026-06-15,2026-06-15,https://www.nic.br/noticia/na-midia/sobre-o-desenvolvimento-e-uso-responsaveis-de-inteligencia-artificial-na-educacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,STF forma maioria para que plataformas implementem em 60 dias alterações exigidas pela Corte,2026-06-12,2026-06-12,https://www.nic.br/noticia/na-midia/stf-forma-maioria-para-que-plataformas-implementem-em-60-dias-alteracoes-exigidas-pela-corte/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,STF julga recursos contra ampliação de responsabilidade de redes sociais,2026-06-11,2026-06-11,https://www.nic.br/noticia/na-midia/stf-julga-recursos-contra-ampliacao-de-responsabilidade-de-redes-sociais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,STF libera zero rating também para sites de agências parceiras de checagem da Fato ou Boato,2026-10-01,2026-10-01,https://www.nic.br/noticia/na-midia/stf-libera-i-zero-rating-i-tambem-para-i-sites-i-de-agencias-parceiras-de-checagem-da-fato-ou-boato/,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nnic,STF: Toffoli defende responsabilizar marketplaces e flexibiliza exigência de sede para big techs no país,2026-06-12,2026-06-12,https://www.nic.br/noticia/na-midia/stf-toffoli-defende-responsabilizar-i-marketplaces-i-e-flexibiliza-exigencia-de-sede-para-i-big-techs-i-no-pais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"\'Sua fisionomia hoje é uma senha\', alerta Demi Getschko em entrevista à Galileu",2026-08-10,2026-08-10,https://www.nic.br/noticia/na-midia/sua-fisionomia-hoje-e-uma-senha-alerta-demi-getschko-em-entrevista-a-galileu/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Tendências tecnológicas para o setor público em 2026,2026-03-02,2026-03-02,https://www.nic.br/noticia/na-midia/tendencias-tecnologicas-para-o-setor-publico-em-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Tentativas de golpes por meios como mensagens ou ligações afetam 45 milhões no Brasil, aponta Cetic.br",2026-09-01,2026-09-01,https://www.nic.br/noticia/na-midia/tentativas-de-golpes-por-meios-como-mensagens-ou-ligacoes-afetam-45-milhoes-no-brasil-aponta-cetic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,TIC Educação: 80% das escolas debatem uso das telas e saúde mental,2026-08-12,2026-08-12,https://www.nic.br/noticia/na-midia/tic-educacao-80-das-escolas-debatem-uso-das-telas-e-saude-mental/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,TIC Educação: celulares são liberados em escolas sem acesso à Internet e aos computadores,2026-08-07,2026-08-07,https://www.nic.br/noticia/na-midia/tic-educacao-celulares-sao-liberados-em-escolas-sem-acesso-a-internet-e-aos-computadores/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,TIC Governo aponta crescimento de uso de IA nas prefeituras,2026-07-29,2026-07-29,https://www.nic.br/noticia/na-midia/tic-governo-aponta-que-cresce-uso-de-ia-nas-prefeituras/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,TIC Governo: Seis em cada dez órgãos federais e estaduais já usam inteligência artificial,2026-07-23,2026-07-23,https://www.nic.br/noticia/na-midia/tic-governo-seis-em-cada-10-orgaos-federais-e-estaduais-ja-usam-inteligencia-artificial/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,TIC Organizações Sem Fins Lucrativos 2025 identifica avanço na adoção de tecnologias no setor,2026-07-07,2026-07-07,https://www.nic.br/noticia/na-midia/tic-organizacoes-sem-fins-lucrativos-2025-identifica-avanco-na-adocao-de-tecnologias-no-setor/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Chatbots viraram a nova enciclopédia dos filhos; qual é o papel dos pais e responsáveis na era digital?,2026-08-19,2026-08-19,https://www.nic.br/noticia/na-midia/tiktok-e-chatgpt-viraram-a-nova-enciclopedia-dos-filhos-qual-e-o-papel-dos-pais-na-era-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Todos usam IA; poucos sabem por quê,2026-03-26,2026-03-26,https://www.nic.br/noticia/na-midia/todos-usam-ia-poucos-sabem-por-que/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Transformação digital: do clique à conversa (e da promessa à resolução),2026-02-20,2026-02-20,https://www.nic.br/noticia/na-midia/transformacao-digital-do-clique-a-conversa-e-da-promessa-a-resolucao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"TSE cria pacote contra deepfakes com IA nas eleições, mas batalha está só no começo",2026-03-16,2026-03-16,https://www.nic.br/noticia/na-midia/tse-cria-pacote-contra-i-deepfakes-i-com-ia-nas-eleicoes-mas-batalha-esta-so-no-comeco/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,TSE lança ferramenta com orientação de plataformas digitais para eleições,2026-08-31,2026-08-31,https://www.nic.br/noticia/na-midia/tse-lanca-ferramenta-com-orientacao-de-plataformas-digitais-para-eleicoes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,UFSCar promove 1º encontro estadual sobre Interação Humano-Computador,2026-06-23,2026-06-23,https://www.nic.br/noticia/na-midia/ufscar-sedia-1-encontro-estadual-sobre-interacao-humano-computador/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Um dia para a defesa dos Direitos Humanos na Internet,2026-02-10,2026-02-10,https://www.nic.br/noticia/na-midia/um-dia-para-a-defesa-dos-direitos-humanos-na-internet/,2026-09-28T21:01:12+00:00,2026-09-28T21:01:12+00:00\nnic,Um em cada três usuários de Internet no Brasil relatou sofrer tentativa de golpe com uso de seus dados pessoais,2026-09-02,2026-09-02,https://www.nic.br/noticia/na-midia/um-em-cada-tres-usuarios-de-internet-no-brasil-relatou-sofrer-tentativa-de-golpe-com-uso-de-seus-dados-pessoais-aponta-cetic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Um guia da dieta de mídia digital brasileira,2026-04-08,2026-04-08,https://www.nic.br/noticia/na-midia/um-guia-da-dieta-de-midia-digital-brasileira/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Um referencial para o uso da inteligência artificial na educação,2026-07-02,2026-07-02,https://www.nic.br/noticia/na-midia/um-referencial-para-o-uso-da-inteligencia-artificial-na-educacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Um terço das ONGs brasileiras já usa IA generativa, mostra pesquisa",2026-07-14,2026-07-14,https://www.nic.br/noticia/na-midia/um-terco-das-ongs-brasileiras-ja-usa-ia-generativa-mostra-pesquisa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Um terço das organizações sem fins de lucro no Brasil já usa IA generativa, aponta pesquisa do Cetic.br",2026-07-07,2026-07-07,https://www.nic.br/noticia/na-midia/um-terco-das-organizacoes-sem-fins-de-lucro-no-brasil-ja-usa-ia-generativa-aponta-pesquisa-do-cetic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,UnB realiza seminário para debater desafios contemporâneos,2026-08-14,2026-08-14,https://www.nic.br/noticia/na-midia/unb-realiza-seminario-para-debater-desafios-contemporaneos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Unesco aponta para baixa implementação da educação midiática nos currículos mundo afora,2026-02-26,2026-02-26,https://www.nic.br/noticia/na-midia/unesco-aponta-para-baixa-implementacao-da-educacao-midiatica-nos-curriculos-mundo-afora/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,UNESCO e Cetic.br/NIC.br lançam Observatório de Inteligência Artificial na Educação para a América Latina e o Caribe,2026-04-27,2026-04-27,https://www.nic.br/noticia/na-midia/unesco-lanca-observatorio-de-inteligencia-artificial-na-educacao-para-a-america-latina-e-o-caribe/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Uso de IA avança e já está presente em 59% dos órgãos públicos brasileiros,2026-07-27,2026-07-27,https://www.nic.br/noticia/na-midia/uso-de-ia-avanca-e-ja-esta-presente-em-59-dos-orgaos-publicos-brasileiros/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Uso de IA generativa entre jovens amplia debate sobre formação tecnológica e sustentabilidade,2026-06-25,2026-06-25,https://www.nic.br/noticia/na-midia/uso-de-ia-generativa-entre-jovens-amplia-debate-sobre-formacao-tecnologica-e-sustentabilidade/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Uso de IA na saúde chega a 18% dos estabelecimentos do país,2026-05-13,2026-05-13,https://www.nic.br/noticia/na-midia/uso-de-ia-na-saude-chega-a-18-dos-estabelecimentos-do-pais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Uso de IA nas empresas brasileiras cresce 4% no último ano,2026-06-17,2026-06-17,https://www.nic.br/noticia/na-midia/uso-de-ia-nas-empresas-brasileiras-cresce-4-no-ultimo-ano/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Uso de inteligência artificial cresce no setor de saúde, com foco em tarefas administrativa",2026-05-19,2026-05-19,https://www.nic.br/noticia/na-midia/uso-de-inteligencia-artificial-cresce-no-setor-de-saude-com-foco-em-tarefas-administrativa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Uso de Internet por crianças entre 6 e 8 anos dobrou na última década,2026-03-13,2026-03-13,https://www.nic.br/noticia/na-midia/uso-de-internet-por-criancas-entre-6-e-8-anos-dobrou-na-ultima-decada/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Uso inteligente da tecnologia na gestão escolar,2026-03-12,2026-03-12,https://www.nic.br/noticia/na-midia/uso-inteligente-da-tecnologia-na-gestao-escolar/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Vale do Silício forjou e dissemina design enganoso de apps vetado no Brasil,2026-03-27,2026-03-27,https://www.nic.br/noticia/na-midia/vale-do-silicio-forjou-e-dissemina-i-design-i-enganoso-de-i-apps-i-vetado-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Medidor Simet ajuda consumidor a verificar qualidade de banda larga fixa,2026-07-22,2026-07-22,https://www.nic.br/noticia/na-midia/velocidade-e-contratada-nao-garantida/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Verificação de idade e privacidade: convivência impossível?,2026-03-02,2026-03-02,https://www.nic.br/noticia/na-midia/verificacao-de-idade-e-privacidade-convivencia-impossivel/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Vício em apostas pode ter raízes ainda na infância, alertam especialistas",2026-07-13,2026-07-13,https://www.nic.br/noticia/na-midia/vicio-em-apostas-pode-ter-raizes-ainda-na-infancia-alertam-especialistas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Vídeos falsos e IA superam fake news como ameaça à democracia, alerta novo guia do CGI.br",2026-09-25,2026-09-25,https://www.nic.br/noticia/na-midia/videos-falsos-e-ia-superam-i-fake-news-i-como-ameaca-a-democracia-alerta-novo-guia-do-cgi-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Você não está atrasado: 7 passos para aprender IA em menos de 30 minutos,2026-05-05,2026-05-05,https://www.nic.br/noticia/na-midia/voce-nao-esta-atrasado-7-passos-para-aprender-ia-em-menos-de-30-minutos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,WhatsApp amplia ferramentas de controle parental no Brasil,2026-10-01,2026-10-01,https://www.nic.br/noticia/na-midia/whatsapp-amplia-ferramentas-de-controle-parental-no-brasil/,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nnic,Wi-Fi doméstico é desafio para provedores de banda larga,2026-05-11,2026-05-11,https://www.nic.br/noticia/na-midia/wi-fi-domestico-e-desafio-para-provedores-de-banda-larga/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Acessibilidade na Web: W3C reconhecido pela Forbes,2026-05-22,2026-05-22,https://www.nic.br/noticia/notas/acessibilidade-na-web-w3c-reconhecido-pela-forbes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Brasil tem representação em Grupo de Assessoramento ao IGF 2026,2026-08-03,2026-08-03,https://www.nic.br/noticia/notas/brasil-tem-representacao-em-grupo-de-assessoramento-ao-igf-2026/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Cetic.br integra Observatório regional de IA na educação liderado pela UNESCO,2026-04-28,2026-04-28,https://www.nic.br/noticia/notas/cetic-br-integra-observatorio-regional-de-ia-na-educacao-liderado-pela-unesco/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Ceweb.br lança podcast para estimular aplicação da norma brasileira de acessibilidade digital,2026-03-11,2026-03-11,https://www.nic.br/noticia/notas/ceweb-br-lanca-podcast-para-estimular-aplicacao-da-norma-brasileira-de-acessibilidade-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Especialistas associam regulação da IA e de plataformas digitais à soberania nacional,2026-05-29,2026-05-29,https://www.nic.br/noticia/notas/especialistas-associam-regulacao-da-ia-e-de-plataformas-digitais-a-soberania-nacional/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Fórum da Internet no Brasil debate caminhos multissetoriais para a soberania digital,2026-05-28,2026-05-28,https://www.nic.br/noticia/notas/forum-da-internet-no-brasil-debate-caminhos-multissetoriais-para-a-soberania-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA amplia assimetria na segurança digital e exige novas estratégias de defesa, avalia CERT.br",2026-08-03,2026-08-03,https://www.nic.br/noticia/notas/ia-amplia-assimetria-na-seguranca-digital-e-exige-novas-estrategias-de-defesa-avalia-cert-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,NIC.br integra nova composição do Conselho Diretor da LACTLD,2026-06-22,2026-06-22,https://www.nic.br/noticia/notas/nic-br-integra-nova-composicao-do-conselho-diretor-da-lactld/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Nota de pesar: Bertrand de La Chapelle, pioneiro na implementação de processos de governança multissetoriais",2026-08-26,2026-08-26,https://www.nic.br/noticia/notas/nota-de-pesar-bertrand-de-la-chapelle-pioneiro-na-implementacao-de-processos-de-governanca-multissetoriais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Parlamentares da Comissão de Ciência, Tecnologia e Inovação da Câmara dos Deputados visitam instalações do NIC.br em SP",2026-05-14,2026-05-14,https://www.nic.br/noticia/notas/parlamentares-da-comissao-de-ciencia-tecnologia-e-inovacao-da-camara-dos-deputados-visitam-instalacoes-do-nic-br-em-sp/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Presidente e CEO da ICANN visita instalações do NIC.br em São Paulo,2026-09-16,2026-09-16,https://www.nic.br/noticia/notas/presidente-e-ceo-da-icann-visita-instalacoes-do-nic-br-em-sao-paulo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Statement of Condolence: Bertrand de La Chapelle, Pioneer in the Implementation of Multistakeholder Governance Processes",2026-08-26,2026-08-26,https://www.nic.br/noticia/notas/statement-of-condolence-bertrand-de-la-chapelle-pioneer-in-the-implementation-of-multistakeholder-governance-processes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Transparência e proteção infantojuvenil centralizam debate sobre ECA Digital no FIB16,2026-05-29,2026-05-29,https://www.nic.br/noticia/notas/transparencia-e-protecao-infantojuvenil-centralizam-debate-sobre-eca-digital-no-fib16/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Valorização profissional e sustentabilidade criativa são desafios enfrentados por produtores culturais na era da IA, apontam especialistas",2026-05-28,2026-05-28,https://www.nic.br/noticia/notas/valorizacao-profissional-e-sustentabilidade-criativa-sao-desafios-enfrentados-por-produtores-culturais-na-era-da-ia-apontam-especialistas/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,17º Fórum da Internet no Brasil recebe propostas de workshops e estreia formato de mesa-redonda,2026-07-29,2026-07-29,https://www.nic.br/noticia/releases/17-forum-da-internet-no-brasil-recebe-propostas-de-i-workshops-i-e-estreia-formato-de-mesa-redonda/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,17º Seminário de Proteção de Dados debate os desafios da implementação do ECA Digital e o impacto da IA nas eleições,2026-08-28,2026-08-28,https://www.nic.br/noticia/releases/17-seminario-de-protecao-de-dados-debate-os-desafios-da-implementacao-do-eca-digital-e-o-impacto-da-ia-nas-eleicoes/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"2ª edição do Workshop de Credenciais Verificáveis debate privacidade, soberania de dados e verificação de idade na Internet",2026-06-10,2026-06-10,https://www.nic.br/noticia/releases/2-edicao-do-i-workshop-i-de-credenciais-verificaveis-debate-privacidade-soberania-de-dados-e-verificacao-de-idade-na-internet/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"3º Seminário do OBIA traz debates sobre sustentabilidade, governança e estado da IA no Brasil",2026-08-28,2026-08-28,https://www.nic.br/noticia/releases/3-seminario-do-obia-traz-debates-sobre-sustentabilidade-governanca-e-estado-da-ia-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Caiu em um golpe? Novo fascículo do CERT.br dá orientações sobre como agir,2026-05-26,2026-05-26,https://www.nic.br/noticia/releases/caiu-em-um-golpe-novo-fasciculo-do-cert-br-da-orientacoes-sobre-como-agir/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CERT.br lança novos fascículos da Cartilha de Segurança para Internet com foco na prevenção de golpes e fraudes online,2026-03-31,2026-03-31,https://www.nic.br/noticia/releases/cert-br-lanca-novos-fasciculos-da-cartilha-de-seguranca-para-internet-com-foco-na-prevencao-de-golpes-e-fraudes-i-online-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br abre inscrições para 16º Fórum da Internet no Brasil e divulga workshops selecionados,2026-03-11,2026-03-11,https://www.nic.br/noticia/releases/cgi-br-abre-inscricoes-para-16-forum-da-internet-no-brasil-e-divulga-i-workshops-i-selecionados/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br apresenta 46 diretrizes para embasar a regulação de redes sociais no Brasil,2026-09-17,2026-09-17,https://www.nic.br/noticia/releases/cgi-br-apresenta-46-diretrizes-para-embasar-a-regulacao-de-redes-sociais-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,CGI.br e NIC.br abrem inscrições para 17º Seminário de Proteção à Privacidade e para o 3º Seminário do OBIA,2026-08-19,2026-08-19,https://www.nic.br/noticia/releases/cgi-br-e-nic-br-abrem-inscricoes-para-17-seminario-de-protecao-a-privacidade-e-para-o-3-seminario-do-obia/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Curso online gratuito sobre IA e Transformação Digital no Governo ganha versão em português,2026-07-02,2026-07-02,https://www.nic.br/noticia/releases/curso-i-online-i-gratuito-sobre-ia-e-transformacao-digital-no-governo-ganha-versao-em-portugues/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Escola de Governança da Internet no Brasil do CGI.br abre inscrições para 5ª edição de curso EaD gratuito,2026-08-20,2026-08-20,https://www.nic.br/noticia/releases/escola-de-governanca-da-internet-no-brasil-do-cgi-br-abre-inscricoes-para-5-edicao-de-curso-ead-gratuito/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Estudo inédito mapeia práticas de aferição de idade em serviços digitais no Brasil,2026-03-18,2026-03-18,https://www.nic.br/noticia/releases/estudo-inedito-mapeia-praticas-de-afericao-de-idade-em-servicos-digitais-no-brasil/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Evento debate perspectivas globais e multissetoriais para a implementação do ECA Digital,2026-03-10,2026-03-10,https://www.nic.br/noticia/releases/evento-debate-perspectivas-globais-e-multissetoriais-para-a-implementacao-do-eca-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Free online course on AI and Digital Transformation in Government now available in Portuguese,2026-07-02,2026-07-02,https://www.nic.br/noticia/releases/free-online-course-on-ai-and-digital-transformation-in-government-now-available-in-portuguese/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IA está presente em seis de cada dez órgãos públicos federais e estaduais e chega a 94% no Judiciário, aponta TIC Governo",2026-07-23,2026-07-23,https://www.nic.br/noticia/releases/ia-esta-presente-em-seis-de-cada-dez-orgaos-publicos-federais-e-estaduais-e-chega-a-94-no-judiciario-aponta-tic-governo/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IX.br bate recorde de 50 Tbit/s de tráfego Internet agregado, impulsionado por conteúdo e serviços digitais",2026-03-20,2026-03-20,https://www.nic.br/noticia/releases/ix-br-bate-recorde-de-50-tbit-s-de-trafego-internet-agregado-impulsionado-por-conteudo-e-servicos-digitais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IX.br hits record 50 Tbit/s of aggregated Internet traffic, driven by content and digital services",2026-03-27,2026-03-27,https://www.nic.br/noticia/releases/ix-br-hits-record-50-tbit-s-of-aggregated-internet-traffic-driven-by-content-and-digital-services/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,IX.br inaugura no Amapá 40º Ponto de Troca de Tráfego e alcança todos os estados brasileiros,2026-09-23,2026-09-23,https://www.nic.br/noticia/releases/ix-br-inaugura-no-amapa-40-ponto-de-troca-de-trafego-e-alcanca-todos-os-estados-brasileiros/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"IX Fórum Fortaleza 2026 debate bloqueios na Internet, mudanças regulatórias e uso da faixa de 6 GHz para Wi-Fi",2026-03-17,2026-03-17,https://www.nic.br/noticia/releases/ix-forum-fortaleza-2026-debate-bloqueios-na-internet-mudancas-regulatorias-e-uso-da-faixa-de-6-ghz-para-i-wi-fi-i/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,NIC.br abre inscrições para o Desafio BCOP 2026 e inaugura novas categorias de participação,2026-05-13,2026-05-13,https://www.nic.br/noticia/releases/nic-br-abre-inscricoes-para-o-desafio-bcop-2026-e-inaugura-novas-categorias-de-participacao/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,NIC.br e CGI.br abrem inscrições gratuitas para o 11º Simpósio Crianças e Adolescentes na Internet,2026-09-30,2026-09-30,https://www.nic.br/noticia/releases/nic-br-e-cgi-br-abrem-inscricoes-gratuitas-para-o-11-simposio-criancas-e-adolescentes-na-internet/,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nnic,OpenCDN atinge marca de 1 Tb/s de tráfego total,2026-04-28,2026-04-28,https://www.nic.br/noticia/releases/opencdn-atinge-marca-de-1-tb-s-de-trafego-total/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Plataformas da Internet são o principal meio de acesso à informação entre brasileiros, e superam rádio e TV, aponta nova pesquisa",2026-04-10,2026-04-10,https://www.nic.br/noticia/releases/plataformas-da-internet-sao-o-principal-meio-de-acesso-a-informacao-entre-brasileiros-e-superam-radio-e-tv-aponta-nova-pesquisa/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Programação do FIB16 terá debates sobre IA, cibersegurança e proteção de crianças e adolescentes no ambiente digital",2026-05-08,2026-05-08,https://www.nic.br/noticia/releases/programacao-do-fib16-tera-debates-sobre-ia-ciberseguranca-e-protecao-de-criancas-e-adolescentes-no-ambiente-digital/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,TIC Educação 2025 mostra que 8 em cada 10 escolas brasileiras debatem impacto do uso de telas na saúde mental dos alunos,2026-08-04,2026-08-04,https://www.nic.br/noticia/releases/tic-educacao-2025-mostra-que-8-em-cada-10-escolas-brasileiras-debatem-impacto-do-uso-de-telas-na-saude-mental-dos-alunos/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Um em cada três usuários de Internet no Brasil relatou sofrer tentativa de golpe com uso de seus dados pessoais, aponta Cetic.br",2026-08-31,2026-08-31,https://www.nic.br/noticia/releases/um-em-cada-tres-usuarios-de-internet-no-brasil-relatou-sofrer-tentativa-de-golpe-com-uso-de-seus-dados-pessoais-aponta-cetic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Um terço das organizações sem fins de lucro no Brasil já usa IA generativa, aponta pesquisa do Cetic.br",2026-07-07,2026-07-07,https://www.nic.br/noticia/releases/um-terco-das-organizacoes-sem-fins-de-lucro-no-brasil-ja-usa-ia-generativa-aponta-pesquisa-do-cetic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Uso de Inteligência Artificial avança na saúde brasileira, mas ainda se concentra em tarefas operacionais",2026-05-12,2026-05-12,https://www.nic.br/noticia/releases/uso-de-inteligencia-artificial-avanca-na-saude-brasileira-mas-ainda-se-concentra-em-tarefas-operacionais/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,"Uso de Inteligência Artificial por empresas brasileiras avança e atinge 17%, aponta pesquisa do Cetic.br",2026-06-12,2026-06-12,https://www.nic.br/noticia/releases/uso-de-inteligencia-artificial-por-empresas-brasileiras-avanca-e-atinge-17-aponta-pesquisa-do-cetic-br/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nnic,Vai usar IA? Novo fascículo do CERT.br dá dicas de como tirar proveito da tecnologia de forma segura e responsável,2026-09-03,2026-09-03,https://www.nic.br/noticia/releases/vai-usar-ia-novo-fasciculo-do-cert-br-da-dicas-de-como-tirar-proveito-da-tecnologia-de-forma-segura-e-responsavel/,2026-09-28T21:01:12+00:00,2026-10-01T16:58:06+00:00\nsenado_federal,Editadas medidas provisórias contra bets e endividamento das famílias,2026-09-25,25/09/2026 20h27,https://www12.senado.leg.br/noticias/materias/2026/09/25/poder-executivo-anuncia-mps-contra-bets-e-endividamento-das-familias,2026-09-28T21:01:12+00:00,2026-09-28T21:01:12+00:00\nsenado_federal,Sessão destaca excelência do ensino da Escola Superior de Ciências da Saúde,2026-09-25,25/09/2026 18h09,https://www12.senado.leg.br/noticias/materias/2026/09/25/sessao-destaca-excelencia-do-ensino-da-escola-superior-de-ciencias-da-saude,2026-09-28T21:01:12+00:00,2026-09-28T21:01:12+00:00\nsenado_federal,Chega ao Congresso projeto que torna crime a exploração de bets,2026-09-28,28/09/2026 16h17,https://www12.senado.leg.br/noticias/materias/2026/09/28/chega-ao-congresso-projeto-que-torna-crime-a-exploracao-de-bets,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nsenado_federal,Congresso passa a ter agendamento prévio para visitação,2026-09-28,28/09/2026 14h52,https://www12.senado.leg.br/noticias/materias/2026/09/28/congresso-passa-a-ter-agendamento-previo-para-visitacao,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nsenado_federal,Desenrola Brasil 3.0 já está em vigor; veja como funciona,2026-09-28,28/09/2026 11h59,https://www12.senado.leg.br/noticias/materias/2026/09/28/desenrola-brasil-3-0-ja-esta-em-vigor-veja-como-funciona,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nsenado_federal,Eleições 2026: veja o que é fato sobre o voto para o Senado,2026-09-28,28/09/2026 11h43,https://www12.senado.leg.br/noticias/materias/2026/09/28/eleicoes-2026-veja-o-que-e-fato-sobre-o-voto-para-o-senado,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nsenado_federal,MP proíbe bets e determina encerramento das operações em 30 dias,2026-09-28,28/09/2026 10h51,https://www12.senado.leg.br/noticias/materias/2026/09/28/mp-proibe-bets-e-determina-encerramento-das-operacoes-em-30-dias,2026-09-28T21:01:12+00:00,2026-09-29T23:31:45+00:00\nsenado_federal,Lei determina avaliação da qualidade de hospitais,2026-09-29,29/09/2026 16h27,https://www12.senado.leg.br/noticias/materias/2026/09/29/lei-determina-avaliacao-da-qualidade-de-hospitais,2026-09-29T23:31:45+00:00,2026-10-01T16:53:19+00:00\nsenado_federal,Lei reduz CSLL de resseguradoras locais para 9% a partir de 2027,2026-09-29,29/09/2026 16h21,https://www12.senado.leg.br/noticias/materias/2026/09/29/lei-reduz-csll-de-resseguradoras-locais-para-9-a-partir-de-2027,2026-09-29T23:31:45+00:00,2026-09-29T23:31:45+00:00\nsenado_federal,Lei torna futebol feminino prioridade na política de esporte,2026-09-29,29/09/2026 09h53,https://www12.senado.leg.br/noticias/materias/2026/09/29/lei-torna-futebol-feminino-prioridade-na-politica-de-esporte-1,2026-09-29T23:31:45+00:00,2026-09-29T23:31:45+00:00\nsenado_federal,Sancionado repasse de loterias para Comitê Brasileiro de Clubes,2026-09-29,29/09/2026 10h34,https://www12.senado.leg.br/noticias/materias/2026/09/29/sancionado-repasse-de-loterias-para-comite-brasileiro-de-clubes,2026-09-29T23:31:45+00:00,2026-09-29T23:31:45+00:00\nsenado_federal,Confira o que pode e o que não pode até o dia da votação,2026-09-30,30/09/2026 12h00,https://www12.senado.leg.br/noticias/materias/2026/09/30/confira-o-que-pode-e-o-que-nao-pode-ate-o-dia-da-votacao,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nsenado_federal,Lei fortalece seguro rural e fundo para catástrofes que atinjam a produção,2026-09-30,30/09/2026 17h32,https://www12.senado.leg.br/noticias/materias/2026/09/30/lei-fortalece-seguro-rural-e-fundo-para-catastrofes-que-atinjam-a-producao,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nsenado_federal,Renzo Braz toma posse como senador por Minas Gerais,2026-09-30,30/09/2026 16h44,https://www12.senado.leg.br/noticias/materias/2026/09/30/renzo-braz-toma-posse-como-senador-por-minas-gerais,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nsenado_federal,CCS debate audiovisual e comunicação pública na segunda-feira,2026-10-01,01/10/2026 11h47,https://www12.senado.leg.br/noticias/materias/2026/10/01/ccs-debate-audiovisual-e-comunicacao-publica-na-segunda-feira-5,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\nsenado_federal,CDH faz na segunda último debate de ciclo sobre violência contra criança,2026-10-01,01/10/2026 09h52,https://www12.senado.leg.br/noticias/materias/2026/10/01/cdh-faz-na-segunda-ultimo-debate-de-ciclo-sobre-violencia-contra-crianca,2026-10-01T16:45:22+00:00,2026-10-01T16:58:06+00:00\n', 'data/runs/latest.json': '{\n  "generated_at": "2026-10-01T16:58:06+00:00",\n  "dataset": "data/news.csv",\n  "total_records": 592,\n  "published_last_7_days": 35,\n  "sources": [\n    {\n      "source_id": "nic",\n      "status": "ok",\n      "seen": 489,\n      "new": 7\n    },\n    {\n      "source_id": "edpb",\n      "status": "ok",\n      "seen": 16,\n      "new": 0\n    },\n    {\n      "source_id": "mercociudades",\n      "status": "ok",\n      "seen": 50,\n      "new": 1\n    },\n    {\n      "source_id": "senado_federal",\n      "status": "ok",\n      "seen": 5,\n      "new": 5\n    }\n  ],\n  "notebooks": {\n    "count": 23,\n    "syntax_ok": 23,\n    "results": [\n      {\n        "path": "Analise/agente_fapesp_unificado.ipynb",\n        "code_cells": 8,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Curadoria/Agente_curadoria_FAPESP.ipynb",\n        "code_cells": 11,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/Parlamento uruguaio/parlamento_uy.ipynb",\n        "code_cells": 14,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/aaip.ipynb",\n        "code_cells": 19,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/camaraFederal.ipynb",\n        "code_cells": 11,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/cgi.ipynb",\n        "code_cells": 6,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/dataprivacy.ipynb",\n        "code_cells": 9,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/icncongresoargentina.ipynb",\n        "code_cells": 6,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/internetlab.ipynb",\n        "code_cells": 15,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/mercociudades.ipynb",\n        "code_cells": 5,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/nic_.ipynb",\n        "code_cells": 10,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/observacom.ipynb",\n        "code_cells": 9,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/senado.ipynb",\n        "code_cells": 23,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/senadofederal.ipynb",\n        "code_cells": 18,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "Mercosul/urcdp.ipynb",\n        "code_cells": 14,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "UE/charts.ipynb",\n        "code_cells": 11,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "UE/comissaoEuropeia.ipynb",\n        "code_cells": 6,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "UE/database_analysis.ipynb",\n        "code_cells": 7,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "UE/edpb.ipynb",\n        "code_cells": 22,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "UE/google_transparency.ipynb",\n        "code_cells": 21,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "UE/icann.ipynb",\n        "code_cells": 6,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "UE/uniaoEuropeia.ipynb",\n        "code_cells": 5,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      },\n      {\n        "path": "UE/uniaoEuropeia_news (EUR-Lex).ipynb",\n        "code_cells": 16,\n        "status": "syntax_ok",\n        "errors": [],\n        "runtime_status": "not_executed"\n      }\n    ]\n  },\n  "scope": "Coleta recente de 4 fontes; verificação sintática dos 21 notebooks originais e dos 2 agentes. Não executa Colab."\n}\n', 'data/curadoria/Catalogo_de_fontes_FAPESP.md': '# Catálogo de fontes — relatório final FAPESP\n\nAtualizado em 25 de setembro de 2026.\n\nEste catálogo reúne fontes selecionadas para documentar a pesquisa sobre o Brasil como mediador ou ponte regulatória na governança digital. A prioridade é: (1) documentos oficiais e fontes primárias; (2) artigos e relatórios acadêmicos; e (3) notícias ou entrevistas com valor documental. A caracterização do Brasil como “ponte” é uma hipótese analítica, não um fato presumido.\n\n## 1. Documentos oficiais e fontes primárias\n\n### Metodologia de testagem do Sandbox Regulatório em IA e Proteção de Dados\n\n- **Instituição:** Agência Nacional de Proteção de Dados (ANPD), com metodologia desenvolvida pelo CIAAM/USP\n- **Data:** 19 de agosto de 2026\n- **Tipo:** metodologia institucional oficial\n- **Link:** https://www.gov.br/anpd/pt-br/assuntos/noticias/anpd-publica-metodologia-de-testagem-do-sandbox-regulatorio-em-inteligencia-artificial-e-protecao-de-dados\n- **Uso no relatório:** evidência de que a ANPD transforma princípios de gestão de riscos, transparência e accountability em procedimentos próprios de experimentação regulatória.\n- **Classificação:** evidência documental primária; a leitura como tradução de padrões europeus é interpretação da pesquisa.\n- **Acesso:** aberto.\n\n### Ata nº 01/2026 do Grupo Agenda Digital do Mercosul\n\n- **Instituição:** Grupo Agenda Digital do Mercosul\n- **Data:** 30 de abril de 2026\n- **Tipo:** ata oficial regional\n- **Link:** https://calendario.mercosur.int/simfiles/docreuniones/111473_GAD_2026_ACTA01_ES.pdf\n- **Uso no relatório:** prova documental de articulação regional entre autoridades de proteção de dados e/ou inteligência artificial; permite investigar o papel efetivo do Brasil na circulação de práticas.\n- **Classificação:** evidência documental primária; liderança brasileira precisa ser demonstrada por comparação com outras atas e propostas.\n- **Acesso:** aberto.\n\n### Decisões recíprocas de adequação entre Brasil e União Europeia\n\n- **Instituições:** Comissão Europeia e ANPD\n- **Data:** 26 de janeiro de 2026\n- **Tipo:** decisão oficial de adequação em proteção de dados\n- **Link:** https://ec.europa.eu/commission/presscorner/detail/ro/ip_26_229\n- **Uso no relatório:** marco central para demonstrar a passagem da influência do GDPR sobre a LGPD para um reconhecimento jurídico recíproco, com efeitos sobre fluxos de dados e interoperabilidade.\n- **Classificação:** evidência documental primária da convergência reconhecida; o efeito de “ponte” para o Mercosul permanece hipótese.\n- **Acesso:** aberto.\n\n### Parceria Digital Brasil–União Europeia\n\n- **Instituições:** Governo brasileiro e Comissão Europeia\n- **Data:** 12 de junho de 2026\n- **Tipo:** instrumento bilateral oficial\n- **Texto integral:** https://aplicacao.itamaraty.gov.br/ApiConcordia/Documento/download/34485\n- **Apresentação oficial da UE:** https://digital-strategy.ec.europa.eu/en/news/eu-and-brazil-deepen-ties-through-digital-partnership\n- **Uso no relatório:** principal evidência do Brasil como interlocutor estratégico da UE em dados, IA, plataformas, cibersegurança e infraestrutura; permite examinar a articulação entre bilateralismo e inter-regionalismo UE–Mercosul.\n- **Classificação:** evidência documental primária; a função mediadora do Brasil é interpretação a testar por resultados posteriores.\n- **Acesso:** aberto.\n\n### Tomadas de subsídios para as agendas regulatórias da ANPD\n\n- **Instituição:** ANPD\n- **Data:** 1º de setembro de 2026\n- **Tipo:** consulta pública e documentos regulatórios\n- **Link:** https://www.gov.br/anpd/pt-br/assuntos/noticias/anpd-abre-ts-agenda-regulatoria-e-avaliacao-de-resultado-regulatorio\n- **Uso no relatório:** mostra como a Agência pretende avaliar IA, decisões automatizadas, reconhecimento facial, proteção infantil, plataformas e transferências internacionais de dados.\n- **Classificação:** evidência documental primária; a futura circulação regional dessas práticas ainda precisa ser verificada.\n- **Acesso:** aberto.\n\n### Monitoramento de provedores de IA e plataformas digitais — Nota Técnica nº 58/2026\n\n- **Instituição:** ANPD\n- **Data:** 19 de agosto de 2026\n- **Tipo:** processo oficial de fiscalização\n- **Link:** https://www.gov.br/anpd/pt-br/assuntos/noticias/processo-02-sei_00261-005032_2026_78.pdf/%40%40display-file/file\n- **Uso no relatório:** estudo de caso para conectar LGPD, IA, moderação de conteúdo, segurança digital e deveres das plataformas em uma arquitetura regulatória brasileira transversal.\n- **Classificação:** evidência documental primária; a ideia de modelo híbrido brasileiro é interpretação analítica.\n- **Acesso:** aberto.\n\n## 2. Artigos, capítulos e relatórios acadêmicos\n\n### International Regulatory Cooperation and Services Trade: Argentina and Uruguay Adequacy Decisions on Data Protection\n\n- **Autores:** Martina F. Ferracane, Bernard Hoekman, Filippo Santi e Erik van der Marel\n- **Data:** 2026\n- **Tipo:** artigo acadêmico, *Economía LACEA Journal*\n- **Link:** https://economia.lse.ac.uk/articles/10.31389/eco.543\n- **Uso no relatório:** oferece comparação regional indispensável entre Brasil, Argentina e Uruguai e esclarece limites jurídicos do “clube de adequação” europeu.\n- **Classificação:** evidência empírica e interpretação acadêmica.\n- **Acesso:** aberto.\n\n### Towards AI Regulation in Brazil: Past Policies, Future Paths, and Present Uncertainties\n\n- **Autores:** Luca Belli, Filipe Medon e Walter Britto Gaspar\n- **Data:** 24 de fevereiro de 2026\n- **Tipo:** capítulo acadêmico do *Oxford Handbook of Algorithmic Governance and the Law*\n- **Link:** https://www.researchgate.net/publication/401206626_Towards_AI_Regulation_in_Brazil_Past_Policies_Future_Paths_and_Present_Uncertainties\n- **Uso no relatório:** base para distinguir adaptação brasileira e reprodução do AI Act, conectando regulação algorítmica, proteção de dados e participação social.\n- **Classificação:** interpretação acadêmica; o processo legislativo pode exigir atualização.\n- **Acesso:** versão pública disponibilizada por autor; edição Oxford pode ser restrita.\n\n### The Digital Trade Data Heist: Trade Agreement Limits on Data Transfer and Storage Regulation Could Undercut Data Governance\n\n- **Autores:** Daniel Rangel, Jai Vipra e Lori Wallach\n- **Instituição:** South Centre\n- **Data:** 29 de maio de 2026\n- **Tipo:** Research Paper nº 234\n- **Link:** https://www.southcentre.int/research-paper-234-29-may-2026/\n- **Uso no relatório:** perspectiva do Sul Global sobre as tensões entre comércio digital, circulação de dados, autonomia regulatória e compromissos do Mercosul.\n- **Classificação:** análise institucional independente baseada em documentos; conclusões sobre perda de espaço regulatório são interpretações dos autores.\n- **Acesso:** aberto.\n\n### Global Index on Responsible AI: 2026 Report\n\n- **Autores:** Rachel Adams, Fola Adeleke, Ayantola Alayande, Selamawit Engida Abdella, Ana Florido, Nicolás Grossman e Leah Junck\n- **Data:** 16 de julho de 2026\n- **Tipo:** relatório internacional comparativo\n- **Link:** https://arxiv.org/abs/2607.14782\n- **Uso no relatório:** fornece indicadores para comparar capacidade institucional, instrumentos vinculantes e participação social no Brasil, no Mercosul e em outros países do Sul Global.\n- **Classificação:** evidência quantitativa acompanhada de interpretação institucional; dados vão até setembro de 2025.\n- **Acesso:** aberto.\n\n### Open Veins of Algorithmic Auditing: Why AI Assessment Lags Behind Its Deployment in the Global South\n\n- **Autoras:** Gemma Galdon Clavell e Alexandra Magaard\n- **Data:** 23 de julho de 2026\n- **Tipo:** working paper/preprint\n- **Link:** https://arxiv.org/abs/2607.21317\n- **Uso no relatório:** contraponto ao argumento de difusão normativa: aproximação legal não garante capacidade de auditoria, fiscalização nem produção autônoma de evidências.\n- **Classificação:** evidência empírica e interpretação acadêmica preliminares.\n- **Acesso:** aberto; sem indicação de revisão por pares.\n\n### Navigating the Sociotechnical Imaginaries of Brazilian Tech Workers\n\n- **Autor:** Kenzo Soares Seto\n- **Data:** 9 de janeiro de 2026\n- **Tipo:** capítulo/preprint acadêmico\n- **Link:** https://arxiv.org/abs/2601.05961\n- **Uso no relatório:** incorpora atores privados e mostra como profissionais brasileiros traduzem, resistem ou modificam padrões internacionais na implementação.\n- **Classificação:** evidência qualitativa e interpretação acadêmica; amostra de 26 entrevistas.\n- **Acesso:** aberto; revisão por pares não confirmada.\n\n### Mapping Emerging Climate Misinformation Playbooks in the Global South\n\n- **Autores:** Marcelo Sartori Locatelli, Wenchao Dong, Pedro Loures Alzamora, Pedro Dutenhefner, Wagner Meira, Meeyoung Cha e Virgilio Almeida\n- **Data:** 27 de abril de 2026\n- **Tipo:** working paper/preprint com análise computacional\n- **Link:** https://arxiv.org/abs/2604.24223\n- **Uso no relatório:** dados brasileiros para demonstrar limites de políticas convencionais de moderação e a necessidade de adaptação ao contexto discursivo do Sul Global.\n- **Classificação:** evidência empírica de plataforma com interpretação das narrativas.\n- **Acesso:** aberto; revisão por pares não confirmada.\n\n### Beyond the Sandbox: A Critical Assessment of Regulatory Sandboxes for AI\n\n- **Autora:** Giovana Carneiro\n- **Data:** 11 de setembro de 2026\n- **Tipo:** análise acadêmica/institucional\n- **Link:** https://regulatorystudies.columbian.gwu.edu/beyond-sandbox-critical-assessment-regulatory-sandboxes-ai\n- **Uso no relatório:** ajuda a avaliar criticamente o sandbox da ANPD e a evitar que experimentação regulatória seja tratada automaticamente como prova de efetividade.\n- **Classificação:** interpretação acadêmica; deve ser lida em conjunto com a metodologia oficial da ANPD.\n- **Acesso:** aberto.\n\n## 3. Notícias e entrevistas com valor documental\n\n### “Soberania digital não é autossuficiência nem isolamento tecnológico”\n\n- **Fonte:** entrevista com Esther Dweck, ministra da Gestão e da Inovação em Serviços Públicos\n- **Data:** 29 de agosto de 2026\n- **Tipo:** entrevista de política pública\n- **Link:** https://www1.folha.uol.com.br/mercado/2026/08/soberania-digital-nao-e-autossuficiencia-nem-isolamento-tecnologico-diz-esther-dweck.shtml\n- **Uso no relatório:** fonte primária do discurso governamental sobre autonomia, parcerias estratégicas e posicionamento entre diferentes polos tecnológicos.\n- **Classificação:** evidencia a posição declarada do governo, não sua capacidade material ou influência efetiva.\n- **Acesso:** restrito a assinantes.\n\n### ANPD prepara centro de inteligência artificial e amplia regulação de plataformas\n\n- **Fonte:** entrevista com Lorena Giuberti Coutinho, diretora da ANPD\n- **Data:** 9 de setembro de 2026\n- **Tipo:** entrevista sobre política regulatória em desenvolvimento\n- **Link:** https://www1.folha.uol.com.br/cotidiano/2026/09/anpd-prepara-centro-de-inteligencia-artificial-e-mira-rolagem-infinita-diz-diretora.shtml\n- **Uso no relatório:** evidência discursiva da expansão da ANPD da proteção de dados para IA, design de plataformas e moderação.\n- **Classificação:** fonte primária sobre intenção institucional; o centro ainda precisa ser consolidado.\n- **Acesso:** restrito a assinantes.\n\n## 4. Atualizações incorporadas em 25 de setembro de 2026\n\n### CGI.br divulga 46 diretrizes para redes sociais, inteligência artificial e dados\n\n- **Instituição responsável pelas diretrizes:** Comitê Gestor da Internet no Brasil (CGI.br)\n- **Data:** 18 de setembro de 2026\n- **Tipo:** atualização institucional noticiada; documento de diretrizes regulatórias\n- **Link:** https://www.uol.com.br/tilt/noticias/redacao/2026/09/18/cgibr-propoe-freio-a-algoritmos-e-responsabilidade-de-plataformas.ghtm\n- **Uso no relatório:** oferece material diretamente relacionado à formulação brasileira de regras para transparência algorítmica, auditoria, moderação, responsabilidade de plataformas, proteção infantil, aferição de idade e uso de dados sensíveis. É particularmente útil para demonstrar a tentativa de combinar LGPD, IA e governança de plataformas em uma agenda nacional integrada.\n- **Classificação:** a existência e o conteúdo geral das 46 diretrizes são evidência documental atribuída ao CGI.br; a notícia é fonte secundária e deve ser substituída pelo texto integral oficial quando ele estiver localizável.\n- **Acesso:** aberto.\n\n### Digital Government Outlook 2026: Brazil\n\n- **Instituição:** Organização para a Cooperação e Desenvolvimento Econômico (OCDE)\n- **Data:** 15 de junho de 2026\n- **Tipo:** nota-país de relatório institucional comparativo\n- **Link:** https://www.oecd.org/en/publications/digital-government-outlook-2026_d46c0555-en/brazil_019a138e-en.html\n- **Uso no relatório:** fornece evidência comparativa da capacidade estatal brasileira: nota 0,79 no Digital Government Index e 0,70 no OURdata Index, ambas acima das médias da OCDE. Ao mesmo tempo, identifica lacunas em transparência de IA e supervisão de sistemas usados em decisões públicas, permitindo equilibrar o argumento de liderança regional com limites institucionais concretos.\n- **Classificação:** evidência quantitativa e avaliação institucional da OCDE. A conclusão de que essa capacidade sustenta um papel de ponte regulatória é interpretação da pesquisa.\n- **Acesso:** aberto, com PDF disponível.\n\n### The EU seeks a new strategy towards a multipolar Latin America and the Caribbean\n\n- **Autora:** Anna Ayuso, CIDOB\n- **Data:** fevereiro de 2026\n- **Tipo:** relatório/análise institucional\n- **Link:** https://www.cidob.org/en/publications/eu-seeks-new-strategy-towards-multipolar-latin-america-and-caribbean\n- **Uso no relatório:** conecta o acordo UE–Mercosul, a Aliança Digital UE–ALC, a difusão do GDPR e a competição tecnológica com Estados Unidos e China. O texto também identifica o Brasil como possível participante de cooperação triangular com outros atores do Sul Global, oferecendo uma formulação próxima à hipótese de mediação regulatória.\n- **Classificação:** análise especializada baseada em documentos e dados institucionais; a atribuição de função mediadora ao Brasil é interpretação geopolítica, não resultado comprovado.\n- **Acesso:** aberto, com PDF disponível.\n\n### Section 301 — Brazil’s Acts, Policies, and Practices Related to Digital Trade\n\n- **Instituição:** Office of the United States Trade Representative (USTR)\n- **Atualização documental mais recente listada:** 15 de julho de 2026\n- **Tipo:** processo oficial de investigação comercial, com avisos, comentários e transcrições de audiências\n- **Link:** https://ustr.gov/trade-topics/enforcement/section-301-investigations/section-301-brazils-acts-policies-and-practices-related-digital-trade-and-electronic-payment\n- **Uso no relatório:** constitui fonte primária para analisar pressões externas sobre as políticas brasileiras de comércio digital, pagamentos eletrônicos e regulação tecnológica. Ajuda a situar a “ponte” brasileira não apenas entre UE e Mercosul, mas também diante do modelo norte-americano de livre comércio digital e das disputas por autonomia regulatória.\n- **Classificação:** evidência documental primária das alegações e medidas do governo dos Estados Unidos; não prova que as políticas brasileiras sejam discriminatórias ou indevidas.\n- **Acesso:** aberto; algumas transcrições são indicadas como sujeitas a correções.\n\n### Platform signatories publish latest reports under the Code of Conduct on Disinformation\n\n- **Instituição:** Comissão Europeia, DG CONNECT\n- **Data:** 24 de setembro de 2026\n- **Tipo:** atualização oficial e relatórios semestrais de transparência\n- **Link:** https://digital-strategy.ec.europa.eu/en/news/platform-signatories-publish-latest-reports-under-code-conduct-disinformation\n- **Uso no relatório:** fornece um referencial europeu atualizado para comparar obrigações de prestação de contas de Google, Meta, Microsoft e TikTok, com dados sobre janeiro a junho de 2026. A integração do Código ao DSA e a auditoria independente oferecem dimensões concretas para comparar a prática europeia com os mecanismos brasileiros de transparência e moderação.\n- **Classificação:** evidência documental primária sobre o funcionamento do marco co-regulatório europeu. Qualquer influência sobre o Brasil ou o Mercosul deverá ser demonstrada por documentos brasileiros e regionais.\n- **Acesso:** aberto.\n\n## Prioridade de uso no relatório final\n\n1. **Núcleo documental:** adequação Brasil–UE; Parceria Digital Brasil–UE; Ata do Grupo Agenda Digital do Mercosul; metodologia do sandbox; Nota Técnica nº 58/2026.\n2. **Base comparativa:** artigo sobre adequação de Argentina e Uruguai; relatório do South Centre; Global Index on Responsible AI.\n3. **Contrapontos e limites:** auditoria algorítmica no Sul Global; avaliação crítica dos sandboxes; estudo sobre trabalhadores brasileiros de tecnologia.\n4. **Discurso institucional:** entrevistas de Esther Dweck e da diretora da ANPD, sempre apresentadas como declarações e não como prova autônoma de resultados.\n\n## Critério para novas inclusões\n\nEntram apenas fontes que sejam novas, diretamente pertinentes ao projeto e úteis como evidência. Notícias repetitivas, textos sem autoria clara, opinião sem documentação e republicações ficam de fora. Cada nova entrada deve registrar data, autoria ou instituição, tipo, link, acesso, síntese de uso e distinção entre evidência documental e interpretação.\n'}
for nome, conteudo in ARQUIVOS_AGENTE.items():
    destino = AGENTE / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    if nome != 'config/curadoria.json' or not destino.exists():
        destino.write_text(conteudo, encoding='utf-8')
for nome, conteudo in BASE_INCLUIDA.items():
    destino = AGENTE / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists():
        destino.write_text(conteudo, encoding='utf-8')
if str(AGENTE / 'scripts') not in sys.path:
    sys.path.insert(0, str(AGENTE / 'scripts'))
import curadoria_agent as agente
import curadoria_dashboard as painel
import importlib
importlib.reload(painel)
import fapesp_weekly as coletor
from IPython.display import display, HTML, FileLink, Markdown
print('Código carregado; histórico local preservado ao reexecutar.')

## Atualizar e ampliar as entradas
O catálogo reúne fontes já selecionadas; a base dos scrapers reúne notícias coletadas.
A tabela registra os dois tipos. As palavras-chave são calculadas no título por padrão.
Se você habilitar as notas do catálogo, os números passam a incluir também essas notas editoriais.

Para importar outros notebooks, exporte seus resultados em CSV, coloque-os numa pasta e
preencha **PASTA_CSV_EXTRA**. O agente reconhece título/title, link/url, data/date e fonte/source.
Datas sem confirmação permanecem sem data; nenhum arquivo de entrada é alterado.

In [ ]:
#@title 4. Buscar o histórico mais recente do GitHub
import csv
import io
import requests

URL_BASE = "https://raw.githubusercontent.com/anamacao/FAPESP-PIBIC-scrapping/main/"
AVISOS_ENTRADA = []
if BAIXAR_BASE_GITHUB:
    for relativo in ("data/news.csv", "data/runs/latest.json"):
        try:
            resposta = requests.get(URL_BASE + relativo, timeout=20)
            resposta.raise_for_status()
            destino = AGENTE / relativo
            if relativo.endswith(".csv"):
                novas = list(csv.DictReader(io.StringIO(resposta.content.decode("utf-8-sig"))))
                if not novas or not set(coletor.FIELDS).issubset(novas[0]):
                    raise ValueError("O CSV remoto não tem a estrutura esperada.")
                historico = coletor.read_history(destino)
                for item in novas:
                    chave = (item["source_id"], item["url"])
                    antiga = historico.get(chave)
                    if antiga:
                        if antiga.get("last_seen", "") > item.get("last_seen", ""):
                            continue
                        vistas = [v for v in (antiga.get("first_seen"), item.get("first_seen")) if v]
                        if vistas:
                            item["first_seen"] = min(vistas)
                    historico[chave] = {k: item.get(k, "") for k in coletor.FIELDS}
                with destino.open("w", newline="", encoding="utf-8") as handle:
                    writer = csv.DictWriter(handle, fieldnames=coletor.FIELDS)
                    writer.writeheader()
                    writer.writerows(historico.values())
            else:
                diagnostico = resposta.json()
                if not isinstance(diagnostico.get("sources"), list):
                    raise ValueError("Diagnóstico remoto inválido.")
                destino.write_text(json.dumps(diagnostico, ensure_ascii=False, indent=2), encoding="utf-8")
            print("Atualizado:", relativo)
        except (requests.RequestException, ValueError, UnicodeError) as exc:
            aviso = "Atualização do GitHub indisponível para " + relativo + ": " + str(exc)[:180]
            AVISOS_ENTRADA.append(aviso)
            print(aviso, "— usando a cópia disponível.")
else:
    print("Usando a base local incluída; consulte a data da coleta no relatório.")

In [ ]:
#@title 5. Executar a coleta recente e consultar o diagnóstico
if ATUALIZAR_NOTICIAS:
    print("Coletando NIC.br, EDPB, Mercociudades e Senado Federal…")
    diagnostico = coletor.run(AGENTE / "data")
    for fonte in diagnostico["sources"]:
        print(fonte["source_id"], fonte["status"], "| vistos:", fonte["seen"], "| novos:", fonte["new"])
else:
    diagnostico = json.loads((AGENTE / "data/runs/latest.json").read_text(encoding="utf-8"))
    AVISOS_ENTRADA.append("Coleta ao vivo desativada nesta execução; resultados usam o histórico disponível.")
print("Coleta registrada em:", diagnostico.get("generated_at", "não informada"))
print("A rotina coleta notícias; não executa os outros notebooks.")

In [ ]:
#@title 6. Fontes adicionais, catálogo e palavras-chave personalizadas
import shutil

if SALVAR_NO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Montagem do Drive disponível na sessão do Google Colab.") from exc
    drive.mount("/content/drive")

CONFIGURACAO = AGENTE / "config/curadoria.json"
config = agente.load_config(CONFIGURACAO)
# Exemplo: {"Auditoria algorítmica": ["auditoria algorítmica", "algorithmic audit"]}
PALAVRAS_EXTRAS = {}
for termo, sinonimos in PALAVRAS_EXTRAS.items():
    config["keywords"][termo] = sinonimos
    if termo not in config["relevance_keywords"]:
        config["relevance_keywords"].append(termo)
CONFIGURACAO.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

CATALOGO = Path(CAMINHO_CATALOGO_PROPRIO) if CAMINHO_CATALOGO_PROPRIO else AGENTE / "data/curadoria/Catalogo_de_fontes_FAPESP.md"
ENTRADA_NOTICIAS = AGENTE / "data/news.csv"
if PASTA_CSV_EXTRA:
    pasta = Path(PASTA_CSV_EXTRA)
    arquivos = sorted(pasta.glob("*.csv"))
    if not arquivos:
        raise ValueError("Nenhum CSV encontrado em PASTA_CSV_EXTRA.")
    todas = agente.read_news(ENTRADA_NOTICIAS)
    for arquivo in arquivos:
        adicionais = painel.import_csv(arquivo)
        todas.extend(adicionais)
        print(arquivo.name, ":", len(adicionais), "linhas")
    ENTRADA_NOTICIAS = AGENTE / "data/noticias_com_importacoes.csv"
    # Grava apenas uma entrada derivada; mantém intactos os CSVs originais e o histórico.
    campos = ["source_id", "title", "published_date", "url", "first_seen", "last_seen",
              "summary", "publication_type"]
    with ENTRADA_NOTICIAS.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=campos, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(todas)
if not CATALOGO.is_file():
    raise FileNotFoundError("Catálogo não encontrado: " + str(CATALOGO))
print("Entradas:", ENTRADA_NOTICIAS.name, "|", CATALOGO.name, "|", len(config["keywords"]), "termos no dicionário")

In [ ]:
#@title 7. Gerar as tabelas, os gráficos e o relatório completo
SAIDA = AGENTE / "resultados" / REFERENCIA.isoformat()
metadata = agente.run(ENTRADA_NOTICIAS, SAIDA, CONFIGURACAO, REFERENCIA, JANELA_DIAS,
                      CATALOGO, AGENTE / "data/runs/latest.json", MODO_TEXTO)
metadata["warnings"].extend(AVISOS_ENTRADA)
# Recria o painel com os avisos de rede desta sessão e os mesmos insumos.
corpus_data = {}
for corpus, bruto in (("scrapers", agente.read_news(ENTRADA_NOTICIAS)), ("catalogo", agente.read_catalog(CATALOGO))):
    registros, qualidade = agente.prepare(bruto, config, MODO_TEXTO)
    corpus_data[corpus] = agente.summarize(registros, config, REFERENCIA, JANELA_DIAS,
                                         diagnostico if corpus == "scrapers" else None)
painel.export_dashboard(SAIDA, corpus_data, config, metadata)
(SAIDA / "resumo.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
for corpus, totais in metadata["corpus"].items():
    print(corpus, ":", totais["documents"], "registros,", totais["candidates_window"], "candidatas lexicais na janela")
for aviso in metadata["warnings"]:
    print("Cobertura:", aviso)
print("Saídas:", SAIDA)

## Tabelas e gráficos da pesquisa
Os números descrevem os documentos observados nas fontes monitoradas.
O catálogo foi selecionado previamente e não é uma amostra aleatória de notícias.
Um documento pode integrar mais de um eixo. A coocorrência não comprova influência regulatória.
Datas apenas mensais entram no gráfico mensal, mas ficam fora da comparação semanal.

In [ ]:
#@title 8. Tabelas de fontes, termos e documentos
if CORPUS_GRAFICOS == "todos":
    registros = painel.merge_documents(corpus_data)
else:
    registros = corpus_data[CORPUS_GRAFICOS]["rows"]
observados = painel.filter_documents(registros, REFERENCIA.isoformat())
medidas = painel.measures(observados, config, REFERENCIA, JANELA_DIAS)
display(Markdown("### Frequência das palavras-chave"))
display(HTML(agente.html_table(medidas["keywords"], [("palavra_chave", "Palavra-chave"), ("documentos", "Documentos")])))
display(Markdown("### Cobertura dos scrapers"))
display(HTML(agente.html_table(corpus_data["scrapers"]["sources"],
    [("fonte", "Fonte"), ("total_base", "Base"), ("publicadas_janela", "Publicadas na janela"),
     ("candidatas_janela", "Candidatas"), ("status_ultima_coleta", "Coleta"), ("observacao", "Observação")])))
display(Markdown("### Documentos para leitura — todos os registros nas tabelas CSV"))
display(HTML(agente.html_table(observados[:30],
    [("title", "Título e link"), ("source", "Fonte"), ("published_date", "Publicação"),
     ("topics", "Eixos"), ("keywords", "Palavras-chave"), ("match_evidence", "Evidência lexical")])))
print("Prévia: até 30 linhas. O painel HTML e os CSVs contêm todas as linhas.")

In [ ]:
#@title 9. Gráficos interativos — passe o mouse, amplie e baixe imagens
import plotly.io as pio
try:
    EM_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    EM_COLAB = False
if EM_COLAB:
    pio.renderers.default = "colab"
else:
    pio.renderers.default = "plotly_mimetype"
FIGURAS = painel.notebook_figures(observados, config, REFERENCIA, JANELA_DIAS)
for nome, figura in FIGURAS.items():
    figura.show()
print("O painel HTML acrescenta filtros globais e a matriz normalizada de palavras-chave por fonte.")

### 9b. Visualizações Interativas Adicionais (Novos Tipos de Gráficos)
Este bloco gera novas perspectivas visuais utilizando os dados estruturados do recorte atual (`observados`), expandindo o catálogo padrão de gráficos.

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from collections import Counter

# --- 1. Gráfico de Rosca: Proporção das Fontes no Recorte ---
source_counts = Counter(r['source'] for r in observados)
source_df = pd.DataFrame(source_counts.items(), columns=['Fonte', 'Documentos']).sort_values(by='Documentos', ascending=False)

# Usando uma paleta integrada existente ou uma lista personalizada de cores
fig_pie = px.pie(
    source_df,
    values='Documentos',
    names='Fonte',
    title='Proporção de Publicações por Fonte (Share)',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Safe
)
fig_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pie.update_layout(template='plotly_white')
fig_pie.show()

# --- 2. Gráfico Sunburst: Relação Hierárquica Eixos -> Palavras-Chave ---
sunburst_data = []
for r in observados:
    for t in r.get('topics', []):
        for k in r.get('keywords', []):
            sunburst_data.append({'Eixo': t, 'Palavra_Chave': k})

if sunburst_data:
    sunburst_df = pd.DataFrame(sunburst_data).value_counts().reset_index(name='Documentos')
    fig_sunburst = px.sunburst(
        sunburst_df,
        path=['Eixo', 'Palavra_Chave'],
        values='Documentos',
        title='Distribuição Hierárquica: Eixos de Pesquisa e suas Palavras-Chave',
        color='Documentos',
        color_continuous_scale='Blues'
    )
    fig_sunburst.update_layout(template='plotly_white', height=550)
    fig_sunburst.show()
else:
    print("Sem dados hierárquicos suficientes (eixos/palavras-chave) para exibir o gráfico Sunburst.")

# --- 3. Gráfico de Bolhas (Bubble Chart): Relação de Coocorrência de Termos ---
# Uma alternativa dinâmica ao heatmap tradicional
pair_counts = Counter()
for r in observados:
    for a in r.get('keywords', []):
        for b in r.get('keywords', []):
            if a != b:
                pair_counts[tuple(sorted([a, b]))] += 1

bubble_data = []
for (term1, term2), count in pair_counts.items():
    bubble_data.append({'Termo A': term1, 'Termo B': term2, 'Frequência': count // 2}) # Ajusta duplicatas de permutação

if bubble_data:
    bubble_df = pd.DataFrame(bubble_data).drop_duplicates().sort_values(by='Frequência', ascending=False).head(40)
    fig_bubble = px.scatter(
        bubble_df,
        x='Termo A',
        y='Termo B',
        size='Frequência',
        color='Frequência',
        hover_name='Frequência',
        title='Relação de Coocorrência de Palavras-Chave (Bubble Chart)',
        color_continuous_scale='Tealgrn',
        size_max=40
    )
    fig_bubble.update_layout(
        template='plotly_white',
        xaxis=dict(tickangle=-45),
        height=550
    )
    fig_bubble.show()
else:
    print("Sem coocorrências suficientes para exibir o gráfico de bolhas.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# 1. Preparar e agrupar os dados do recorte atual
source_counts = Counter(r['source'] for r in observados)
source_df = pd.DataFrame(source_counts.items(), columns=['Fonte', 'Quantidade'])
source_df = source_df.sort_values(by='Quantidade', ascending=False)

# Limitar às top 4 fontes e agrupar o resto como 'Outras'
top_n = 4
if len(source_df) > top_n:
    top_sources = source_df.iloc[:top_n].copy()
    others_count = source_df.iloc[top_n:]['Quantidade'].sum()
    others_row = pd.DataFrame([{'Fonte': 'Outras', 'Quantidade': others_count}])
    waffle_df = pd.concat([top_sources, others_row], ignore_index=True)
else:
    waffle_df = source_df.copy()

# 2. Calcular proporção para uma grade de 10x10 (100 quadradinhos)
total_docs = waffle_df['Quantidade'].sum()
waffle_df['Porcentagem'] = (waffle_df['Quantidade'] / total_docs * 100).round().astype(int)

# Ajustar arredondamentos para garantir exatamente 100 blocos
diferenca = 100 - waffle_df['Porcentagem'].sum()
if diferenca != 0:
    waffle_df.loc[waffle_df['Porcentagem'].idxmax(), 'Porcentagem'] += diferenca

# 3. Criar a matriz da grade 10x10
waffle_grid = []
for idx, row in waffle_df.iterrows():
    waffle_grid.extend([idx] * row['Porcentagem'])

waffle_matrix = np.array(waffle_grid).reshape((10, 10))

# 4. Desenhar o gráfico de Waffle (Caderno Quadriculado)
fig, ax = plt.subplots(figsize=(8, 8), facecolor='white')

# Paleta de cores suave para cada categoria
cores = ['#153b4d', '#237c83', '#5d7180', '#b9e5e2', '#dce5ea']

# Plotar a matriz como imagem discreta
im = ax.imshow(waffle_matrix, cmap=plt.cm.colors.ListedColormap(cores[:len(waffle_df)]), interpolation='nearest')

# Adicionar bordas visíveis em cada quadradinho
ax.set_xticks(np.arange(-0.5, 10, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 10, 1), minor=True)
ax.grid(which='minor', color='#ffffff', linestyle='-', linewidth=3.5)

# Remover ticks e numerações dos eixos principais
ax.set_xticks([])
ax.set_yticks([])

# Criar legenda personalizada com as fontes e suas porcentagens
labels = [f"{r['Fonte']} ({r['Porcentagem']}%%)" for _, r in waffle_df.iterrows()]
patches = [plt.plot([],[], marker="s", ms=12, ls="", color=cores[i],
                    label=labels[i])[0] for i in range(len(waffle_df))]

ax.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.05),
          ncol=2, frameon=False, fontsize=11)

plt.title('Distribuição de Fontes (Gráfico de Waffle / Quadriculado)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Configurar o estilo visual do gráfico
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False
})

# 2. Criar a figura
fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')

# 3. Gerar o gráfico de barras horizontais usando os dados de waffle_df
# (que já estão agregados e calculados com as porcentagens e a categoria 'Outras')
palete_cores = ['#153b4d', '#237c83', '#5d7180', '#b9e5e2', '#a6b7c7']

barplot = sns.barplot(
    data=waffle_df,
    x='Quantidade',
    y='Fonte',
    palette=palete_cores[:len(waffle_df)],
    hue='Fonte',
    legend=False,
    ax=ax
)

# 4. Adicionar rótulos de valores e porcentagens no final de cada barra
for i, bar in enumerate(ax.patches):
    quantidade = waffle_df.iloc[i]['Quantidade']
    porcentagem = waffle_df.iloc[i]['Porcentagem']
    ax.text(
        bar.get_width() + (max(waffle_df['Quantidade']) * 0.01),
        bar.get_y() + bar.get_height()/2,
        f"{int(quantidade)} u. ({porcentagem}%)",
        va='center',
        ha='left',
        fontweight='bold',
        color='#21394b'
    )

# 5. Personalizar títulos e eixos
ax.set_title('Distribuição de Documentos por Fonte de Informação', fontsize=14, fontweight='bold', pad=15, loc='left', color='#153b4d')
ax.set_xlabel('Quantidade de Documentos', fontsize=12, fontweight='semibold', color='#21394b')
ax.set_ylabel('Fonte', fontsize=12, fontweight='semibold', color='#21394b')

# Ajustar os limites do eixo X para dar espaço aos rótulos de texto
ax.set_xlim(0, max(waffle_df['Quantidade']) * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# Criar uma cópia dos documentos filtrados para não alterar o estado original
pool = [dict(r) for r in observados]

# Calcular a pontuação de prioridade para cada documento conforme a lógica do relatório
for row in pool:
    row["priority_score"] = (
        min(5, len(row.get("keywords", [])))
        + 2 * min(3, len(row.get("topics", [])))
        + 3 * bool(row.get("manually_selected", False))
    )

# Ordenar por pontuação (decrescente) e data de publicação
top_20 = sorted(
    pool,
    key=lambda r: (r["priority_score"], r.get("published_date", ""), r.get("title", "")),
    reverse=True
)[:20]

# Estruturar os dados em um DataFrame
data_table = []
for i, row in enumerate(top_20, 1):
    data_table.append({
        "Posição": f"<b>{i}º</b>",
        "Título": f'<a href="{row["url"]}" target="_blank" style="color: #176c81; font-weight: 500;">{row["title"]}</a>',
        "Fonte": row["source"],
        "Data": row.get("published_date", "Não informada"),
        "Eixos Temáticos": ", ".join(row.get("topics", [])) or "<i>Nenhum</i>",
        "Pontuação": f'<span style="background-color: #e4f0f1; padding: 4px 8px; border-radius: 4px; font-weight: bold;">{row["priority_score"]} pts</span>'
    })

df_top20 = pd.DataFrame(data_table)

# Estilização em HTML para garantir ótima legibilidade no Colab
html_style = """
<style>
    .top20-table { width: 100%; border-collapse: collapse; font-family: sans-serif; font-size: 13px; margin: 10px 0; }
    .top20-table th { background-color: #153b4d; color: white; padding: 12px; text-align: left; }
    .top20-table td { padding: 10px 12px; border-bottom: 1px solid #dce5ea; vertical-align: top; }
    .top20-table tr:hover { background-color: #f5f9fa; }
</style>
"""

table_html = df_top20.to_html(index=False, escape=False, classes="top20-table")

display(HTML(html_style + "<h3>📋 Top 20 Publicações Mais Importantes (Critério de Prioridade)</h3>" + table_html))

In [ ]:
#@title 10. Resumo semanal e até cinco publicações para leitura
display(Markdown((SAIDA / "resumo_semanal.md").read_text(encoding="utf-8")))

In [ ]:
#@title 11. Salvar e baixar o pacote de resultados
ARQUIVO_ZIP = Path(shutil.make_archive(str(AGENTE / ("Curadoria_FAPESP_" + REFERENCIA.isoformat())), "zip", SAIDA))
if SALVAR_NO_GOOGLE_DRIVE:
    destino = Path(PASTA_DRIVE) / REFERENCIA.isoformat()
    destino.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SAIDA, destino, dirs_exist_ok=True)
    shutil.copy2(ARQUIVO_ZIP, destino.parent / ARQUIVO_ZIP.name)
    print("Resultados salvos em:", destino)
print("Abra painel_interativo.html depois de extrair o ZIP.")
display(FileLink(str(ARQUIVO_ZIP)))
if BAIXAR_ZIP_AO_TERMINAR and EM_COLAB:
    from google.colab import files
    files.download(str(ARQUIVO_ZIP))
print("Inclui: painel HTML, CSVs, gráficos PNG, resumo semanal e registro dos insumos.")

## Executar toda semana
O Colab serve para executar e explorar. O repositório já tem uma rotina de coleta por
GitHub Actions às **sextas-feiras às 07h de Brasília**, antes da curadoria das 08h.
A integração deste painel está ativa: os gráficos e tabelas são gerados após cada coleta
agendada e ficam disponíveis no artefato `curadoria-fapesp` do GitHub Actions.

O catálogo incluído é uma cópia datada, não uma sincronização automática com conversas.
Atualize o arquivo do catálogo ou indique um catálogo próprio para incorporar novas seleções.
As publicações sugeridas devem ser lidas antes de virar evidência no relatório FAPESP.